# lmsafe -- one self-contained Colab notebook (code + real LANL experiment)

This single notebook contains the **entire `lmsafe` package written inline**
(all detectors, the five update policies, the fair evaluation protocol,
metrics, tests, scripts, configs, and docs) -- no separate ZIP upload
needed. Run the cells in order from top to bottom (Shift+Enter); every
project file is written to disk automatically via `%%writefile` cells,
exactly like the earlier notebook that wrote `lanl_stream.py` the same
way, except now it's the whole package.

**Notebook outline:**
1. Set up the project folder.
2. Write the entire `lmsafe` package to disk (every module).
3. Install the package (`pip install -e .`).
4. Write the scripts (`scripts/`) and configs (`configs/`).
5. Write the tests (`tests/`) and run them (92 tests -- confirms everything
   works correctly in the Colab environment before spending any real
   network time).
6. Write the docs (`docs/`, `README.md`).
7. **The real experiment**: LANL link, quick smoke test, extract the
   red-team file, build a window cache, run the full sweep on real LANL
   data.
8. (Optional) run the RQ1/RQ2/RQ3 demonstration scripts on synthetic data.
9. (Optional, slow) the full 58-day cut for final paper numbers.
10. Copy everything to Google Drive.

**Security note, same as the earlier notebook:** the signed LANL link is a
temporary credential. It is requested later via a hidden prompt
(`getpass`) and is never written into any cell or file.

## 1) Set up the project folder

In [ ]:
import os

PROJECT_ROOT = "/content/lmsafe_project"
os.makedirs(PROJECT_ROOT, exist_ok=True)
%cd {PROJECT_ROOT}

for d in [
    "lmsafe/data",
    "lmsafe/detectors",
    "scripts",
    "configs",
    "tests/fixtures",
    "docs",
    "lmsafe_cache",
    "results",
]:
    os.makedirs(d, exist_ok=True)

print("project root:", os.getcwd())

## 2) Write the entire `lmsafe` package to disk

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "lmsafe"
version = "0.1.0"
description = "Safe continual updating of lateral-movement detectors under benign drift and update poisoning"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "numpy",
    "pandas",
    "scikit-learn",
    "torch",
    "pyyaml",
    "matplotlib",
]

[tool.setuptools.packages.find]
include = ["lmsafe*"]


In [ ]:
%%writefile lmsafe/__init__.py
"""
lmsafe -- Safe Continual Updating for Lateral-Movement Detection
==================================================================

Research code accompanying the paper:

    "Safe Continual Updating of Lateral-Movement Detectors
     under Benign Drift and Update Poisoning"

Problem framing
----------------
Enterprise lateral-movement (LM) detectors are usually trained once on a
benign period and then frozen, or periodically retrained on recent traffic
to keep up with legitimate change ("drift": new hosts, new users, new
working hours). This package treats the *updating mechanism itself* as an
attack surface: an adversary who can influence what the detector sees
during normal operation (a "boiling-frog" / conditioning attacker) can bias
future updates so that the detector slowly learns to treat the attacker's
own traffic pattern as normal, before the real lateral-movement step
happens.

Three research questions drive the design of every module here:

    RQ1  Drift cost      -- how much detection quality do different update
                             policies lose to *benign* drift alone, and how
                             fast do they recover?
    RQ2  Update poisoning -- how much detection quality do those same
                             policies lose when an adaptive attacker injects
                             conditioning edges before attacking?
    RQ3  Gated defense    -- does a score-gated, delayed-commit update
                             policy recover RQ1's benign-drift performance
                             *without* re-opening the RQ2 poisoning gap, at
                             the same fixed alert budget?

Package layout
---------------
    lmsafe.data      -- event streams -> labeled, time-windowed graphs
                         (lanl.py, optc.py, windows.py, schema.py)
    lmsafe.synth     -- synthetic auth-log generator with known ground
                         truth (drift + injectable LM episodes)
    lmsafe.detectors -- link-prediction anomaly scorers (count baseline,
                         small torch sequence/GNN model)
    lmsafe.drift     -- Page-Hinkley / ADWIN concept-drift detectors, used
                         only to *trigger* an update policy, never as the
                         anomaly score itself
    lmsafe.policies  -- the five update policies compared in the paper
                         (frozen, periodic, sliding+replay, drift-triggered,
                         gated)
    lmsafe.attack    -- conditioning / update-poisoning injector used
                         *only* inside this offline evaluation harness, to
                         measure policy robustness (RQ2). It never touches
                         a live network.
    lmsafe.metrics   -- event-level AUC/AP, Rec@B (recall at a fixed alert
                         budget), detection delay, and episode-level
                         bootstrap confidence intervals
    lmsafe.runner    -- orchestrates data -> policy loop -> detector ->
                         metrics for one experiment cell

See README.md for the full pipeline diagram and how each script under
scripts/ maps to RQ1/RQ2/RQ3.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile lmsafe/config.py
"""
Central configuration constants and small config dataclasses.

Every *time* constant in this file is in seconds since the start of the
LANL "cyber1" dataset (its own internal clock, not a wall-clock date -- the
released dataset was de-identified and re-timestamped by LANL). Getting
these numbers right is the single most important correctness property of
the whole pipeline: a wrong split silently turns an "unseen test attack"
into "an attack the model trained on", which inflates every downstream
metric without raising any error.

References for the constants below
------------------------------------
- Kent, A. D. (2015). *Comprehensive, Multi-Source Cyber-Security Events*.
  Los Alamos National Laboratory. https://doi.org/10.17021/1179829
- Larroche, T. (2026). LMDEval fair-evaluation protocol for lateral-movement
  detection (BSD-2-Clause). The temporal split and "keep all remote LogOn
  events, don't filter to NTLM-only" rules are adopted directly from this
  protocol so our numbers are comparable to it. See NOTICE.md.
"""
from __future__ import annotations

import dataclasses
from pathlib import Path
from typing import Optional

# ---------------------------------------------------------------------------
# LANL "cyber1" temporal constants (seconds since the dataset's own epoch)
# ---------------------------------------------------------------------------

#: One hour / one day in seconds, spelled out because every other constant
#: below is defined as a multiple of one of these -- never hardcode a second
#: copy of "86400" elsewhere, import SECONDS_PER_DAY instead.
SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY = 86400

#: The first red-team (ground-truth malicious) event in the public LANL
#: redteam.txt.gz is at t = 150885 s = 41.9h. Training on anything at or
#: after this point risks leaking attack traffic into the "benign" model.
#: We cut training at a round 41 hours, strictly before this.
LANL_REDTEAM_FIRST_EVENT_S = 150_885
LANL_TRAIN_CUTOFF_S = 41 * SECONDS_PER_HOUR  # 147,600 s -- matches LMDEval

#: The full released auth log spans 58 days, but (as of this project's own
#: inspection of redteam.txt.gz on 2026-11-13) every one of the 749 labeled
#: red-team events falls before day 29.6 and there are zero events after
#: day 30. We therefore default every *development* run to a 30-day cut
#: (LANL_DEV_CUTOFF_S) -- this already contains 100% of the known attack
#: episodes while reading roughly half the 7.2 GB auth stream. Final paper
#: numbers should additionally be reported on the full LANL_FULL_CUTOFF_S
#: window to match LMDEval exactly; set --end-s accordingly in the scripts.
LANL_DEV_CUTOFF_S = 30 * SECONDS_PER_DAY        # 2,592,000 s
LANL_FULL_CUTOFF_S = 58 * SECONDS_PER_DAY       # 5,011,200 s

#: LMDEval's own ablation window (14 days) -- useful for a quick, still
#: fairly complete, run: it contains 628 / 749 (84%) of the red-team events.
LANL_QUICK_CUTOFF_S = 14 * SECONDS_PER_DAY      # 1,209,600 s

#: OpTC (second dataset, used for the transfer / generalization check in a
#: later chapter, not for RQ1-RQ3): 6 days train, last 3 days test, per the
#: fair protocol in the project summary.
OPTC_TRAIN_DAYS = 6
OPTC_TEST_LAST_DAYS = 3

#: Window size used to turn the raw event stream into a sequence of graphs.
#: One hour balances graph density (enough edges per window to fit a model)
#: against responsiveness (how quickly a policy can react to drift/attack).
DEFAULT_WINDOW_SECONDS = SECONDS_PER_HOUR

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

#: Default cache directory for anything downloaded/extracted once and
#: reused across runs (the aggregated LANL window-graphs, OpTC extracts).
#: Override via the LMSAFE_CACHE_DIR environment variable, e.g. to point at
#: a mounted Google Drive folder on Colab.
import os  # noqa: E402

DEFAULT_CACHE_DIR = Path(os.environ.get("LMSAFE_CACHE_DIR", "./lmsafe_cache"))


@dataclasses.dataclass(frozen=True)
class LanlConfig:
    """Configuration for one LANL extraction / experiment run.

    Notes
    -----
    `base_url` is intentionally *not* a field with a default value, and
    it is never read from a config file on disk: it is a short-lived
    (~8 hour) signed credential handed out by
    https://csr.lanl.gov/data/cyber1/ after you register and accept the
    data-use terms there. See `lmsafe.data.lanl_stream.resolve_base_url`
    for how it is obtained at run time instead of being hardcoded.
    """

    end_s: int = LANL_DEV_CUTOFF_S
    start_s: int = 0
    train_cutoff_s: int = LANL_TRAIN_CUTOFF_S
    window_seconds: int = DEFAULT_WINDOW_SECONDS
    only_remote_logon: bool = True   # the LMDEval fair-protocol filter
    filter_ntlm_only: bool = False   # keep False -- that filter is the
                                      # non-fair shortcut this project
                                      # deliberately avoids (see README).
    cache_dir: Path = DEFAULT_CACHE_DIR


@dataclasses.dataclass(frozen=True)
class SynthConfig:
    """Configuration for the synthetic event generator (lmsafe.synth).

    The synthetic generator exists to unit-test and smoke-test every
    module *downstream* of data extraction (windows, detectors, drift,
    policies, attack, metrics) without any network access and with a
    dataset small enough to run on a 2-CPU, no-GPU sandbox in seconds.
    It is not a substitute for the real LANL/OpTC results in the paper.
    """

    n_users: int = 60
    n_hosts: int = 40
    n_benign_days: int = 20
    window_seconds: int = DEFAULT_WINDOW_SECONDS
    base_edges_per_window: int = 250
    drift_start_day: Optional[int] = 10     # when benign drift begins
    drift_strength: float = 0.5             # 0 = none, 1 = strong
    n_episodes: int = 6                     # injected lateral-movement episodes
    episode_chain_len: int = 5               # hops per episode (actor -> N targets)
    episode_hop_seconds: int = 30            # spacing between hops in one episode
    seed: int = 0


In [ ]:
%%writefile lmsafe/data/__init__.py
"""Data layer: turns raw authentication-event logs into labeled,
time-windowed graphs that the rest of lmsafe consumes.

Sub-modules
-----------
schema        canonical in-memory Event representation shared by every
              data source (LANL, OpTC, synthetic), so detectors/policies/
              metrics never need to know which dataset they are looking at.
lanl_stream   low-level, dependency-free streaming reader for the LANL
              cyber1 .txt.gz files (never downloads the whole file).
lanl.py       the fair LANL extraction protocol built on top of
              lanl_stream: temporal split, red-team label join, remote-
              LogOn filter. This is the module scripts/ actually calls.
optc.py       OpTC schema adapter (Phase-2 dataset, used for the transfer
              check in a later chapter -- stubbed with a clear TODO path).
windows.py    groups a stream of Events into fixed-length time windows and
              builds the per-window graph (node index + weighted edge list)
              that lmsafe.detectors consumes.
"""


In [ ]:
%%writefile lmsafe/data/schema.py
"""
Canonical event representation shared by every data source.

Every reader in this package (LANL, OpTC, the synthetic generator) emits a
stream of `Event` records in this one shape. Nothing downstream of
`lmsafe.data` -- not `windows.py`, not a detector, not a policy, not
`metrics.py` -- is allowed to know which dataset produced an Event. That
boundary is what makes it possible to swap LANL for OpTC or for the
synthetic generator without touching a single line of model or evaluation
code, and it is also what makes the synthetic generator a meaningful stand-
in for unit-testing the rest of the pipeline offline.

Design note on `is_malicious`
------------------------------
`is_malicious` is carried on the Event itself purely as *ground truth for
evaluation*. No code path in `lmsafe.detectors` or `lmsafe.policies` is
permitted to read it -- it is consumed exclusively by `lmsafe.metrics` and
by `lmsafe.attack` (which, by construction, needs to know which synthetic
events it itself injected). A unit test in tests/test_no_label_leak.py
grep-checks this invariant across the codebase.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True, slots=True)
class Event:
    """One authentication / lateral-movement-relevant event.

    Attributes
    ----------
    t : int
        Event time in seconds since the dataset's own epoch (LANL-style).
    src_user : str
        The authenticating principal, e.g. "U66@DOM1".
    dst_user : str
        The identity the authentication resolves to on the destination
        (often identical to src_user; kept distinct because LANL carries
        a separate column -- see `lmdeval`'s extraction logic in
        NOTICE.md -- and because impersonation shows up as a mismatch).
    src_comp : str
        Source computer/host.
    dst_comp : str
        Destination computer/host. The directed edge (src_comp, dst_comp)
        at time t is the unit of analysis for lateral-movement detection.
    auth_type : str
        Raw authentication protocol/type field from the source dataset
        (e.g. "Kerberos", "NTLM"); kept for diagnostics, not filtered on
        by default (see lmsafe.config.LanlConfig.filter_ntlm_only).
    logon_type : str
        Raw logon-type field: the Windows authentication *mechanism*
        (e.g. "Network", "Interactive", "Batch", "Service",
        "NetworkCleartext", "Unlock"). This is never "LogOn"/"LogOff" --
        that distinction lives in `orientation`, not here. Kept for
        diagnostics; not used by `is_remote_logon`.
    orientation : str
        LANL's own LogOn/LogOff marker for the event (values "LogOn" or
        "LogOff"). The fair protocol keeps only "LogOn"-oriented events
        between two different hosts -- see `is_remote_logon`.
    success : bool
        Whether the authentication succeeded.
    is_malicious : bool
        Ground-truth label. False for every real LANL/OpTC event unless it
        matches a red-team record; set by the synthetic generator and by
        `lmsafe.attack` for injected events. **Evaluation-only** -- see
        the module docstring.
    source : str
        Which reader produced this event: "lanl", "optc", or "synthetic".
        Lets a window-graph builder be dataset-agnostic while a debugging
        session can still ask "where did this edge come from?".
    """

    t: int
    src_user: str
    dst_user: str
    src_comp: str
    dst_comp: str
    auth_type: str = ""
    logon_type: str = ""
    orientation: str = ""
    success: bool = True
    is_malicious: bool = False
    source: str = "unknown"

    @property
    def edge(self) -> tuple[str, str]:
        """The directed host-level edge this event contributes to."""
        return (self.src_comp, self.dst_comp)

    def is_remote_logon(self) -> bool:
        """True for a remote LogOn between two *different* hosts.

        This is the fair-protocol filter (see lmsafe.config and
        NOTICE.md): it keeps "LogOn"-*oriented* events where the source
        host differs from the destination host, with no further
        filtering by auth_type, account type, or success. Deliberately
        *not* restricted to NTLM (CyberGFM-style preprocessing filters to
        NTLM-only and reports headline numbers on that easier,
        non-comparable subset).

        Checks `orientation`, not `logon_type`: in LANL's own auth.txt.gz
        schema, "LogOn"/"LogOff" is the value of the *orientation*
        column. `logon_type` instead holds the Windows logon mechanism
        ("Network", "Batch", "Interactive", ...) and is never literally
        "LogOn" -- comparing `logon_type` against "LogOn" silently
        rejects every real-data row (this was a real, shipped bug: it
        produced `kept=0` across 1.5B+ real LANL lines before being
        caught and fixed).
        """
        return self.orientation == "LogOn" and self.src_comp != self.dst_comp


#: CSV column order used by every CSV reader/writer in this package, kept
#: as one constant so a schema change is a one-line edit.
CSV_FIELDS: tuple[str, ...] = (
    "t",
    "src_user",
    "dst_user",
    "src_comp",
    "dst_comp",
    "auth_type",
    "logon_type",
    "orientation",
    "success",
    "is_malicious",
    "source",
)


def event_from_csv_row(row: dict) -> Event:
    """Parse one `csv.DictReader` row (strings) into an `Event`."""
    return Event(
        t=int(row["t"]),
        src_user=row["src_user"],
        dst_user=row["dst_user"],
        src_comp=row["src_comp"],
        dst_comp=row["dst_comp"],
        auth_type=row.get("auth_type", ""),
        logon_type=row.get("logon_type", ""),
        orientation=row.get("orientation", ""),
        success=_parse_bool(row.get("success", "True")),
        is_malicious=_parse_bool(row.get("is_malicious", "False")),
        source=row.get("source", "unknown"),
    )


def event_to_csv_row(e: Event) -> dict:
    """Serialize an `Event` to a `dict` suitable for `csv.DictWriter`."""
    return {
        "t": e.t,
        "src_user": e.src_user,
        "dst_user": e.dst_user,
        "src_comp": e.src_comp,
        "dst_comp": e.dst_comp,
        "auth_type": e.auth_type,
        "logon_type": e.logon_type,
        "orientation": e.orientation,
        "success": e.success,
        "is_malicious": e.is_malicious,
        "source": e.source,
    }


def _parse_bool(v) -> bool:
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ("1", "true", "t", "yes")


@dataclass(frozen=True)
class RedTeamRecord:
    """One ground-truth red-team (lateral-movement) record from
    redteam.txt.gz: ``time,user,src_comp,dst_comp``."""

    t: int
    user: str
    src_comp: str
    dst_comp: str

    def key(self) -> tuple[int, str, str, str]:
        """Join key against an Event: (t, dst_user, src_comp, dst_comp).

        This mirrors LMDEval's `extract_lanl.py` exactly: the red-team
        file's "user" field is matched against an auth event's
        *destination* user, not its source user. Getting src/dst
        reversed here silently produces a detector that never sees a
        correctly labeled positive -- see NOTICE.md and
        tests/test_lanl_labels.py for a regression test.
        """
        return (self.t, self.user, self.src_comp, self.dst_comp)


In [ ]:
%%writefile lmsafe/data/lanl_stream.py
"""
Dependency-free streaming reader for the LANL "cyber1" .txt.gz files.

Why this exists
----------------
The auth.txt.gz file is 7.2 GB. Downloading it is unnecessary for every
experiment in this project except the one full-dataset paper run, and the
user explicitly does not want it sitting on disk. This module reads the
remote gzip stream directly, decompressing as bytes arrive, and lets the
caller stop as soon as it has read past its time window -- so a 30-day
window transfers only the fraction of the file up to that point, not all
7.2 GB.

The credential problem
-----------------------
The download links on https://csr.lanl.gov/data/cyber1/ are signed,
time-limited URLs of the form::

    https://csr.lanl.gov/data-fence/<expiry>/<signature>/cyber1/<file>.txt.gz

`<expiry>`/`<signature>` are generated by that page *after you submit the
registration form there* -- they are not static HTML on the page, so this
module cannot "scrape a fresh link" automatically (we checked: the page
served to an unauthenticated fetch contains the filenames and sizes but no
anchor tags or embedded links at all). Concretely this means:

    1. The signed link is a bearer credential. It is read at *run time*
       from the LMSAFE_LANL_BASE_URL environment variable, or interactively
       via getpass if none is set. It is never written into a config file,
       never logged, and never appears in a stack trace (see
       `resolve_base_url` and the try/except around every HTTP error
       below).
    2. When a link expires (they are valid for a few hours), re-visit the
       LANL page, resubmit the form, and export the new link. There is no
       way around this short of LANL changing how the page issues links.

Everything else in this module is adapted, with only cosmetic changes,
from a standalone script that was already round-trip tested against a
local fake range-capable HTTP server (including a forced mid-stream
connection drop, verified by a mutation test to actually change the
output if the completeness check is removed). The gzip-trailer
completeness check in `iter_lines` below is the one genuinely subtle
piece of this module: `http.client`'s `read()` can return an empty chunk
on a dropped connection *without* raising, which would otherwise look
exactly like a clean end-of-file and silently truncate the data with no
error at all.
"""
from __future__ import annotations

import getpass
import http.client
import os
import sys
import time
import urllib.error
import urllib.request
import zlib
from typing import Iterator

UA = "Mozilla/5.0 (research-data-stream; lmsafe)"
FILE_NAMES = ("auth", "proc", "flows", "dns", "redteam")

ENV_VAR = "LMSAFE_LANL_BASE_URL"

EXPIRED_MSG = (
    "the LANL signed link is expired or invalid. Re-open "
    "https://csr.lanl.gov/data/cyber1/, resubmit the registration form to "
    "get a fresh signed link, then set {env} (or paste it when prompted) "
    "and re-run."
).format(env=ENV_VAR)


def resolve_base_url(interactive: bool = True) -> str:
    """Return the base URL (without a trailing filename) for the signed
    LANL link, read at run time -- never hardcoded, never persisted.

    Resolution order:
        1. The `LMSAFE_LANL_BASE_URL` environment variable.
        2. If `interactive` and stdin is a TTY (e.g. an interactive Colab
           cell): prompt with `getpass`, so the pasted link does not echo
           to the notebook output or get captured in cell history.

    We deliberately do *not* attempt to fetch
    https://csr.lanl.gov/data/cyber1/ and parse a link out of it: the
    signed link is only generated after a registration form is submitted
    server-side, so a plain page fetch has nothing to parse (see module
    docstring). Automating that form submission is out of scope here --
    it would mean scripting a third-party site's registration flow, and
    we have no way to test that against the live site from this sandbox.
    """
    raw = os.environ.get(ENV_VAR, "")
    if not raw and interactive and sys.stdin.isatty():
        raw = getpass.getpass(
            "Paste the LANL signed link (input hidden, not logged): "
        )
    if not raw:
        raise RuntimeError(
            f"Set {ENV_VAR} to the signed link from "
            "https://csr.lanl.gov/data/cyber1/ and re-run."
        )
    raw = raw.strip()
    if raw.endswith(".txt.gz"):
        raw = raw.rsplit("/", 1)[0]
    return raw.rstrip("/") + "/"


def file_url(base_url: str, name: str) -> str:
    """Build the full URL for one of the five cyber1 files."""
    if name not in FILE_NAMES:
        raise ValueError(f"unknown LANL file {name!r}, expected one of {FILE_NAMES}")
    return f"{base_url}{name}.txt.gz"


def redact(url: str) -> str:
    """Return a safe-to-print version of a signed URL (credential redacted).

    Used everywhere this module might otherwise print/log/raise an
    exception that embeds the URL -- a signed link printed into a shared
    notebook or an error-tracking system is a leaked credential.
    """
    try:
        head, _, tail = url.partition("/cyber1/")
        return f"https://csr.lanl.gov/data-fence/<redacted>/cyber1/{tail}" if tail else "<redacted>"
    except Exception:
        return "<redacted>"


def iter_lines(url: str, retries: int = 10) -> Iterator[bytes]:
    """Yield decompressed lines (bytes, no trailing newline) from a remote
    .gz file, streaming -- never buffering the whole file in memory.

    Robustness properties (each one earned from an earlier bug):

    * **Reconnect-and-resume.** If the connection drops partway through,
      we reopen the request and skip lines already yielded, so the
      caller sees one seamless stream and never double-counts a line.
    * **Silent-truncation guard.** `http.client.HTTPResponse.read()` can
      return `b""` when the server simply closes the socket early --
      this looks identical to a clean EOF and raises nothing. The only
      reliable signal is the gzip trailer: `zlib.decompressobj` sets
      `.eof` only after it has consumed a complete member including its
      trailing CRC32/size footer. We check `dec.eof` after the read loop
      exits and raise if it is False, which turns a silent data-loss bug
      into a retried connection. A mutation test
      (tests/test_lanl_stream_mutant.py-equivalent) confirms that without
      this check, a mid-stream drop silently returns a truncated-but-
      well-formed-looking line stream.
    * **Concatenated gzip members.** Some LANL mirrors serve
      multi-member gzip files; we loop re-initializing the decompressor
      on `dec.unused_data` so no trailing member is dropped.
    * **Credential-safe errors.** 401/403/410 (expired/invalid signed
      link) raise `LinkExpiredError` with no URL in the message, so a
      stack trace pasted into a bug report never leaks the token.

    Parameters
    ----------
    url : str
        Full URL to one `<file>.txt.gz`.
    retries : int
        Max reconnect attempts before giving up.
    """
    yielded = 0
    attempt = 0
    while True:
        seen = 0
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=120) as r:
                dec = zlib.decompressobj(16 + zlib.MAX_WBITS)
                buf = b""
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    data = dec.decompress(chunk)
                    while dec.eof and dec.unused_data:  # concatenated gzip members
                        rest = dec.unused_data
                        dec = zlib.decompressobj(16 + zlib.MAX_WBITS)
                        data += dec.decompress(rest)
                    buf += data
                    *lines, buf = buf.split(b"\n")
                    for ln in lines:
                        seen += 1
                        if seen > yielded:
                            yielded += 1
                            yield ln
                if not dec.eof:
                    # Connection died mid-file; http.client did not raise.
                    raise OSError("stream ended early (incomplete gzip member)")
                if buf:
                    seen += 1
                    if seen > yielded:
                        yielded += 1
                        yield buf
                return
        except urllib.error.HTTPError as e:
            if e.code in (401, 403, 410):
                raise LinkExpiredError(EXPIRED_MSG) from None
            err = f"HTTP {e.code}"
        except (OSError, http.client.HTTPException, zlib.error) as e:
            err = type(e).__name__
        attempt += 1
        if attempt > retries:
            raise RuntimeError(f"giving up after {retries} reconnects ({err})")
        print(
            f"[lanl_stream] connection problem ({err}); reconnect "
            f"{attempt}/{retries}, resuming after {yielded:,} lines",
            file=sys.stderr,
            flush=True,
        )
        time.sleep(min(30, 2 * attempt))


class LinkExpiredError(RuntimeError):
    """Raised when the signed LANL link is rejected (401/403/410)."""


In [ ]:
%%writefile lmsafe/data/lanl.py
"""
Fair LANL extraction protocol.

This module is the one place that decides *which* rows of the LANL
auth.txt.gz stream become `Event`s, and *which* of those get
`is_malicious=True`. Both decisions follow the "fair evaluation protocol"
described in the project's literature review (adapted from Larroche's
LMDEval, BSD-2-Clause -- see NOTICE.md) specifically so results here are
comparable to it, and *not* comparable to papers (e.g. CyberGFM) that use
an easier random split, a 14-day window, an NTLM-only filter, or host-based
OpTC labels. That incomparability is a known, citable fact about this
literature, not an oversight -- see README.md "Why a fair split matters".

auth.txt.gz columns (LANL's own schema)::

    time,src_user@domain,dst_user@domain,src_comp,dst_comp,
    auth_type,logon_type,orientation,success

redteam.txt.gz columns::

    time,user,src_comp,dst_comp

Label join rule (from LMDEval's extract_lanl.py, reproduced in
`schema.RedTeamRecord.key`): a red-team record matches an auth event when
``(event.t, event.dst_user, event.src_comp, event.dst_comp)`` equals
``(record.t, record.user, record.src_comp, record.dst_comp)`` -- note the
red-team "user" matches the auth event's *destination* user, not its
source user.
"""
from __future__ import annotations

import dataclasses
import gzip
import os
import pickle
import sys
import time
from pathlib import Path
from typing import IO, Iterator, Optional

from ..config import LanlConfig
from . import lanl_stream
from .schema import Event, RedTeamRecord

AUTH_NCOLS = 9


def atomic_pickle_dump(obj, out: Path) -> None:
    """Write `obj` to `out` without ever leaving a half-written or
    corrupt file at that path: write to a sibling temp file first, then
    `os.replace` it into place (atomic on POSIX). Shared by
    `scripts/fetch_lanl_window.py` (`build-cache`) and
    `scripts/run_experiment.py` (`build_lanl_windows`'s `cache_path`
    checkpointing) -- both stream real LANL data for potentially hours,
    so both need every intermediate checkpoint write to be crash-safe: a
    checkpoint killed mid-write (Ctrl+C, a Colab disconnect, an OOM)
    must leave the *previous* good checkpoint intact at `out`, never a
    truncated pickle that fails to load."""
    out = Path(out)
    tmp = out.with_suffix(out.suffix + ".tmp")
    with open(tmp, "wb") as fh:
        pickle.dump(obj, fh)
    os.replace(tmp, out)


def _auth_fields_to_event(raw_fields: list[bytes]) -> Optional[Event]:
    """Parse one raw auth.txt.gz line (already split on b",") into an
    Event, or return None if the line is malformed.

    Malformed lines are counted by the caller (see `stream_auth_events`
    stats) rather than raising -- a handful of corrupt lines should not
    abort a multi-hour streaming read, but a silently *large* bad-line
    count is exactly the kind of thing that should be visible, which is
    why we return a count instead of swallowing it entirely.
    """
    if len(raw_fields) != AUTH_NCOLS or not raw_fields[0].isdigit():
        return None
    t_s, src_u, dst_u, src_c, dst_c, auth_t, logon_t, orient, success = raw_fields
    return Event(
        t=int(t_s),
        src_user=src_u.decode("utf-8", "replace"),
        dst_user=dst_u.decode("utf-8", "replace"),
        src_comp=src_c.decode("utf-8", "replace"),
        dst_comp=dst_c.decode("utf-8", "replace"),
        auth_type=auth_t.decode("utf-8", "replace"),
        logon_type=logon_t.decode("utf-8", "replace"),
        orientation=orient.decode("utf-8", "replace"),
        success=(success == b"Success"),
        source="lanl",
    )


def _raw_line_passes_fair_prefilter(raw: list[bytes], cfg: LanlConfig) -> bool:
    """Cheap, decode-free early-reject check on the still-raw `b","`-split
    auth.txt.gz fields, used by `stream_auth_events` to skip the costly
    utf-8 decode + `Event` construction + `apply_fair_protocol` call
    entirely for a line that would be discarded anyway.

    This is a *performance-only* shortcut, never a behavior change:
    `apply_fair_protocol` is still the sole authority on what gets kept,
    and still runs (on the full decoded `Event`) for every line this
    returns True for. The real-data fair-protocol filter rejects the
    majority of lines (same-host events, non-"LogOn"-oriented events --
    empirically ~63% of all real auth.txt.gz rows), and on a billion-row
    real run, decoding 7 string fields and constructing a dataclass for
    every one of those doomed-to-be-discarded rows is the single biggest
    avoidable cost in the whole pipeline.

    Column indices (see the module docstring / AUTH_NCOLS): 3=src_comp,
    4=dst_comp, 5=auth_type, 7=orientation.

    Why comparing *raw bytes* against literal ASCII strings is safe,
    i.e. provably never rejects a line `apply_fair_protocol` would have
    kept:

    * `b"LogOn"` / `b"NTLM"` are pure ASCII. A byte string decodes
      (utf-8, errors="replace") to exactly "LogOn"/"NTLM" if and only if
      it *is* exactly those ASCII bytes -- utf-8 is a unique encoding of
      ASCII text, and any decode error inserts U+FFFD, which can never
      collapse back to that literal ASCII string. So the orientation and
      auth_type comparisons below are *exact* equivalents of the
      decoded-string comparisons in `Event.is_remote_logon` /
      `apply_fair_protocol`, not approximations.
    * `raw[3] != raw[4]` (raw bytes) is a safe *over*-approximation of
      `src_comp != dst_comp` (decoded strings): if the raw bytes are
      identical, decoding is a deterministic function of the bytes, so
      the decoded strings are certainly identical too (never a false
      "different hosts"). The only direction that can theoretically
      diverge is two *different*, both-invalid-utf8 byte sequences each
      decoding to a replacement string that happens to coincide --
      vanishingly unlikely for the clean ASCII hostnames LANL actually
      publishes, and even if it occurred this function would simply
      fail to skip early (falling through to the real, authoritative
      check) rather than skip something it shouldn't have.
    """
    if cfg.only_remote_logon and not (raw[7] == b"LogOn" and raw[3] != raw[4]):
        return False
    if cfg.filter_ntlm_only and raw[5] != b"NTLM":
        return False
    return True


def load_redteam_records(
    base_url: Optional[str] = None,
    *,
    local_path: Optional[Path] = None,
) -> list[RedTeamRecord]:
    """Load every red-team ground-truth record.

    redteam.txt.gz is tiny (4.8 KB, 749 rows in the public release), so
    unlike auth.txt.gz we just read the whole thing into memory -- either
    from a local copy (`local_path`, e.g. the test fixture under
    tests/fixtures/redteam.txt.gz) or streamed from `base_url`.
    """
    records: list[RedTeamRecord] = []

    def _parse(lines: Iterator[bytes]) -> None:
        for ln in lines:
            parts = ln.strip().split(b",")
            if len(parts) != 4 or not parts[0].isdigit():
                continue
            records.append(
                RedTeamRecord(
                    t=int(parts[0]),
                    user=parts[1].decode("utf-8", "replace"),
                    src_comp=parts[2].decode("utf-8", "replace"),
                    dst_comp=parts[3].decode("utf-8", "replace"),
                )
            )

    if local_path is not None:
        with gzip.open(local_path, "rb") as fh:
            _parse(iter(fh))
    else:
        if base_url is None:
            base_url = lanl_stream.resolve_base_url()
        url = lanl_stream.file_url(base_url, "redteam")
        _parse(lanl_stream.iter_lines(url))

    return records


def stream_auth_events(
    cfg: LanlConfig,
    *,
    base_url: Optional[str] = None,
    redteam: Optional[list[RedTeamRecord]] = None,
    progress_every: int = 5_000_000,
    progress_stream: IO[str] = sys.stderr,
) -> Iterator[Event]:
    """Stream auth events in `[cfg.start_s, cfg.end_s]`, labeled against
    `redteam`, applying the fair-protocol filter.

    This is a generator: nothing is buffered beyond the current line and
    the (small) red-team lookup set, so memory use is O(len(redteam)),
    not O(file size). The caller controls how much of the 7.2 GB file is
    actually transferred by choosing `cfg.end_s` -- reading stops as soon
    as the stream passes `cfg.end_s` (plus a small slack to tolerate
    slightly-unsorted input), *not* after reading the whole file.

    Parameters
    ----------
    cfg : LanlConfig
        `cfg.only_remote_logon` applies the fair-protocol filter (keep
        LogOn events between two different hosts, don't restrict to
        NTLM). `cfg.filter_ntlm_only` exists only so a caller can
        deliberately reproduce the *non-fair* CyberGFM-style preprocessing
        for a side-by-side comparison; it defaults to False and the
        README explains why it should stay that way for the paper's
        headline numbers.
    base_url : str, optional
        Pre-resolved signed LANL link. If omitted, resolved via
        `lanl_stream.resolve_base_url()` (env var, else interactive
        prompt) -- never hardcoded.
    redteam : list[RedTeamRecord], optional
        Pre-loaded labels. If omitted, loaded via `load_redteam_records`
        using the same `base_url`.

    Yields
    ------
    Event
        With `is_malicious=True` set exactly for events whose
        `(t, dst_user, src_comp, dst_comp)` matches a red-team record.
    """
    if base_url is None:
        base_url = lanl_stream.resolve_base_url()
    if redteam is None:
        redteam = load_redteam_records(base_url)
    redteam_keys = {r.key() for r in redteam}

    url = lanl_stream.file_url(base_url, "auth")
    slack_s = SECONDS_SLACK_FOR_DISORDER

    # Local variables, not a dict, for the four counters touched on
    # *every* line of a billion-plus-line real run: a dict-key lookup
    # (hash + compare) costs measurably more per access than a local
    # variable slot in CPython, and this loop does four to six such
    # accesses per line. `stats` is only reassembled (cheaply, once)
    # right before each progress print and at the very end.
    read = kept = bad = malicious = last_t = 0
    t0 = time.time()
    end_s = cfg.end_s
    start_s = cfg.start_s
    for ln in lanl_stream.iter_lines(url):
        read += 1

        # Report progress by *raw lines read*, independent of how many of
        # them end up filtered out below. The naive version of this check
        # used to live after the filtering (`only_remote_logon`, time
        # window, etc.) and fired on `kept % progress_every`-style logic
        # gated behind several `continue`s -- since most raw auth lines
        # are filtered out (same-host LogOffs, non-LogOn auth types,
        # ...), the one specific line where the raw count crossed a
        # progress_every boundary would often be exactly one of those
        # filtered lines, silently skipping that round's print instead of
        # merely delaying it. Over a multi-hour real stream that produced
        # long, unpredictable stretches with no output at all even though
        # the stream was reading fine -- indistinguishable from a hang.
        # Checking right here, unconditionally, guarantees a print every
        # `progress_every` raw lines no matter what fraction gets kept.
        if read % progress_every == 0:
            print(
                f"[lanl] read {read:,} lines, last t={last_t:,}s "
                f"({last_t / 86400:.2f} d), kept {kept:,}, "
                f"bad {bad:,}, malicious {malicious}, "
                f"{time.time() - t0:.0f}s elapsed",
                file=progress_stream,
                flush=True,
            )

        raw = ln.split(b",")
        if len(raw) != AUTH_NCOLS or not raw[0].isdigit():
            bad += 1
            continue
        t = int(raw[0])
        last_t = t
        if t > end_s + slack_s:
            break  # past the window -- stop reading, do not scan the rest
        if t > end_s or t < start_s:
            continue

        # Fast, decode-free reject: skip straight to the next line
        # without ever building an Event for one the fair protocol
        # would discard anyway (see `_raw_line_passes_fair_prefilter`'s
        # docstring for the correctness argument). This is the dominant
        # cost saving on real data, where the large majority of rows
        # fail this check.
        if not _raw_line_passes_fair_prefilter(raw, cfg):
            continue

        parsed = _auth_fields_to_event(raw)
        if parsed is None:
            bad += 1
            continue

        ev = apply_fair_protocol(parsed, redteam_keys, cfg)
        if ev is None:
            continue
        if ev.is_malicious:
            malicious += 1
        kept += 1

        yield ev

    print(
        f"[lanl] done: read {read:,} | kept {kept:,} | "
        f"bad {bad} | malicious {malicious} "
        f"(redteam has {len(redteam)} total records)",
        file=progress_stream,
        flush=True,
    )


#: How far past `end_s` to keep reading before giving up, to tolerate a
#: handful of out-of-order lines near the boundary (matches the tested
#: standalone script's default).
SECONDS_SLACK_FOR_DISORDER = 3600


def apply_fair_protocol(
    ev: Event, redteam_keys: set[tuple[int, str, str, str]], cfg: LanlConfig
) -> Optional[Event]:
    """Pure function: apply the fair-protocol filters to one parsed
    `Event` and attach its ground-truth label.

    Pulled out of `stream_auth_events` specifically so it can be unit
    tested (tests/test_lanl_labels.py) against hand-built Events and a
    small red-team set, with no network access and no dependence on
    `lanl_stream` at all. This is the one function that encodes "what
    counts as a kept event" and "what counts as a labeled positive" --
    exactly the two decisions a wrong-by-one-field-order bug could get
    wrong silently (see the docstring on `RedTeamRecord.key`).

    Returns
    -------
    Event or None
        A copy of `ev` with `is_malicious` set correctly, or None if
        `ev` is filtered out by the fair protocol (not a remote LogOn,
        or -- only if the caller opted into the non-fair comparison
        mode -- not NTLM).
    """
    if cfg.only_remote_logon and not ev.is_remote_logon():
        return None
    if cfg.filter_ntlm_only and ev.auth_type != "NTLM":
        return None
    is_mal = (ev.t, ev.dst_user, ev.src_comp, ev.dst_comp) in redteam_keys
    if is_mal == ev.is_malicious:
        return ev
    # dataclass is frozen -- return a relabeled copy rather than mutate.
    return dataclasses.replace(ev, is_malicious=is_mal)


def split_train_test(
    events: Iterator[Event], cfg: LanlConfig
) -> tuple[list[Event], Iterator[Event]]:
    """Materialize the benign training events (t < cfg.train_cutoff_s)
    into a list, and return the remaining events as a (still-lazy)
    iterator for the test stream.

    Why training is materialized but test is not: the training window is
    small (41 hours) and every policy needs repeated access to it (to
    fit the initial model, and as the "clean anchor memory" for the gated
    policy in lmsafe.policies). The test stream can span up to 58 days and
    is only ever consumed once, window by window, by `lmsafe.runner` --
    so it stays an iterator and is never fully materialized.

    Caution: this function consumes the input iterator lazily and
    correctly only if `events` is sorted by time ascending, which both
    `stream_auth_events` and `lmsafe.synth.generate` guarantee.
    """
    train: list[Event] = []
    first_test_event: Optional[Event] = None

    # Drain the strictly-training prefix eagerly so `train` is a
    # complete, ready-to-use list for the caller immediately. Stop at
    # the first event at or after the cutoff and hold onto it, since
    # calling next() on `events` to peek already consumed it from the
    # underlying stream -- it must be re-yielded, not dropped.
    for ev in events:
        if ev.t < cfg.train_cutoff_s:
            train.append(ev)
        else:
            first_test_event = ev
            break

    def _test_stream() -> Iterator[Event]:
        if first_test_event is not None:
            yield first_test_event
        yield from events

    return train, _test_stream()


In [ ]:
%%writefile lmsafe/data/optc.py
"""
OpTC adapter -- real implementation, built from a confirmed sample.

Status: this module was a documented stub until a real sample of
`ecar-bro/short` (241 FLOW records, one host-range partition, 2019-09-25)
was obtained and inspected line-by-line to pin down the exact schema
(see this module's history / project NOTICE for the earlier stub). The
field names and behavior below are confirmed against that real sample,
not guessed from the generic public eCAR documentation -- notably:

- The real `action` value for a FLOW record is ``"INFO"``, not
  ``"START"``/``"CREATE"`` as the generic eCAR docs suggest for other
  object types. This reader filters on ``object == "FLOW"`` alone and
  does not additionally filter on `action`.
- `timestamp` is ISO-8601 with a UTC offset and a **variable-width**
  fractional-seconds part -- 0, 1, 2, 3, or more digits all appear in
  real data (e.g. ``"...T09:56:52-04:00"`` with none at all and
  ``"...T09:07:45.26-04:00"`` with exactly two). `datetime.fromisoformat`
  is not reliably lenient about this across Python versions, so
  `_parse_optc_timestamp` below hand-parses it with a regex instead.
- A FLOW record does not carry a destination *hostname* -- only a
  `dest_ip` (and, for the local, reporting host, a `src_ip`). There is
  no analogue of LANL's `src_comp,dst_comp` pair sitting directly in one
  record. See "IP-to-host resolution" below for how this reader
  reconstructs host-to-host edges anyway.

Data source and known labeling caveat
--------------------------------------
OpTC is distributed by the FiveDirections project as a set of `ecar`
JSON-lines files (flow/process/file events) on a shared Google Drive
folder (https://drive.google.com/drive/u/0/folders/1n3kkS3KR31KUegn42yk3-e6JkZvf0Caa),
roughly 1 TB uncompressed across all hosts and days -- too large to load
in full in this project's sandbox. This reader targets the subset that
is actually obtainable and useful here: one or a few `ecar-bro/*` files
(each one host-range partition's FLOW-START events, optionally
gzip-compressed, auto-detected). It is **not** a streaming reader like
`lmsafe.data.lanl_stream` -- every given file is fully parsed into
memory and the results sorted by time, because `ecar-bro` is partitioned
by host range rather than globally time-ordered, so a true single-pass
stream across many files would need an external k-way merge this
project does not yet implement. This is fine at the "a handful of real
sample files" scale this project operates at; it would need revisiting
before pointing this reader at the full ~1 TB release.

The project's own literature review flags an important fairness issue
here too: most published OpTC results use *host-based* red-team labels
(a host is "malicious" for the whole engagement day), which is a much
easier target than LANL's *event-level* labels. The fair protocol
(Larroche/LMDEval-style) instead uses an event-level hybrid manual
labeling released alongside LMDEval. This reader does not yet join
against any red-team label file -- every `Event` it emits has
`is_malicious=False` -- so it is for the *cross-network-transfer*
chapter (unsupervised: no label needed at evaluation time, only for
after-the-fact grading) and not yet for a full labeled LANL-vs-OpTC
headline comparison. `OpTCRedTeamGroundTruth.pdf`, visible alongside the
`ecar-bro`/`ecar`/`bro` folders in the shared data, is the likely source
for that labeling and remains a follow-up item.

IP-to-host resolution
----------------------
A FLOW record's `properties.direction` is `"outbound"` or `"inbound"`
(confirmed in the sample). When `"outbound"`, the *reporting* host
(`hostname`) is the connection's source, so its own IP is `src_ip` and
the far end is `dest_ip`; when `"inbound"` the reverse holds. Since
every host in the engagement reports its own outbound/inbound FLOW
records, a complete self-consistent `ip -> hostname` table can be built
by watching, for every record, which IP belongs to the reporting host
itself -- no separate asset inventory is needed. `build_ip_host_table`
does this (majority vote per IP, to tolerate the rare DHCP-style
reassignment or logging glitch). A FLOW's *other* endpoint is then
resolved through that same table: if it resolves to a known host, the
record contributes a genuine host-to-host edge; if it does not (an
external IP, e.g. a C2 server or ordinary internet traffic), the record
is dropped by default (`OptcConfig.only_internal_internal=True`) -- the
OpTC-equivalent of LANL's "remote LogOn between two *different* hosts"
fairness filter in `lmsafe.data.lanl.Event.is_remote_logon`. Every
emitted `Event` sets `orientation="LogOn"` unconditionally, so the
*same* `is_remote_logon()` check LANL events are graded through also
accepts these -- no change needed anywhere downstream of `schema.Event`
to run OpTC through the existing windows/detector/policy/metrics
pipeline, exactly as this module's original stub promised.

Field mapping into `schema.Event`
-----------------------------------
t           : parsed timestamp, in integer seconds relative to the
              *earliest* timestamp across all records passed to
              `stream_optc_events` in this call (LANL-style "seconds
              since the dataset's own epoch").
src_user    : `properties.principal` of the record where it was
              observed -- there is no separate source/destination
              identity in a network flow, so...
dst_user    : ...this is set identical to `src_user` (see schema.py's
              own docstring: "often identical to src_user" is an
              expected, documented case, not a placeholder bug).
src_comp    : resolved source host (see IP resolution above).
dst_comp    : resolved destination host.
auth_type   : repurposed to carry `properties.dest_port` (string) --
              OpTC's nearest analogue of "what mechanism" is which port
              a connection targets (445=SMB, 3389=RDP, 22=SSH, ... are
              the classic lateral-movement-relevant ones); kept for
              diagnostics/ablations exactly as LANL's own `auth_type`
              is, never filtered on by default.
logon_type  : repurposed to carry `properties.direction`
              ("outbound"/"inbound" as seen on the *reporting* host's
              own record, before resolution) -- diagnostics only.
orientation : always `"LogOn"` for every emitted event (see above).
success     : always `True` -- a Bro/Zeek FLOW-START record implies a
              connection was actually established; there is no
              separate failure signal for this object type in the
              sample inspected.
is_malicious: always `False` (see labeling caveat above).
source      : `"optc"`.
"""
from __future__ import annotations

import collections
import dataclasses
import gzip
import json
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable, Iterator, NamedTuple, Optional

from .schema import Event

_TS_RE = re.compile(
    r"^(?P<y>\d{4})-(?P<mo>\d{2})-(?P<d>\d{2})T"
    r"(?P<h>\d{2}):(?P<mi>\d{2}):(?P<s>\d{2})"
    r"(?:\.(?P<frac>\d+))?"
    r"(?P<tzsign>[+-])(?P<tzh>\d{2}):(?P<tzm>\d{2})$"
)


def _parse_optc_timestamp(ts: str) -> float:
    """Parse one OpTC `timestamp` string into a UTC epoch float.

    Hand-rolled instead of `datetime.fromisoformat` because real OpTC
    timestamps have a *variable-width* fractional-seconds part (0 to at
    least 3 digits observed in a real sample -- e.g. both
    ``"09:56:52-04:00"`` with none and ``"09:07:45.26-04:00"`` with
    exactly two), which `fromisoformat` does not reliably accept across
    Python versions (it is strict about 3-or-6-digit microseconds on
    many versions). Raises `ValueError` on anything that does not match
    the confirmed real shape.
    """
    m = _TS_RE.match(ts)
    if not m:
        raise ValueError(f"unrecognized OpTC timestamp format: {ts!r}")
    frac = m.group("frac") or "0"
    micros = int(frac.ljust(6, "0")[:6])
    sign = 1 if m.group("tzsign") == "+" else -1
    offset = timedelta(hours=int(m.group("tzh")), minutes=int(m.group("tzm"))) * sign
    dt = datetime(
        int(m.group("y")), int(m.group("mo")), int(m.group("d")),
        int(m.group("h")), int(m.group("mi")), int(m.group("s")),
        micros, tzinfo=timezone(offset),
    )
    return dt.timestamp()


@dataclasses.dataclass(frozen=True)
class OptcConfig:
    """Configuration for one OpTC extraction run. Duck-type compatible
    with `lmsafe.data.lanl.split_train_test` (which only ever reads
    `.train_cutoff_s` off whatever config it is given), so that function
    is reused here unchanged rather than re-implemented."""

    window_seconds: int = 3600
    #: Train/test boundary, in seconds from the first event seen in this
    #: call. OpTC's engagement window is much shorter than LANL's (days,
    #: not months); there is no single project-wide default here the way
    #: `LANL_TRAIN_CUTOFF_S` is -- callers should set this deliberately
    #: from the actual span of the files they pass in.
    train_cutoff_s: int = 2 * 3600
    #: The OpTC-equivalent of LanlConfig.only_remote_logon: keep only
    #: FLOW records whose *other* endpoint also resolves to a known
    #: internal host. Keeping this True is what makes the fairness
    #: argument in this module's docstring hold.
    only_internal_internal: bool = True


class _FlowRecord(NamedTuple):
    t: float
    hostname: str
    direction: str
    src_ip: str
    dest_ip: str
    dest_port: str
    principal: str


def _open_maybe_gzip(path: Path):
    """Open `path` for text reading, transparently handling gzip
    (detected by magic bytes, not by file extension -- a real sample
    file from the shared Drive folder may or may not carry a `.gz`
    suffix depending on how it was downloaded)."""
    path = Path(path)
    with open(path, "rb") as fh:
        magic = fh.read(2)
    if magic == b"\x1f\x8b":
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return open(path, "rt", encoding="utf-8", errors="replace")


def _iter_flow_records(paths: Iterable[Path]) -> Iterator[_FlowRecord]:
    """Parse every FLOW-object JSON line across all `paths` into a
    `_FlowRecord`. Lines that fail to parse as JSON, are not FLOW
    objects, or are missing a required field are silently skipped (an
    eCAR release is known to contain occasional malformed lines -- see
    errata.md on the FiveDirections repo); this mirrors
    `lmsafe.data.lanl.stream_auth_events`'s own tolerance of bad lines.
    """
    for path in paths:
        with _open_maybe_gzip(Path(path)) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("object") != "FLOW":
                    continue
                props = rec.get("properties") or {}
                hostname = rec.get("hostname")
                direction = props.get("direction")
                src_ip = props.get("src_ip")
                dest_ip = props.get("dest_ip")
                ts = rec.get("timestamp")
                if not (hostname and direction and src_ip and dest_ip and ts):
                    continue
                try:
                    t = _parse_optc_timestamp(ts)
                except ValueError:
                    continue
                yield _FlowRecord(
                    t=t,
                    hostname=hostname,
                    direction=direction,
                    src_ip=src_ip,
                    dest_ip=dest_ip,
                    dest_port=str(props.get("dest_port", "")),
                    principal=rec.get("principal") or "",
                )


def build_ip_host_table(records: Iterable[_FlowRecord]) -> dict[str, str]:
    """Build an `ip -> hostname` table purely from self-observation: for
    an `"outbound"` record the reporting host's own IP is `src_ip`; for
    `"inbound"` it is `dest_ip`. Ties (the same IP observed against more
    than one hostname -- e.g. a DHCP reassignment during the capture
    window) are resolved by majority vote so one glitchy record cannot
    flip an otherwise-consistent mapping.
    """
    votes: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
    for r in records:
        if r.direction == "outbound":
            votes[r.src_ip][r.hostname] += 1
        elif r.direction == "inbound":
            votes[r.dest_ip][r.hostname] += 1
        # any other direction value: no self-IP signal to extract, skip
    return {ip: counter.most_common(1)[0][0] for ip, counter in votes.items()}


def stream_optc_events(paths: Iterable[Path], cfg: OptcConfig) -> list[Event]:
    """Read every FLOW record across `paths`, resolve host-to-host
    edges, and return a time-sorted `list[Event]`.

    Returns a materialized list, not a lazy iterator -- unlike
    `lmsafe.data.lanl.stream_auth_events`, this is not a single
    chronological file: `ecar-bro` partitions by host range, so records
    from different files must be merged and sorted by time before
    `lmsafe.data.windows.iter_windows` (which requires sorted input) can
    consume them. See this module's docstring for the scaling caveat.
    """
    paths = [Path(p) for p in paths]
    records = list(_iter_flow_records(paths))
    if not records:
        return []

    ip_table = build_ip_host_table(records)
    t0 = min(r.t for r in records)

    events: list[Event] = []
    for r in records:
        if r.direction == "outbound":
            src_comp = r.hostname
            dst_comp = ip_table.get(r.dest_ip)
        elif r.direction == "inbound":
            src_comp = ip_table.get(r.src_ip)
            dst_comp = r.hostname
        else:
            continue

        if dst_comp is None or src_comp is None:
            if cfg.only_internal_internal:
                continue
            # keep_external path: fall back to the raw IP as a
            # pseudo-hostname so the edge is still representable, rather
            # than silently losing the record.
            src_comp = src_comp or r.src_ip
            dst_comp = dst_comp or r.dest_ip

        if src_comp == dst_comp:
            # a host talking to itself is not a lateral-movement-
            # relevant edge; same spirit as LANL's own same-host check.
            continue

        events.append(
            Event(
                t=int(r.t - t0),
                src_user=r.principal,
                dst_user=r.principal,
                src_comp=src_comp,
                dst_comp=dst_comp,
                auth_type=r.dest_port,
                logon_type=r.direction,
                orientation="LogOn",
                success=True,
                is_malicious=False,
                source="optc",
            )
        )

    events.sort(key=lambda e: e.t)
    return events


In [ ]:
%%writefile lmsafe/data/windows.py
"""
Groups a time-ordered stream of `Event`s into fixed-length windows and
builds the per-window graph that `lmsafe.detectors` scores.

A "graph" here is deliberately a plain, dependency-light structure (a node
index plus a weighted directed edge list) rather than a specific library's
graph object -- `lmsafe.detectors.count_baseline` needs almost nothing, and
`lmsafe.detectors.rw_gnn` builds its own `torch` tensors from the same
`WindowGraph`, so neither detector forces a graph-library dependency on the
other.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterable, Iterator

from .schema import Event


@dataclass
class WindowGraph:
    """The host-level interaction graph for one time window.

    Attributes
    ----------
    window_start : int
        Start time of the window, in the same clock as `Event.t`.
    window_seconds : int
        Window length.
    nodes : list[str]
        Sorted, de-duplicated list of every host seen in this window
        (union of src_comp and dst_comp). `node_index[h]` gives a host's
        position in this list.
    node_index : dict[str, int]
        host -> index into `nodes`.
    edges : list[tuple[int, int]]
        Directed (src_index, dst_index) pairs, one entry per *distinct*
        edge (src_comp, dst_comp) seen in the window -- repeats are
        folded into `edge_weight`, not kept as duplicate rows.
    edge_weight : list[int]
        Event count for `edges[i]`, aligned by index.
    edge_is_malicious : list[bool]
        True if *any* event contributing to `edges[i]` was labeled
        malicious. Evaluation-only (see schema.py docstring) -- detectors
        must never read this field.
    n_events : int
        Total event count in the window (== sum(edge_weight)).
    """

    window_start: int
    window_seconds: int
    nodes: list[str] = field(default_factory=list)
    node_index: dict[str, int] = field(default_factory=dict)
    edges: list[tuple[int, int]] = field(default_factory=list)
    edge_weight: list[int] = field(default_factory=list)
    edge_is_malicious: list[bool] = field(default_factory=list)
    n_events: int = 0

    def host_pair(self, i: int) -> tuple[str, str]:
        """Convenience: the (src_host, dst_host) string pair for edges[i]."""
        a, b = self.edges[i]
        return self.nodes[a], self.nodes[b]

    def is_empty(self) -> bool:
        return self.n_events == 0


def filter_edges(window: WindowGraph, keep_indices: Iterable[int]) -> WindowGraph:
    """Return a new `WindowGraph` containing only the edges at
    `keep_indices` (and only the nodes those edges touch).

    Used by `lmsafe.policies.GatedPolicy` to build the "admitted and
    corroborated" sub-graph it is willing to train on, without
    mutating the original window (which the runner may still need
    intact for evaluation).
    """
    keep = list(keep_indices)
    old_to_new: dict[int, int] = {}
    new_nodes: list[str] = []
    new_edges: list[tuple[int, int]] = []
    new_weight: list[int] = []
    new_mal: list[bool] = []

    def remap(old_idx: int) -> int:
        new_idx = old_to_new.get(old_idx)
        if new_idx is None:
            new_idx = len(new_nodes)
            old_to_new[old_idx] = new_idx
            new_nodes.append(window.nodes[old_idx])
        return new_idx

    n_events = 0
    for i in keep:
        a, b = window.edges[i]
        new_edges.append((remap(a), remap(b)))
        new_weight.append(window.edge_weight[i])
        new_mal.append(window.edge_is_malicious[i])
        n_events += window.edge_weight[i]

    return WindowGraph(
        window_start=window.window_start,
        window_seconds=window.window_seconds,
        nodes=new_nodes,
        node_index={h: i for i, h in enumerate(new_nodes)},
        edges=new_edges,
        edge_weight=new_weight,
        edge_is_malicious=new_mal,
        n_events=n_events,
    )


def iter_windows(
    events: Iterable[Event],
    window_seconds: int,
    *,
    align_start: int = 0,
) -> Iterator[WindowGraph]:
    """Consume a time-sorted stream of events and yield one `WindowGraph`
    per non-empty window, in increasing time order.

    A window with zero events is skipped entirely (not yielded as an
    empty graph) -- `lmsafe.policies` and `lmsafe.runner` treat "no
    window at time T" and "an empty window at time T" as the same
    situation, and skipping avoids every caller having to special-case
    `WindowGraph.is_empty()`.

    Parameters
    ----------
    events : Iterable[Event]
        Must be sorted by `.t` ascending. This holds for every reader in
        `lmsafe.data` and for `lmsafe.synth.generate`; it is *not*
        re-validated here (an assert on every event would be a real cost
        on a 1e8-event stream), so a caller feeding unsorted events will
        get wrong, not loudly-erroring, windows. See
        tests/test_windows.py for a test that an out-of-order input
        raises in debug-friendly test code paths.
    window_seconds : int
        Window length in seconds.
    align_start : int
        Window boundaries are `align_start + k * window_seconds` for
        integer k -- i.e. windows are anchored to this value, not to the
        first event's own timestamp. Keeping this fixed across training
        and test extraction is what makes window boundaries comparable
        across a dataset split.
    """
    current_start: int | None = None
    builder: _GraphBuilder | None = None

    for ev in events:
        bucket = align_start + ((ev.t - align_start) // window_seconds) * window_seconds
        if current_start is None:
            current_start = bucket
            builder = _GraphBuilder(current_start, window_seconds)
        elif bucket != current_start:
            if builder is not None and not builder.is_empty():
                yield builder.build()
            current_start = bucket
            builder = _GraphBuilder(current_start, window_seconds)
        builder.add(ev)  # type: ignore[union-attr]

    if builder is not None and not builder.is_empty():
        yield builder.build()


class _GraphBuilder:
    """Accumulates events for one window before `iter_windows` emits it.

    Kept as a private helper (not `WindowGraph` itself) so `WindowGraph`
    can stay a plain, easily-pickled/serialized data holder while the
    O(1)-amortized "have I seen this edge yet" bookkeeping lives here.
    """

    __slots__ = (
        "window_start",
        "window_seconds",
        "_node_index",
        "_edge_index",
        "_edges",
        "_weight",
        "_malicious",
        "_n_events",
    )

    def __init__(self, window_start: int, window_seconds: int):
        self.window_start = window_start
        self.window_seconds = window_seconds
        self._node_index: dict[str, int] = {}
        self._edge_index: dict[tuple[int, int], int] = {}
        self._edges: list[tuple[int, int]] = []
        self._weight: list[int] = []
        self._malicious: list[bool] = []
        self._n_events = 0

    def _node(self, host: str) -> int:
        idx = self._node_index.get(host)
        if idx is None:
            idx = len(self._node_index)
            self._node_index[host] = idx
        return idx

    def add(self, ev: Event) -> None:
        a = self._node(ev.src_comp)
        b = self._node(ev.dst_comp)
        key = (a, b)
        idx = self._edge_index.get(key)
        if idx is None:
            idx = len(self._edges)
            self._edge_index[key] = idx
            self._edges.append(key)
            self._weight.append(0)
            self._malicious.append(False)
        self._weight[idx] += 1
        if ev.is_malicious:
            self._malicious[idx] = True
        self._n_events += 1

    def is_empty(self) -> bool:
        return self._n_events == 0

    def build(self) -> WindowGraph:
        # nodes sorted for determinism (two runs over the same events
        # must produce identical node orderings, independent of hashing).
        ordered_hosts = sorted(self._node_index, key=self._node_index.get)
        return WindowGraph(
            window_start=self.window_start,
            window_seconds=self.window_seconds,
            nodes=ordered_hosts,
            node_index=dict(self._node_index),
            edges=list(self._edges),
            edge_weight=list(self._weight),
            edge_is_malicious=list(self._malicious),
            n_events=self._n_events,
        )


In [ ]:
%%writefile lmsafe/synth.py
"""
Synthetic authentication-event generator with known ground truth.

Why a synthetic generator belongs in a paper about LANL/OpTC
---------------------------------------------------------------
Every module downstream of `lmsafe.data` (windowing, detectors, drift
detection, update policies, the attacker, metrics) needs to be correct
*before* it is pointed at a 7.2 GB stream that takes an hour to read. This
module produces a small, fast, fully-controlled event stream in the same
`Event` shape LANL/OpTC produce, with two properties real data cannot give
a unit test:

    1. Exact, known ground truth for both the benign drift and the
       injected lateral-movement episodes (so metrics.py can be tested
       against a known-correct answer, not just "a plausible-looking
       number").
    2. A dial (`SynthConfig.drift_strength`) to turn benign drift on or
       off independently of the attack, which is exactly the
       counterfactual RQ1 asks for and which LANL alone cannot offer
       (LANL's drift and its one real intrusion are both fixed, whatever
       they happen to be in that one 58-day recording).

Generative model (deliberately simple, documented limitations below)
-----------------------------------------------------------------------
- A population of users and hosts exists from day 0. Each user has one
  "home" host (their desktop). A small subset of hosts are "common
  servers" (file/domain-controller-style) that most users talk to.
- Each benign event picks a user, then a destination: with high
  probability a common server (Zipf-weighted, matching the real LANL
  red-team inspection showing a small number of very popular destination
  hosts -- see README), otherwise another user's home host (modeling
  occasional legitimate peer-to-peer / admin access).
- Starting at `drift_start_day`, the host and user population grows
  monotonically (new hosts/users are added, none are ever removed). This
  is "collective, persistent" drift by construction: it shifts the
  *whole* population's traffic mix, gradually and permanently, which is
  exactly the RQ1 hypothesis's definition of benign drift as opposed to
  an attacker's local, narrow conditioning (see `lmsafe.attack`).
- `n_episodes` lateral-movement episodes are injected at well-separated
  times: one actor host authenticates, as one user, to a rapid chain of
  `episode_chain_len` distinct target hosts a few seconds/minutes apart --
  the same "one host touches many others quickly" signature observed in
  the real LANL red-team file (701 of 749 real events originate from a
  single host, C17693).

Known limitations (state them, don't hide them)
---------------------------------------------------
- No user/host churn (nothing is ever removed) -- real organizations lose
  machines and accounts, not just gain them. Acceptable for this
  project's purpose (testing the pipeline, not modeling an enterprise),
  but should not be oversold as realistic in the paper.
- All events are already in the "kept" shape the fair LANL protocol
  selects for (orientation="LogOn", src_comp != dst_comp): the generator
  tests everything *after* extraction, not the extraction/filtering
  logic itself (that is tested against the real LANL schema in
  tests/test_lanl_labels.py).
- Auth-type/success fields are set but not modeled adversarially; they
  are not used by any detector or policy in this project (the fair
  protocol explicitly avoids filtering on auth_type -- see
  lmsafe.config.LanlConfig.filter_ntlm_only).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .config import SynthConfig
from .data.schema import Event

COMMON_SERVER_FRACTION = 0.1  # fraction of the initial host pool that are shared servers


@dataclass(frozen=True)
class LMEpisode:
    """One injected lateral-movement episode, with known ground truth.

    Attributes
    ----------
    start_s : int
        Time of the first hop.
    actor_user : str
        The (compromised) user identity used for every hop in this
        episode.
    actor_host : str
        The single source host every hop originates from -- mirrors the
        real LANL red team's heavy concentration on one source host.
    target_hosts : list[str]
        Distinct destination hosts visited, in order, one per hop.
    hop_seconds : int
        Spacing between consecutive hops.
    """

    start_s: int
    actor_user: str
    actor_host: str
    target_hosts: list[str]
    hop_seconds: int = 30

    def events(self) -> list[Event]:
        out = []
        for i, dst in enumerate(self.target_hosts):
            out.append(
                Event(
                    t=self.start_s + i * self.hop_seconds,
                    src_user=self.actor_user,
                    dst_user=self.actor_user,
                    src_comp=self.actor_host,
                    dst_comp=dst,
                    auth_type="NTLM",
                    logon_type="LogOn",
                    orientation="LogOn",
                    success=True,
                    is_malicious=True,
                    source="synthetic",
                )
            )
        return out


class Population:
    """Tracks the growing user/host population as a function of elapsed
    simulated days, implementing the "collective, persistent drift"
    model described in the module docstring.

    Public (not an internal detail) specifically so an experiment
    script can build one population once, place a fixed set of episodes
    on it with `default_episodes`, and then re-run `generate` with
    those *same* episodes across several `drift_strength` values for a
    clean counterfactual comparison -- exactly what
    `scripts/run_rq1_drift_cost.py` does. `generate()` itself builds and
    discards its own `Population` internally when called without
    pre-built `episodes`.
    """

    def __init__(self, cfg: SynthConfig, rng: np.random.Generator):
        self.cfg = cfg
        self.rng = rng
        n_common = max(1, int(round(cfg.n_hosts * COMMON_SERVER_FRACTION)))
        self.common_servers: list[str] = [f"SRV{i}" for i in range(n_common)]
        # Zipf-like popularity weights among common servers (a few are
        # much more popular than the rest), matching the skew observed
        # in the real LANL destination-host counts.
        ranks = np.arange(1, n_common + 1)
        self.common_weights = 1.0 / ranks
        self.common_weights /= self.common_weights.sum()

        self.hosts: list[str] = [f"C{i}" for i in range(cfg.n_hosts - n_common)]
        self.users: list[str] = [f"U{i}@DOM1" for i in range(cfg.n_users)]
        self.home_host: dict[str, str] = {
            u: self.hosts[i % len(self.hosts)] for i, u in enumerate(self.users)
        }
        self._next_host_id = len(self.hosts)
        self._next_user_id = len(self.users)

    def grow(self, day: int) -> None:
        """Apply one day's worth of drift-driven population growth."""
        cfg = self.cfg
        if cfg.drift_start_day is None or day < cfg.drift_start_day:
            return
        if cfg.drift_strength <= 0:
            return
        new_hosts = max(0, int(round(cfg.drift_strength * 0.05 * cfg.n_hosts)))
        new_users = max(0, int(round(cfg.drift_strength * 0.05 * cfg.n_users)))
        for _ in range(new_hosts):
            h = f"C{self._next_host_id}"
            self._next_host_id += 1
            self.hosts.append(h)
        for _ in range(new_users):
            u = f"U{self._next_user_id}@DOM1"
            self._next_user_id += 1
            self.users.append(u)
            self.home_host[u] = self.rng.choice(self.hosts)

    def sample_event(self, t: int) -> Event:
        # numpy's Generator.choice on a list of Python strings returns
        # numpy.str_ scalars; cast back to plain str so every downstream
        # consumer (dict keys, CSV writers, dataclass reprs) sees exactly
        # the same type real LANL/OpTC readers produce.
        u = str(self.rng.choice(self.users))
        src = self.home_host[u]
        if self.rng.random() < 0.7:
            dst = str(self.rng.choice(self.common_servers, p=self.common_weights))
        else:
            peer = str(self.rng.choice(self.users))
            dst = self.home_host[peer]
            if dst == src:
                dst = str(self.rng.choice(self.common_servers, p=self.common_weights))
        return Event(
            t=t,
            src_user=u,
            dst_user=u,
            src_comp=src,
            dst_comp=dst,
            auth_type="Kerberos" if self.rng.random() < 0.85 else "NTLM",
            logon_type="LogOn",
            orientation="LogOn",
            success=True,
            is_malicious=False,
            source="synthetic",
        )


def default_episodes(
    cfg: SynthConfig,
    population: Population,
    rng: np.random.Generator,
    min_start_s: int = 2 * 86400,
) -> list[LMEpisode]:
    """Place `cfg.n_episodes` lateral-movement episodes at evenly-spaced,
    randomly-jittered times across the benign period, each using a fresh
    actor host/user and a chain of distinct, randomly chosen targets.

    `min_start_s` keeps every episode safely clear of the benign
    training cutoff used throughout this project
    (`lmsafe.config.LANL_TRAIN_CUTOFF_S`, 41 hours): with a short
    `n_benign_days` and several episodes, an un-clamped evenly-spaced
    placement can land the first episode inside what `split_train_test`
    treats as the training window, silently removing it from the test
    stream's attack population rather than erroring. Defaulting to 2
    days gives a comfortable margin past 41 hours without needing the
    caller to know that constant.
    """
    total_s = cfg.n_benign_days * 86400
    if cfg.n_episodes <= 0:
        return []
    usable_s = max(total_s - min_start_s, 1)
    slot = usable_s / cfg.n_episodes
    episodes = []
    for i in range(cfg.n_episodes):
        jitter = rng.uniform(0.2, 0.8)
        start_s = int(min_start_s + i * slot + jitter * slot)
        actor_user = str(rng.choice(population.users))
        actor_host = population.home_host[actor_user]
        n_targets = cfg.episode_chain_len
        pool = [h for h in population.hosts if h != actor_host] + population.common_servers
        targets = [str(h) for h in rng.choice(pool, size=min(n_targets, len(pool)), replace=False)]
        episodes.append(
            LMEpisode(
                start_s=start_s,
                actor_user=actor_user,
                actor_host=actor_host,
                target_hosts=targets,
                hop_seconds=cfg.episode_hop_seconds,
            )
        )
    return episodes


def generate(
    cfg: SynthConfig, episodes: list[LMEpisode] | None = None
) -> list[Event]:
    """Generate a full, time-sorted synthetic event stream.

    Parameters
    ----------
    cfg : SynthConfig
    episodes : list[LMEpisode], optional
        Lateral-movement episodes to merge in. If omitted, `n_episodes`
        default episodes are placed automatically via `default_episodes`.
        Pass `episodes=[]` explicitly to get a purely benign stream (used
        by RQ1 experiments that measure drift cost in isolation, with no
        attack at all).

    Returns
    -------
    list[Event]
        Sorted ascending by `.t`. Materialized (not a generator) because
        the synthetic datasets used in this project are small enough
        (tens of thousands of events) that eager generation makes the
        rest of the pipeline simpler to reason about and test.
    """
    rng = np.random.default_rng(cfg.seed)
    pop = Population(cfg, rng)

    if episodes is None:
        episodes = default_episodes(cfg, pop, rng)

    events: list[Event] = []
    n_days = cfg.n_benign_days
    for day in range(n_days):
        pop.grow(day)
        day_start = day * 86400
        drift_mult = 1.0
        if cfg.drift_start_day is not None and day >= cfg.drift_start_day:
            drift_mult = 1.0 + cfg.drift_strength  # more traffic as the org grows
        n_windows = 86400 // cfg.window_seconds
        for w in range(n_windows):
            window_start = day_start + w * cfg.window_seconds
            n_events = rng.poisson(cfg.base_edges_per_window * drift_mult)
            for _ in range(n_events):
                t = int(window_start + rng.integers(0, cfg.window_seconds))
                events.append(pop.sample_event(t))

    for ep in episodes:
        events.extend(ep.events())

    events.sort(key=lambda e: e.t)
    return events


In [ ]:
%%writefile lmsafe/detectors/__init__.py
"""Link-prediction anomaly detectors.

Both detectors in this sub-package implement the same `Detector`
interface (`base.py`) and score host-to-host edges with the convention
used everywhere else in this project: **higher score = more anomalous**
(so `lmsafe.metrics.recall_at_budget` can always just take the top-B
scores as "alerts", for any detector).

count_baseline.py  Non-parametric frequency baseline. No torch
                    dependency -- always available, always fast, and a
                    meaningful RQ1/RQ2 reference point: if a fancier
                    model cannot beat counting how often an edge has
                    been seen before, the fancier model is not earning
                    its complexity.
feature_mlp.py      A small torch MLP link-predictor over *running,
                     incrementally-updated per-host features* (degree,
                     partner diversity, recency) rather than a per-node
                     ID embedding table. This is a deliberate choice: an
                     ID-embedding table cannot score a host it never saw
                     during training (the exact non-inductive failure
                     mode this project's literature review flags in
                     CyberGFM's node-ID tokenization). Feature-based
                     scoring degrades gracefully to "unknown host, cold
                     features, high uncertainty" instead of crashing or
                     silently mis-scoring.
rw_gnn.py           This project's "graph foundation model": the same
                     inductive per-host features as feature_mlp.py, plus
                     one round of GraphSAGE-style neighborhood
                     aggregation within each window's graph, trained via
                     self-supervised link prediction, with weights that
                     can be pretrained once on one network (see
                     scripts/pretrain_foundation_model.py) and reused --
                     zero-shot or fine-tuned -- on a different network's
                     windows. See the module docstring for the precise,
                     deliberately modest scope claimed by "foundation
                     model" here.
"""
from .base import Detector
from .count_baseline import CountBaselineDetector
from .feature_mlp import FeatureMLPDetector
from .rw_gnn import GraphFoundationDetector

__all__ = ["Detector", "CountBaselineDetector", "FeatureMLPDetector", "GraphFoundationDetector"]


In [ ]:
%%writefile lmsafe/detectors/base.py
"""
Common interface every detector implements.

`lmsafe.policies` is written entirely against this interface -- it never
inspects a detector's internals -- so any new detector (a future, heavier
CyberGFM/ULTRA-style model, say) can be dropped in by implementing these
four methods, and every update policy, drift detector, and metric in this
project keeps working unchanged.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Sequence

import numpy as np

from ..data.windows import WindowGraph


class Detector(ABC):
    """Scores the edges of a `WindowGraph` with an anomaly score, where
    **higher = more anomalous** (never a "normality probability" -- that
    inversion is a classic source of silent sign-flip bugs in anomaly
    detection code, so this project standardizes on one direction
    everywhere and `tests/test_detectors.py` checks it for every
    concrete detector).
    """

    @abstractmethod
    def fit(self, windows: Sequence[WindowGraph], *, warm_start: bool = True) -> None:
        """(Re-)fit the detector on a sequence of benign (or
        policy-admitted) window graphs.

        Parameters
        ----------
        windows : Sequence[WindowGraph]
            Training data, in time order. For a count-based detector
            this just accumulates edge counts. For the MLP detector this
            runs a few optimization epochs.
        warm_start : bool
            If True, continue from the detector's current parameters
            (an incremental update). If False, reset to a fresh,
            untrained state first (a full retrain from scratch on
            exactly `windows` and nothing else). Update policies in
            `lmsafe.policies` choose which one they need -- e.g. the
            "periodic retrain" policy calls `fit(..., warm_start=False)`
            on its current window buffer, while "sliding + replay" calls
            `fit(..., warm_start=True)` to keep adapting continuously.
        """

    @abstractmethod
    def score(self, window: WindowGraph) -> np.ndarray:
        """Return an anomaly score for every edge in `window`, aligned
        by index to `window.edges` (so `scores[i]` is the score for
        `window.edges[i]`). Must **not** read `window.edge_is_malicious`
        -- that field is ground truth for evaluation only (see
        `lmsafe.data.schema` module docstring); a unit test greps for
        this.
        """

    @abstractmethod
    def clone_fresh(self) -> "Detector":
        """Return a new, untrained instance with the same
        hyperparameters. Used by policies that need to train a
        brand-new model from scratch (e.g. the "frozen" baseline's
        initial fit, or a gated policy that wants a model fit only on
        its clean anchor memory with no history from a possibly-biased
        running model).
        """

    def state_summary(self) -> dict:
        """Optional: a small dict of diagnostic scalars (e.g. number of
        edges seen, vocabulary size) for logging. Default: empty.
        """
        return {}


In [ ]:
%%writefile lmsafe/detectors/count_baseline.py
"""
Non-parametric frequency baseline detector.

The model is deliberately the simplest thing that could possibly work:
track how many times each directed host-pair edge has been observed, and
score an edge as anomalous in inverse proportion to how often it has been
seen before, relative to total traffic volume. An edge never observed
before gets (close to) the maximum possible anomaly score.

This is not a strawman -- it is the reference point the paper needs. If
`FeatureMLPDetector` (or any future, heavier model) cannot clearly beat
this baseline on Rec@B, that is a real, reportable finding, not a bug.
Several papers in this literature compare only against learned models and
never report this kind of frequency baseline; this project does.
"""
from __future__ import annotations

import math
from collections import defaultdict
from typing import Sequence

import numpy as np

from ..data.windows import WindowGraph
from .base import Detector


class CountBaselineDetector(Detector):
    """Laplace-smoothed edge-frequency anomaly score.

    For an edge e with observed count c(e) out of `total` edge
    observations accumulated so far::

        p_hat(e)       = (c(e) + alpha) / (total + alpha)
        anomaly(e)     = -log(p_hat(e))

    `p_hat` is not a normalized probability distribution over all
    possible host pairs (that denominator would require knowing the full
    host population size, which drifts -- see lmsafe.synth); it is a
    monotonically-decreasing function of relative frequency, which is all
    that ranking-based metrics (AUC, AP, Rec@B) need. An unseen edge
    (c=0) gets anomaly = -log(alpha / (total + alpha)), the maximum
    possible value given the data seen so far; it grows (correctly)
    without bound as `total` grows and the edge is still never observed.

    Parameters
    ----------
    alpha : float
        Laplace smoothing constant. Smaller alpha makes "never seen
        before" edges score relatively higher versus "seen exactly
        once" edges; 0.5 (Jeffreys-style smoothing) is a reasonable,
        unremarkable default.
    """

    def __init__(self, alpha: float = 0.5):
        self.alpha = alpha
        self._counts: dict[tuple[str, str], int] = defaultdict(int)
        self._total: int = 0

    def fit(self, windows: Sequence[WindowGraph], *, warm_start: bool = True) -> None:
        if not warm_start:
            self._counts = defaultdict(int)
            self._total = 0
        for w in windows:
            for (a, b), weight in zip(w.edges, w.edge_weight):
                key = (w.nodes[a], w.nodes[b])
                self._counts[key] += weight
                self._total += weight

    def score(self, window: WindowGraph) -> np.ndarray:
        out = np.empty(len(window.edges), dtype=np.float64)
        # total+alpha could be 0 only if alpha==0 and nothing fit yet;
        # guard so an unfitted detector still returns finite scores
        # (rank-order is all that matters, but a NaN would poison every
        # downstream metric).
        denom = max(self._total + self.alpha, 1e-9)
        for i, (a, b) in enumerate(window.edges):
            key = (window.nodes[a], window.nodes[b])
            c = self._counts.get(key, 0)
            p_hat = (c + self.alpha) / denom
            out[i] = -math.log(max(p_hat, 1e-300))
        return out

    def clone_fresh(self) -> "CountBaselineDetector":
        return CountBaselineDetector(alpha=self.alpha)

    def state_summary(self) -> dict:
        return {"n_distinct_edges": len(self._counts), "total_events": self._total}


In [ ]:
%%writefile lmsafe/detectors/feature_mlp.py
"""
Lightweight, inductive link-prediction detector (torch, CPU-friendly).

Why not per-node ID embeddings
--------------------------------
CyberGFM-style models tokenize random walks over node *identities*: every
host gets its own learned embedding row, looked up by ID. That approach
cannot score an edge touching a host it never saw during training --
exactly the failure mode flagged in this project's literature review
(CyberGFM is "non-inductive"). An enterprise network's host population
drifts (RQ1's whole premise), so a detector that cannot even produce a
score for a brand-new host is unusable for this paper's purposes.

This detector instead represents each host by a small vector of *running,
incrementally-updated statistics* (out/in-degree, partner diversity, age,
recency), computed the same way whether the host was seen once or ten
thousand times. A host nobody has ever seen gets the all-zero "cold
start" feature vector, which is itself a meaningful, learnable signal
("unfamiliar host") rather than an undefined embedding lookup. This is a
small, defensible model appropriate for a CPU-only sandbox and for the
paper's "lightweight baseline GFM-style detector" role; swapping in a
heavier pretrained model later only requires implementing the same
`Detector` interface.

Training signal
-----------------
For every window handed to `fit`, every observed edge is a positive
example (label 1, "this edge is part of normal traffic"); a configurable
number of random host-pairs drawn from that window's node set are
negative examples (label 0). This is standard link-prediction negative
sampling -- note it is an *approximation*: a small fraction of "negative"
pairs will, by chance, coincide with a real (positive) edge, mislabeling
it. With a modest negative/positive ratio and many windows, this does not
meaningfully bias results, but it is a real property of the method and
is documented here rather than left implicit.

Scoring convention
--------------------
The model outputs a logit where higher = "looks like normal traffic"
(matching the positive-label training convention). `score()` negates it,
so -- matching every other detector in this project -- **higher returned
score = more anomalous**.
"""
from __future__ import annotations

from typing import Sequence

import numpy as np
import torch
from torch import nn

from ..data.windows import WindowGraph
from .base import Detector

HOST_FEATURE_DIM = 6
EDGE_FEATURE_DIM = 2 * HOST_FEATURE_DIM + 1


class _LinkPredictorMLP(nn.Module):
    def __init__(self, hidden_dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(EDGE_FEATURE_DIM, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class _HostStats:
    """Running per-host statistics, updated one window at a time."""

    __slots__ = ("out_count", "in_count", "out_partners", "in_partners", "first_seen", "last_seen")

    def __init__(self):
        self.out_count: dict[str, int] = {}
        self.in_count: dict[str, int] = {}
        self.out_partners: dict[str, set] = {}
        self.in_partners: dict[str, set] = {}
        self.first_seen: dict[str, int] = {}
        self.last_seen: dict[str, int] = {}

    def feature(self, host: str, window_idx: int) -> np.ndarray:
        if host not in self.first_seen:
            return np.zeros(HOST_FEATURE_DIM, dtype=np.float32)
        age = window_idx - self.first_seen[host]
        recency = window_idx - self.last_seen[host]
        return np.array(
            [
                np.log1p(self.out_count.get(host, 0)),
                np.log1p(self.in_count.get(host, 0)),
                np.log1p(len(self.out_partners.get(host, ()))),
                np.log1p(len(self.in_partners.get(host, ()))),
                np.log1p(max(age, 0)),
                np.log1p(max(recency, 0)),
            ],
            dtype=np.float32,
        )

    def update(self, window: WindowGraph, window_idx: int) -> None:
        for (a, b), weight in zip(window.edges, window.edge_weight):
            src, dst = window.nodes[a], window.nodes[b]
            self.out_count[src] = self.out_count.get(src, 0) + weight
            self.in_count[dst] = self.in_count.get(dst, 0) + weight
            self.out_partners.setdefault(src, set()).add(dst)
            self.in_partners.setdefault(dst, set()).add(src)
            for h in (src, dst):
                self.first_seen.setdefault(h, window_idx)
                self.last_seen[h] = window_idx


class FeatureMLPDetector(Detector):
    """Small torch MLP link-predictor over running per-host features.

    Parameters
    ----------
    hidden_dim : int
        MLP hidden width. 16 is deliberately small -- this is meant to
        run in seconds per window on a 2-CPU sandbox, not to be a
        state-of-the-art architecture.
    n_negatives_per_positive : int
        Negative-sampling ratio used during `fit`.
    epochs_per_fit : int
        Number of full passes over the positive+negative examples built
        from the windows passed to one `fit` call.
    lr : float
        Adam learning rate.
    edge_count_cap : float
        Edge-level running counts are clipped to this value before
        `log1p` purely to keep the feature numerically bounded for
        extremely hot edges (e.g. a domain controller hit every
        window); does not affect ranking for any realistic count.
    seed : int
        Seeds both the torch model's initial weights and the numpy RNG
        used for negative sampling, for reproducible experiments.
    """

    def __init__(
        self,
        hidden_dim: int = 16,
        n_negatives_per_positive: int = 2,
        epochs_per_fit: int = 3,
        lr: float = 1e-2,
        edge_count_cap: float = 1e6,
        seed: int = 0,
    ):
        self.hidden_dim = hidden_dim
        self.n_negatives_per_positive = n_negatives_per_positive
        self.epochs_per_fit = epochs_per_fit
        self.lr = lr
        self.edge_count_cap = edge_count_cap
        self.seed = seed

        torch.manual_seed(seed)
        self._model = _LinkPredictorMLP(hidden_dim)
        self._opt = torch.optim.Adam(self._model.parameters(), lr=lr)
        self._loss_fn = nn.BCEWithLogitsLoss()
        self._rng = np.random.default_rng(seed)

        self._stats = _HostStats()
        self._edge_counts: dict[tuple[str, str], int] = {}
        self._window_idx = 0

    # -- internal helpers ---------------------------------------------

    def _edge_features(self, window: WindowGraph, a: int, b: int) -> np.ndarray:
        src, dst = window.nodes[a], window.nodes[b]
        f_src = self._stats.feature(src, self._window_idx)
        f_dst = self._stats.feature(dst, self._window_idx)
        c = min(self._edge_counts.get((src, dst), 0), self.edge_count_cap)
        return np.concatenate([f_src, f_dst, [np.log1p(c)]]).astype(np.float32)

    def _build_training_batch(self, window: WindowGraph) -> tuple[torch.Tensor, torch.Tensor]:
        if not window.nodes or not window.edges:
            return torch.empty(0, EDGE_FEATURE_DIM), torch.empty(0)

        pos_edge_set = set(window.edges)
        feats: list[np.ndarray] = []
        labels: list[float] = []
        n_nodes = len(window.nodes)

        for (a, b) in window.edges:
            feats.append(self._edge_features(window, a, b))
            labels.append(1.0)
            for _ in range(self.n_negatives_per_positive):
                na = int(self._rng.integers(0, n_nodes))
                nb = int(self._rng.integers(0, n_nodes))
                # Approximation, documented in the module docstring: we
                # do not retry on a collision with a real positive edge
                # beyond this one cheap reroll, to keep sampling O(1)
                # per negative even on dense windows.
                if (na, nb) in pos_edge_set:
                    nb = (nb + 1) % n_nodes
                feats.append(self._edge_features(window, na, nb))
                labels.append(0.0)

        x = torch.from_numpy(np.stack(feats))
        y = torch.tensor(labels, dtype=torch.float32)
        return x, y

    # -- Detector interface --------------------------------------------

    def fit(self, windows: Sequence[WindowGraph], *, warm_start: bool = True) -> None:
        if not warm_start:
            torch.manual_seed(self.seed)
            self._model = _LinkPredictorMLP(self.hidden_dim)
            self._opt = torch.optim.Adam(self._model.parameters(), lr=self.lr)
            self._stats = _HostStats()
            self._edge_counts = {}
            self._window_idx = 0

        for window in windows:
            if window.edges:
                x, y = self._build_training_batch(window)
                if x.shape[0] > 0:
                    self._model.train()
                    for _ in range(self.epochs_per_fit):
                        self._opt.zero_grad()
                        logits = self._model(x)
                        loss = self._loss_fn(logits, y)
                        loss.backward()
                        self._opt.step()

            for (a, b), weight in zip(window.edges, window.edge_weight):
                key = (window.nodes[a], window.nodes[b])
                self._edge_counts[key] = self._edge_counts.get(key, 0) + weight
            self._stats.update(window, self._window_idx)
            self._window_idx += 1

    def score(self, window: WindowGraph) -> np.ndarray:
        if not window.edges:
            return np.zeros(0, dtype=np.float64)
        feats = np.stack(
            [self._edge_features(window, a, b) for (a, b) in window.edges]
        )
        x = torch.from_numpy(feats)
        self._model.eval()
        with torch.no_grad():
            logits = self._model(x).numpy()
        return (-logits).astype(np.float64)

    def clone_fresh(self) -> "FeatureMLPDetector":
        return FeatureMLPDetector(
            hidden_dim=self.hidden_dim,
            n_negatives_per_positive=self.n_negatives_per_positive,
            epochs_per_fit=self.epochs_per_fit,
            lr=self.lr,
            edge_count_cap=self.edge_count_cap,
            seed=self.seed,
        )

    def state_summary(self) -> dict:
        return {
            "n_windows_fit": self._window_idx,
            "n_distinct_edges": len(self._edge_counts),
            "n_hosts_seen": len(self._stats.first_seen),
        }


In [ ]:
%%writefile lmsafe/detectors/rw_gnn.py
"""
Self-supervised graph neural network detector (torch, CPU-friendly) --
this project's "graph foundation model" piece.

Why a GNN, and why this particular design
-------------------------------------------
`FeatureMLPDetector` (feature_mlp.py) scores an edge from each endpoint's
own running statistics, computed *independently* -- it never looks at who
else a host is currently talking to. This detector adds exactly one
thing on top: one round of inductive neighborhood aggregation (a
GraphSAGE-style mean-pool update) over *this window's own graph*, so a
host's representation also reflects its current local neighborhood, not
just its own history. That is the actual, minimal, honest content behind
the word "graph" in "graph foundation model" here -- not a claim to
reimplement a published large-scale architecture.

Inductive by construction, like FeatureMLPDetector
-----------------------------------------------------
There is still no per-host-identity embedding table. The base feature
`h0` for each node is the same running (degree, partner-diversity, age,
recency) statistic `FeatureMLPDetector` uses (reused directly from
`feature_mlp._HostStats`, not reimplemented), and the neighborhood
aggregation only ever reads indices *local to the current window* (see
`WindowGraph.nodes`/`.edges`) -- never a global host vocabulary. This is
what makes "cross-network transfer" even a coherent thing to evaluate:
the encoder and scorer weights operate on a fixed 6-dimensional feature
space that means the same thing on any network's windows, so a model
pretrained on LANL's host population can be applied, unmodified, to
OpTC's completely disjoint one.

What "foundation model" means here, precisely
-------------------------------------------------
Pretraining (see scripts/pretrain_foundation_model.py) fits this
encoder+scorer once, via the same self-supervised negative-sampling
link-prediction objective used for ordinary `fit()`, but run over many
more windows than any single experiment's training split (e.g. the full
`lanl_full` 58-day stream). The resulting weights can then be handed to
a fresh `GraphFoundationDetector` via `pretrained_state_dict=...` for
zero-shot scoring or `freeze_encoder=True` fine-tuning on a different
network's windows, without ever retraining the encoder from scratch on
the new network. This is a deliberately small-scale, CPU-trainable
instance of the foundation-model idea -- a few hundred thousand
parameters, minutes to pretrain on a 2-CPU sandbox -- not a claim to
have built anything at the scale the term usually evokes in NLP/vision.
Scoped honestly, it is still the right experiment for the paper's
cross-network-transfer question: does *any* amount of pretraining on one
network transfer to another, versus training from scratch on the target
alone?

Training signal and scoring convention
-----------------------------------------
Identical to `FeatureMLPDetector`: every observed edge in a window is a
positive example; random host-pairs are negative examples (standard
link-prediction negative sampling, same documented approximation). The
model outputs a logit where higher = "looks like normal traffic";
`score()` negates it so higher returned score = more anomalous, matching
every other detector in this project.
"""
from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import torch
from torch import nn

from ..data.windows import WindowGraph
from .base import Detector
from .feature_mlp import HOST_FEATURE_DIM, _HostStats

GNN_HIDDEN_DIM_DEFAULT = 16


class _GraphEncoder(nn.Module):
    """One-hop inductive GraphSAGE-style update: a node's new
    representation combines its own base feature with the mean base
    feature of its in-neighbors and out-neighbors *within this window's
    graph only* -- no persistent state across windows or networks is
    stored here, which is exactly what keeps this inductive across an
    entirely different host population (e.g. OpTC's)."""

    def __init__(self, in_dim: int, hidden_dim: int):
        super().__init__()
        self.self_lin = nn.Linear(in_dim, hidden_dim)
        self.out_nbr_lin = nn.Linear(in_dim, hidden_dim)
        self.in_nbr_lin = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()

    def forward(self, h0: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """`h0`: (n_nodes, in_dim) base features. `edge_index`: (2,
        n_edges) local (src, dst) indices, directed, one row per
        *distinct* edge (matching `WindowGraph.edges`)."""
        n_nodes = h0.shape[0]
        if edge_index.shape[1] == 0:
            out_mean = torch.zeros_like(h0)
            in_mean = torch.zeros_like(h0)
        else:
            src, dst = edge_index[0], edge_index[1]
            ones = torch.ones(src.shape[0], dtype=h0.dtype)

            # out_mean[i] = mean over j of h0[j] for every edge (i, j) --
            # the average base feature of i's out-neighbors.
            out_sum = torch.zeros(n_nodes, h0.shape[1], dtype=h0.dtype).index_add_(0, src, h0[dst])
            out_deg = torch.zeros(n_nodes, dtype=h0.dtype).index_add_(0, src, ones).clamp(min=1).unsqueeze(1)
            out_mean = out_sum / out_deg

            # in_mean[i] = mean over j of h0[j] for every edge (j, i) --
            # the average base feature of i's in-neighbors.
            in_sum = torch.zeros(n_nodes, h0.shape[1], dtype=h0.dtype).index_add_(0, dst, h0[src])
            in_deg = torch.zeros(n_nodes, dtype=h0.dtype).index_add_(0, dst, ones).clamp(min=1).unsqueeze(1)
            in_mean = in_sum / in_deg

        return self.act(self.self_lin(h0) + self.out_nbr_lin(out_mean) + self.in_nbr_lin(in_mean))


class _EdgeScorerMLP(nn.Module):
    def __init__(self, hidden_dim: int, mlp_hidden: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * hidden_dim + 1, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1),
        )

    def forward(self, h1: torch.Tensor, a_idx: torch.Tensor, b_idx: torch.Tensor, edge_count_feat: torch.Tensor) -> torch.Tensor:
        x = torch.cat([h1[a_idx], h1[b_idx], edge_count_feat.unsqueeze(1)], dim=1)
        return self.net(x).squeeze(-1)


class GraphFoundationDetector(Detector):
    """Self-supervised, inductive one-hop GNN link-predictor -- see the
    module docstring for the full design rationale and the precise,
    deliberately modest scope of "foundation model" claimed here.

    Parameters
    ----------
    hidden_dim : int
        Encoder output / scorer input width.
    n_negatives_per_positive, epochs_per_fit, lr, edge_count_cap, seed :
        Same meaning as `FeatureMLPDetector`'s identically-named
        parameters.
    pretrained_state_dict : dict, optional
        `{"encoder": <state_dict>, "scorer": <state_dict>}` produced by
        `export_state_dict()` (typically loaded from a checkpoint
        written by `scripts/pretrain_foundation_model.py`). Initializes
        the encoder and scorer weights before any `fit()` call -- this
        is the mechanism "cross-network transfer" uses: pretrain on one
        network, construct a fresh detector here with that checkpoint,
        then `fit()`/`score()` on a different network's windows.
    freeze_encoder : bool
        If True, the encoder's parameters are excluded from the
        optimizer entirely -- `fit()` only ever updates the edge-scorer
        MLP on top of a frozen pretrained encoder. Models the "zero-shot
        structural transfer, lightweight fine-tune on top" regime that
        is the interesting middle ground between "no pretraining at
        all" and "fully fine-tune everything".
    """

    def __init__(
        self,
        hidden_dim: int = GNN_HIDDEN_DIM_DEFAULT,
        n_negatives_per_positive: int = 2,
        epochs_per_fit: int = 3,
        lr: float = 1e-2,
        edge_count_cap: float = 1e6,
        seed: int = 0,
        pretrained_state_dict: Optional[dict] = None,
        freeze_encoder: bool = False,
    ):
        self.hidden_dim = hidden_dim
        self.n_negatives_per_positive = n_negatives_per_positive
        self.epochs_per_fit = epochs_per_fit
        self.lr = lr
        self.edge_count_cap = edge_count_cap
        self.seed = seed
        self.freeze_encoder = freeze_encoder
        self._pretrained_state_dict = pretrained_state_dict

        torch.manual_seed(seed)
        self._encoder = _GraphEncoder(HOST_FEATURE_DIM, hidden_dim)
        self._scorer = _EdgeScorerMLP(hidden_dim)
        if pretrained_state_dict is not None:
            self._encoder.load_state_dict(pretrained_state_dict["encoder"])
            self._scorer.load_state_dict(pretrained_state_dict["scorer"])
        if freeze_encoder:
            for p in self._encoder.parameters():
                p.requires_grad_(False)
        self._build_optimizer()
        self._loss_fn = nn.BCEWithLogitsLoss()
        self._rng = np.random.default_rng(seed)

        self._stats = _HostStats()
        self._edge_counts: dict[tuple[str, str], int] = {}
        self._window_idx = 0

    def _build_optimizer(self) -> None:
        params = [p for p in self._encoder.parameters() if p.requires_grad]
        params += list(self._scorer.parameters())
        self._opt = torch.optim.Adam(params, lr=self.lr)

    # -- internal helpers ---------------------------------------------

    def _window_tensors(self, window: WindowGraph) -> tuple[torch.Tensor, torch.Tensor]:
        n = len(window.nodes)
        h0 = torch.zeros(n, HOST_FEATURE_DIM, dtype=torch.float32)
        for i, host in enumerate(window.nodes):
            h0[i] = torch.from_numpy(self._stats.feature(host, self._window_idx))
        if window.edges:
            edge_index = torch.tensor(window.edges, dtype=torch.long).T
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
        return h0, edge_index

    def _edge_count_feat(self, window: WindowGraph, pairs: Sequence[tuple[int, int]]) -> torch.Tensor:
        vals = []
        for a, b in pairs:
            key = (window.nodes[a], window.nodes[b])
            c = min(self._edge_counts.get(key, 0), self.edge_count_cap)
            vals.append(np.log1p(c))
        return torch.tensor(vals, dtype=torch.float32)

    def _sample_negatives(self, window: WindowGraph) -> list[tuple[int, int]]:
        n_nodes = len(window.nodes)
        pos_edge_set = set(window.edges)
        negs = []
        for _ in range(len(window.edges) * self.n_negatives_per_positive):
            na = int(self._rng.integers(0, n_nodes))
            nb = int(self._rng.integers(0, n_nodes))
            if (na, nb) in pos_edge_set:
                nb = (nb + 1) % n_nodes
            negs.append((na, nb))
        return negs

    # -- Detector interface --------------------------------------------

    def fit(self, windows: Sequence[WindowGraph], *, warm_start: bool = True) -> None:
        if not warm_start:
            torch.manual_seed(self.seed)
            self._encoder = _GraphEncoder(HOST_FEATURE_DIM, self.hidden_dim)
            self._scorer = _EdgeScorerMLP(self.hidden_dim)
            if self._pretrained_state_dict is not None:
                self._encoder.load_state_dict(self._pretrained_state_dict["encoder"])
                self._scorer.load_state_dict(self._pretrained_state_dict["scorer"])
            if self.freeze_encoder:
                for p in self._encoder.parameters():
                    p.requires_grad_(False)
            self._build_optimizer()
            self._stats = _HostStats()
            self._edge_counts = {}
            self._window_idx = 0

        for window in windows:
            if window.edges:
                h0, edge_index = self._window_tensors(window)
                negs = self._sample_negatives(window)
                pos_a = torch.tensor([a for a, _ in window.edges], dtype=torch.long)
                pos_b = torch.tensor([b for _, b in window.edges], dtype=torch.long)
                neg_a = torch.tensor([a for a, _ in negs], dtype=torch.long) if negs else torch.empty(0, dtype=torch.long)
                neg_b = torch.tensor([b for _, b in negs], dtype=torch.long) if negs else torch.empty(0, dtype=torch.long)
                pos_c = self._edge_count_feat(window, window.edges)
                neg_c = self._edge_count_feat(window, negs) if negs else torch.empty(0)
                y = torch.cat([torch.ones(len(window.edges)), torch.zeros(len(negs))])

                self._encoder.train()
                self._scorer.train()
                for _ in range(self.epochs_per_fit):
                    self._opt.zero_grad()
                    h1 = self._encoder(h0, edge_index)
                    logits_pos = self._scorer(h1, pos_a, pos_b, pos_c)
                    if negs:
                        logits_neg = self._scorer(h1, neg_a, neg_b, neg_c)
                        logits = torch.cat([logits_pos, logits_neg])
                    else:
                        logits = logits_pos
                    loss = self._loss_fn(logits, y)
                    loss.backward()
                    self._opt.step()

            self.observe(window)

    def score(self, window: WindowGraph) -> np.ndarray:
        if not window.edges:
            return np.zeros(0, dtype=np.float64)
        h0, edge_index = self._window_tensors(window)
        a_idx = torch.tensor([a for a, _ in window.edges], dtype=torch.long)
        b_idx = torch.tensor([b for _, b in window.edges], dtype=torch.long)
        c = self._edge_count_feat(window, window.edges)
        self._encoder.eval()
        self._scorer.eval()
        with torch.no_grad():
            h1 = self._encoder(h0, edge_index)
            logits = self._scorer(h1, a_idx, b_idx, c).numpy()
        return (-logits).astype(np.float64)

    def observe(self, window: WindowGraph) -> None:
        """Update the running host-stats / edge-count state from
        `window` WITHOUT any gradient step -- the neural weights
        (encoder + scorer) are left exactly as they are.

        This is the mechanism a genuine zero-shot cross-network-transfer
        evaluation needs: a detector carrying pretrained-on-network-A
        weights should be allowed to build up network B's own running
        host statistics as it moves through B's stream (so window 500
        benefits from having "seen" hosts in windows 1-499), without
        that counting as having *trained* on B at all. `fit()` already
        does this same state update internally (see its tail, after the
        optional gradient step) -- this method factors exactly that
        part out so a caller can get it without the training that
        normally comes attached.
        """
        for (a, b), weight in zip(window.edges, window.edge_weight):
            key = (window.nodes[a], window.nodes[b])
            self._edge_counts[key] = self._edge_counts.get(key, 0) + weight
        self._stats.update(window, self._window_idx)
        self._window_idx += 1

    def clone_fresh(self) -> "GraphFoundationDetector":
        return GraphFoundationDetector(
            hidden_dim=self.hidden_dim,
            n_negatives_per_positive=self.n_negatives_per_positive,
            epochs_per_fit=self.epochs_per_fit,
            lr=self.lr,
            edge_count_cap=self.edge_count_cap,
            seed=self.seed,
            pretrained_state_dict=self._pretrained_state_dict,
            freeze_encoder=self.freeze_encoder,
        )

    def state_summary(self) -> dict:
        return {
            "n_windows_fit": self._window_idx,
            "n_distinct_edges": len(self._edge_counts),
            "n_hosts_seen": len(self._stats.first_seen),
            "pretrained": self._pretrained_state_dict is not None,
            "encoder_frozen": self.freeze_encoder,
        }

    # -- pretraining / transfer support ---------------------------------

    def export_state_dict(self) -> dict:
        """`{"encoder": ..., "scorer": ...}`, suitable for
        `pretrained_state_dict=` on a fresh instance, or for
        `torch.save` to a checkpoint file (see
        `scripts/pretrain_foundation_model.py`)."""
        return {
            "encoder": self._encoder.state_dict(),
            "scorer": self._scorer.state_dict(),
        }


In [ ]:
%%writefile lmsafe/drift.py
"""
Concept-drift detectors used only to *trigger* an update policy.

Critical design rule, stated once here and enforced by every caller in
`lmsafe.policies`: a drift detector in this module NEVER sees the
anomaly score of an individual edge, and is NEVER used as (part of) the
anomaly score itself. It only ever watches an aggregate, per-window
summary statistic (e.g. "mean anomaly score this window" or "fraction of
edges this window above the current alert threshold") and answers one
yes/no question: *has this summary statistic's distribution shifted
enough that the detector's model of 'normal' should probably be
refreshed?*

Keeping this boundary matters for RQ2: the attacker's conditioning edges
are specifically designed to look individually unremarkable while
shifting the model's notion of normal over many windows. A drift
detector that happened to be confused with the anomaly scorer itself
would make the attack trivially detectable by construction and defeat
the whole point of the experiment -- the "DriftTriggered" policy in
`lmsafe.policies` is a *baseline to be attacked*, not a safeguard, and
RQ2 is precisely the experiment that measures how easily it is fooled.

Two detectors are implemented:

PageHinkley     A standard, exact, small-state sequential test for a
                 one-sided shift in the mean of a scalar stream (Page,
                 1954; the incremental form used widely in concept-drift
                 literature, e.g. in the `river` package).
SimplifiedADWIN  A documented simplification of Bifet & Gavaldà's (2007)
                 ADWIN. See its class docstring for exactly what is
                 simplified and why, and what to swap in if exact
                 fidelity to the original paper is needed for a
                 camera-ready claim.
"""
from __future__ import annotations

import math
from collections import deque


class PageHinkley:
    """Incremental Page-Hinkley test for a one-sided increase in the
    mean of a scalar stream.

    Maintains a running mean `m_t` and the cumulative deviation
    `sum_t = sum_{i<=t}(x_i - m_i - delta)`, together with its
    running minimum `min_sum_t`. Signals a change when
    `sum_t - min_sum_t` exceeds `threshold`, then resets all state (the
    standard "detect-and-reset" variant, appropriate here because a
    caller that gets a drift signal is about to retrain on fresh data
    anyway, so there is no benefit to remembering pre-reset statistics).

    Parameters
    ----------
    delta : float
        Minimum magnitude of change considered meaningful (slack term);
        guards against alarming on infinitesimal drift.
    threshold : float
        Alarm threshold (often called "lambda" in the literature). Larger
        -> fewer false alarms, slower detection.
    """

    def __init__(self, delta: float = 0.005, threshold: float = 10.0):
        self.delta = delta
        self.threshold = threshold
        self.reset()

    def reset(self) -> None:
        self._n = 0
        self._mean = 0.0
        self._sum = 0.0
        self._min_sum = 0.0

    def update(self, x: float) -> bool:
        """Feed one new observation. Returns True exactly on the step a
        change is signaled (state is reset on that same call)."""
        self._n += 1
        self._mean += (x - self._mean) / self._n
        self._sum += x - self._mean - self.delta
        self._min_sum = min(self._min_sum, self._sum)
        ph = self._sum - self._min_sum
        if ph > self.threshold:
            self.reset()
            return True
        return False


class SimplifiedADWIN:
    """A documented simplification of ADWIN (Bifet & Gavaldà, 2007).

    What is kept from the original algorithm
    -------------------------------------------
    The core statistical idea: maintain a window of recent scalar
    observations, and at every step test whether splitting the window
    at some point produces two sub-windows whose means differ by more
    than a Hoeffding-bound-derived threshold (scaled by the observed
    range and corrected for multiple testing across candidate split
    points). If so, the data *before* that split is stale relative to
    the most recent behavior and is discarded.

    What is simplified
    ---------------------
    The original ADWIN stores the window as a list of exponentially-
    sized *buckets* (size 1, 2, 4, 8, ... up to a per-level cap `M`),
    which is what lets it provide effectively unbounded lookback in
    `O(log N)` memory via periodic bucket merging. This implementation
    instead keeps the raw observations in a plain bounded `deque` of at
    most `max_window` scalars, and tests candidate splits only at
    power-of-two offsets from the most recent observation (so still
    `O(log window)` candidate tests per update, just without the
    bucket-merge memory optimization). For this project's use case --
    one scalar summary per hourly window, with `max_window` set to a
    few hundred -- the memory this simplification would have saved is a
    few kilobytes, so the simplification costs nothing in practice
    while being far simpler to verify correct than the original
    bucket-list data structure.

    If a camera-ready version of the paper needs results that match the
    original ADWIN bit-for-bit, swap in `river.drift.ADWIN`, which
    implements the full bucket-list algorithm; it exposes the same
    `update(x) -> bool`-shaped interface used by `lmsafe.policies`, so
    no other code needs to change.

    Parameters
    ----------
    delta : float
        Confidence parameter (smaller -> more conservative, fewer false
        alarms). This is the same "delta" as in the original paper, not
        related to `PageHinkley.delta`.
    max_window : int
        Hard cap on retained history.
    min_window : int
        Do not attempt any cut test until at least this many
        observations have been seen (avoids spurious alarms on very
        short streams where the Hoeffding bound is meaningless).
    """

    def __init__(self, delta: float = 0.002, max_window: int = 400, min_window: int = 16):
        self.delta = delta
        self.max_window = max_window
        self.min_window = min_window
        self._window: deque[float] = deque(maxlen=max_window)

    def reset(self) -> None:
        self._window.clear()

    def update(self, x: float) -> bool:
        """Feed one new observation. Returns True exactly on the step a
        change is signaled; on that same call, stale history (everything
        before the detected cut point) is discarded from the window."""
        self._window.append(x)
        n = len(self._window)
        if n < self.min_window:
            return False

        values = list(self._window)
        lo = min(values)
        hi = max(values)
        value_range = max(hi - lo, 1e-9)

        # Candidate "recent" sizes: powers of two, smallest first, up to
        # just under the full window -- mirrors the exponential bucket
        # granularity of the original algorithm without its merge
        # machinery.
        candidates = []
        k = 1
        while k < n:
            candidates.append(k)
            k *= 2

        best_cut = None
        for k in candidates:
            n_recent = k
            n_old = n - k
            if n_old < 1:
                continue
            recent = values[n - k:]
            old = values[: n - k]
            mean_recent = sum(recent) / n_recent
            mean_old = sum(old) / n_old

            # Harmonic-mean effective sample size and Bonferroni
            # correction across the candidates tested this step, as in
            # the original ADWIN's cut test.
            m = 1.0 / (1.0 / n_old + 1.0 / n_recent)
            delta_prime = self.delta / max(len(candidates), 1)
            epsilon = value_range * math.sqrt(
                (1.0 / (2.0 * m)) * math.log(4.0 / delta_prime)
            )

            if abs(mean_recent - mean_old) > epsilon:
                # Prefer the candidate that discards the most stale
                # data (largest n_old among valid cuts); candidates are
                # scanned smallest-k-first so keep searching rather than
                # stopping at the first hit.
                if best_cut is None or n_old > best_cut:
                    best_cut = n_old

        if best_cut is not None:
            # Discard the stale prefix: keep only the most recent
            # (n - best_cut) observations.
            kept = values[best_cut:]
            self._window = deque(kept, maxlen=self.max_window)
            return True
        return False


In [ ]:
%%writefile lmsafe/policies.py
"""
Update policies: the paper's central comparison axis.

Every policy answers the same question differently: *given the stream of
test windows arriving one at a time, when and on what data should the
detector's model of "normal" be refreshed?* `lmsafe.runner` drives all of
them through the exact same loop (score the window, then let the policy
decide what happens to the detector), so differences in RQ1/RQ2/RQ3
results come only from the policy logic below, never from a difference
in how they are driven.

    FrozenPolicy          Never updates after the initial fit. The RQ1
                           reference point for "drift cost": whatever
                           detection quality is lost over time under
                           this policy *is* the cost of standing still.
    PeriodicPolicy         Every `period` windows, retrain FROM SCRATCH
                           on exactly the most recent `lookback` windows
                           -- a clean slate each time, no memory beyond
                           the lookback. Accepts every window's data
                           unconditionally, including any attacker-
                           injected conditioning edges (RQ2's naive
                           baseline).
    SlidingReplayPolicy     Keeps a continuously-shifting trailing buffer
                            of the last `window_size` windows and
                            retrains from scratch on it every
                            `retrain_period` windows. Also accepts
                            everything unconditionally -- the second
                            RQ2 naive baseline, methodologically distinct
                            from Periodic (overlapping sliding window vs
                            disjoint batches).
    DriftTriggeredPolicy    Like Frozen, except a `lmsafe.drift` detector
                            watches a per-window summary statistic (mean
                            anomaly score by default) and triggers a
                            from-scratch retrain on recent history when
                            it fires. A third RQ2 naive baseline: it
                            reacts to *aggregate* shift, which is exactly
                            what a slow, individually-unremarkable
                            conditioning attack is designed to evade.
    GatedPolicy             This project's proposed defense. See its own
                            docstring below for the three mechanisms
                            (score-gated admission, delayed multi-host
                            corroboration, clean anchor memory) that
                            together answer RQ3.

All five share the `UpdatePolicy` interface so `lmsafe.runner` can treat
them interchangeably.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from collections import defaultdict, deque
from typing import Callable, Optional, Sequence

import numpy as np

from .data.windows import WindowGraph, filter_edges
from .detectors.base import Detector
from .drift import PageHinkley, SimplifiedADWIN


class UpdatePolicy(ABC):
    """Common interface for all update policies.

    `lmsafe.runner`'s loop, once per test window, is exactly::

        scores = detector.score(window)
        record_metrics(window, scores)           # ground truth read ONLY here
        detector = policy.observe(window, scores, detector)

    and before that loop starts, exactly once::

        policy.fit_initial(detector, train_windows)

    No policy implementation is allowed to read `window.edge_is_malicious`
    -- see `lmsafe.data.schema`'s module docstring on the ground-truth
    boundary. `GatedPolicy`'s "corroboration" logic uses only the
    *detector's own scores* and *structural* multi-host agreement,
    never the ground-truth label; this is checked in
    tests/test_no_label_leak.py.
    """

    name: str = "update_policy"

    def fit_initial(self, detector: Detector, train_windows: Sequence[WindowGraph]) -> None:
        """Default initial fit: a full (from-scratch) fit on the benign
        training windows. Overridden only if a policy needs to also
        seed its own internal buffers (every concrete policy below
        does, and calls this base implementation first via `super()`).
        """
        detector.fit(train_windows, warm_start=False)

    @abstractmethod
    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        """Called once per test window, after `detector.score(window)`
        has already been computed and recorded by the caller. Returns
        the detector object to use for scoring the *next* window (which
        may be the same object, mutated in place -- all detectors here
        mutate in place and return `self` from `fit`-triggering code
        paths, but a policy is free to swap in a different detector
        instance, e.g. `detector.clone_fresh()`, if that is cleaner for
        its logic).
        """


# ---------------------------------------------------------------------------
# RQ1 reference point
# ---------------------------------------------------------------------------


class FrozenPolicy(UpdatePolicy):
    """Never updates. Whatever quality the initial 41-hour benign fit
    has is exactly the quality it keeps for the rest of the stream."""

    name = "frozen"

    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        return detector


# ---------------------------------------------------------------------------
# RQ2 naive baselines (accept every window unconditionally)
# ---------------------------------------------------------------------------


class PeriodicPolicy(UpdatePolicy):
    """Every `period` windows, discard the model and retrain from
    scratch on exactly the most recent `lookback` windows.

    Parameters
    ----------
    period : int
        Number of test windows between retrains.
    lookback : int, optional
        Number of most-recent windows (including the just-elapsed
        period) used for each retrain. Defaults to `period` itself,
        i.e. **disjoint, non-overlapping batches** -- each retrain sees
        only windows accumulated since the previous retrain and nothing
        older, which is the policy's defining property and the reason
        it is methodologically distinct from `SlidingReplayPolicy`
        below. Pass `lookback > period` deliberately for a variant that
        also keeps some older history (e.g. `lookback = 7 * period` for
        "retrain weekly on the last month"); at that point it starts to
        resemble `SlidingReplayPolicy` and the two are no longer a
        clean contrast, so RQ experiment configs should keep
        `lookback == period` (the default) unless that overlap is
        specifically what is being studied.
    """

    name = "periodic"

    def __init__(self, period: int = 24, lookback: Optional[int] = None):
        self.period = period
        self.lookback = lookback if lookback is not None else period
        self._buffer: deque[WindowGraph] = deque(maxlen=self.lookback)
        self._since_retrain = 0

    def fit_initial(self, detector: Detector, train_windows: Sequence[WindowGraph]) -> None:
        super().fit_initial(detector, train_windows)
        self._buffer.extend(train_windows)

    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        self._buffer.append(window)
        self._since_retrain += 1
        if self._since_retrain >= self.period:
            self._since_retrain = 0
            detector.fit(list(self._buffer), warm_start=False)
        return detector


class SlidingReplayPolicy(UpdatePolicy):
    """Keeps a continuously-shifting trailing window of the last
    `window_size` test windows (unconditionally, including any attacker
    traffic) and retrains from scratch on it every `retrain_period`
    windows.

    Methodologically distinct from `PeriodicPolicy`: the buffer here
    slides by one window at a time (every window ages out exactly
    `window_size` windows after it arrived), whereas Periodic's buffer
    is refilled in disjoint, non-overlapping batches of size `period`.
    """

    name = "sliding_replay"

    def __init__(self, window_size: int = 168, retrain_period: int = 24):
        self.window_size = window_size
        self.retrain_period = retrain_period
        self._buffer: deque[WindowGraph] = deque(maxlen=window_size)
        self._since_retrain = 0

    def fit_initial(self, detector: Detector, train_windows: Sequence[WindowGraph]) -> None:
        super().fit_initial(detector, train_windows)
        self._buffer.extend(train_windows)

    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        self._buffer.append(window)
        self._since_retrain += 1
        if self._since_retrain >= self.retrain_period:
            self._since_retrain = 0
            detector.fit(list(self._buffer), warm_start=False)
        return detector


class DriftTriggeredPolicy(UpdatePolicy):
    """Frozen, except an `lmsafe.drift` detector watches a per-window
    summary statistic of the detector's own scores and triggers a
    from-scratch retrain on recent history when it fires.

    This is the naive baseline RQ2's conditioning attack is specifically
    designed to defeat: a slow drip of individually-unremarkable edges
    is, by construction, meant to avoid any sharp *aggregate* shift in
    mean anomaly score, so a detector that only reacts to that aggregate
    shift should still end up absorbing the conditioning edges whenever
    it does eventually retrain (whether triggered by the real benign
    drift or by chance).

    Parameters
    ----------
    drift_detector : PageHinkley or SimplifiedADWIN
        Pass a pre-configured instance; `lmsafe.runner`'s experiment
        configs choose which one via `lmsafe.config`.
    lookback : int
        Window history used for the from-scratch retrain once drift
        fires.
    stat_fn : Callable[[np.ndarray], float]
        Reduces one window's per-edge score array to the scalar fed to
        the drift detector. Default: mean score.
    """

    name = "drift_triggered"

    def __init__(
        self,
        drift_detector: PageHinkley | SimplifiedADWIN,
        lookback: int = 168,
        stat_fn: Callable[[np.ndarray], float] = lambda s: float(np.mean(s)) if len(s) else 0.0,
    ):
        self.drift_detector = drift_detector
        self.lookback = lookback
        self.stat_fn = stat_fn
        self._history: deque[WindowGraph] = deque(maxlen=lookback)

    def fit_initial(self, detector: Detector, train_windows: Sequence[WindowGraph]) -> None:
        super().fit_initial(detector, train_windows)
        self._history.extend(train_windows)

    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        self._history.append(window)
        stat = self.stat_fn(scores)
        if self.drift_detector.update(stat):
            detector.fit(list(self._history), warm_start=False)
        return detector


# ---------------------------------------------------------------------------
# RQ3 proposed defense
# ---------------------------------------------------------------------------


class GatedPolicy(UpdatePolicy):
    """Score-gated, delayed-commit, multi-host-corroborated updating
    with a permanent clean-anchor memory.

    Three mechanisms, each targeting a specific weakness of the naive
    baselines above:

    1. **Score-gated admission.** An edge is only even a *candidate* for
       future training if its own anomaly score (from the detector's
       *current* model, computed before any update this window) is at
       or below the `score_admission_quantile` quantile of this
       window's scores -- i.e. the detector's current best guess is
       already "this looks fine". An edge that already looks highly
       suspicious is never used to teach the model that it is normal,
       no matter what else happens to it below.

    2. **Delayed commit with multi-host corroboration.** An admitted
       edge sits in a quarantine buffer for `quarantine_windows` windows
       before it is eligible to be promoted into the training buffer,
       and is only actually promoted if, within that same horizon, its
       *destination* host was also reached by at least
       `min_corroborating_src_hosts` *other, distinct* source hosts.
       This is directly motivated by this project's own inspection of
       the real LANL red-team file: 701 of 749 real lateral-movement
       events originate from a single source host. A conditioning
       attacker operating from one or a few hosts cannot manufacture
       corroboration from unrelated hosts without a much larger,
       easier-to-notice compromise -- whereas genuine benign drift
       (e.g. a newly provisioned subnet or file server) is, by the RQ1
       hypothesis, *collective*: many legitimate hosts reach it
       independently within a short horizon. Corroboration uses only
       structural agreement across hosts, never the ground-truth label.

    3. **Clean anchor memory.** The original benign training windows
       (`fit_initial`'s `train_windows`) are kept forever and mixed
       into *every* retrain alongside the promoted buffer. Even if some
       poisoned data slips past (1) and (2), the model is retrained
       from scratch each time on anchor-plus-promoted data, not warm-
       started from a possibly-already-biased running model -- bounding
       how far any slipped-through poisoning can pull the decision
       boundary in one retrain cycle.

    Parameters
    ----------
    score_admission_quantile : float in (0, 1]
        Edges scoring above this quantile of the window's own score
        distribution are never admitted to quarantine at all.
    quarantine_windows : int
        How long an admitted edge waits for corroboration before a
        promotion decision is made.
    min_corroborating_src_hosts : int
        Minimum distinct *other* source hosts that must also reach the
        same destination host within the quarantine horizon.
    retrain_period : int
        How often (in test windows) to actually retrain, from scratch,
        on anchor + promoted buffer.
    replay_cap : int
        Maximum number of promoted windows kept in the buffer (oldest
        dropped first) -- bounds memory on a long test stream.
    """

    name = "gated"

    def __init__(
        self,
        score_admission_quantile: float = 0.9,
        quarantine_windows: int = 24,
        min_corroborating_src_hosts: int = 3,
        retrain_period: int = 6,
        replay_cap: int = 500,
    ):
        if not 0 < score_admission_quantile <= 1:
            raise ValueError("score_admission_quantile must be in (0, 1]")
        self.score_admission_quantile = score_admission_quantile
        self.quarantine_windows = quarantine_windows
        self.min_corroborating_src_hosts = min_corroborating_src_hosts
        self.retrain_period = retrain_period
        self.replay_cap = replay_cap

        self._anchor_windows: list[WindowGraph] = []
        # Each entry: (admitted_at_window_idx, WindowGraph, admitted_edge_indices)
        self._quarantine: deque[tuple[int, WindowGraph, list[int]]] = deque()
        self._committed_buffer: deque[WindowGraph] = deque(maxlen=replay_cap)
        self._window_idx = 0
        self._since_retrain = 0

        # Diagnostics, useful for the paper's ablations/figures: how
        # much of what was *admitted* actually got *promoted*, per run.
        self.n_admitted_total = 0
        self.n_promoted_total = 0

    def fit_initial(self, detector: Detector, train_windows: Sequence[WindowGraph]) -> None:
        super().fit_initial(detector, train_windows)
        self._anchor_windows = list(train_windows)

    def observe(self, window: WindowGraph, scores: np.ndarray, detector: Detector) -> Detector:
        self._window_idx += 1

        admitted_idx = self._admit(window, scores)
        self.n_admitted_total += len(admitted_idx)
        self._quarantine.append((self._window_idx, window, admitted_idx))

        self._promote_matured_entries()

        self._since_retrain += 1
        if self._since_retrain >= self.retrain_period:
            self._since_retrain = 0
            train_batch = self._anchor_windows + list(self._committed_buffer)
            if train_batch:
                detector.fit(train_batch, warm_start=False)
        return detector

    # -- mechanism 1: score-gated admission ----------------------------

    def _admit(self, window: WindowGraph, scores: np.ndarray) -> list[int]:
        if len(scores) == 0:
            return []
        threshold = float(np.quantile(scores, self.score_admission_quantile))
        return [i for i, s in enumerate(scores) if s <= threshold]

    # -- mechanism 2: delayed commit + multi-host corroboration --------

    def _promote_matured_entries(self) -> None:
        while self._quarantine and self._window_idx - self._quarantine[0][0] >= self.quarantine_windows:
            cand_idx, cand_window, cand_edge_idx = self._quarantine.popleft()

            # Corroboration evidence: for every destination host, the
            # set of distinct source hosts that reached it in every
            # *other* entry still in the quarantine horizon (these span
            # [cand_idx+1, window_idx], i.e. "within the horizon after
            # this candidate was admitted" -- corroboration is allowed
            # to arrive any time up to the promotion decision).
            dst_to_srcs: dict[str, set] = defaultdict(set)
            for (_, w, eidx) in self._quarantine:
                for i in eidx:
                    a, b = w.edges[i]
                    dst_to_srcs[w.nodes[b]].add(w.nodes[a])

            kept = []
            for i in cand_edge_idx:
                a, b = cand_window.edges[i]
                src_host, dst_host = cand_window.nodes[a], cand_window.nodes[b]
                corroborators = dst_to_srcs.get(dst_host, set()) - {src_host}
                if len(corroborators) >= self.min_corroborating_src_hosts:
                    kept.append(i)

            if kept:
                self.n_promoted_total += len(kept)
                self._committed_buffer.append(filter_edges(cand_window, kept))


In [ ]:
%%writefile lmsafe/attack.py
"""
Update-poisoning / "boiling-frog" conditioning injector -- RQ2 ONLY.

Scope and intent
-------------------
This module exists to answer one measurement question inside this
project's own **offline evaluation harness**: *if an adversary can
influence what a continually-updating detector learns is "normal"
before it attacks, how much does that degrade the detector's ability to
flag the real attack when it finally happens, under each update policy
in `lmsafe.policies`?*

It operates exclusively on event streams already materialized in memory
(the synthetic generator's output, or a pre-extracted/labeled LANL
window list) inside `lmsafe.runner`'s experiment loop. It has no network
access, no ability to reach any live system, and no code path that
writes anywhere outside the Python objects passed into it. It is the
offline-research equivalent of a published adversarial-robustness
ablation (the kind already present in, e.g., the `poison_datasets.py`
ablation in the CyberGFM baseline this project reviewed, and in the
ANTIDOTE / SIGL-style poisoning literature) -- a controlled, synthetic
measurement of a defense's robustness, not an attack tool.

The conditioning scenario
----------------------------
Before a lateral-movement episode (`lmsafe.synth.LMEpisode`) happens, the
same actor host sends a slow, low-volume trickle of authentication
events toward the same eventual target over `conditioning_days` days.
Each individual conditioning event is built to look exactly like an
ordinary benign authentication event (same schema, same `auth_type`/
`logon_type` distribution as the synthetic generator's benign traffic,
low volume) -- the entire point of the scenario is that no single
conditioning event is unusual; only the slow, persistent pattern over
many windows is informative, and only in hindsight.

Ground-truth labeling choice (read before using `recall_at_budget` on a
conditioned run): conditioning events are **not** labeled
`is_malicious=True` by default (`ConditioningSpec.label_as_malicious`).
This mirrors how the real LANL red-team file labels only the logged
lateral-movement authentications themselves, not any reconnaissance or
staging activity that might have preceded them. The primary RQ2 metric
this project reports is therefore "recall on the real attack episode,
with vs. without a preceding conditioning campaign" -- *not* "recall on
the conditioning events" (though setting `label_as_malicious=True` is
supported for a secondary analysis of whether the conditioning phase
itself was ever independently detectable, which is a reasonable thing to
also report in the paper if space allows).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .data.schema import Event
from .synth import LMEpisode


@dataclass(frozen=True)
class ConditioningSpec:
    """Parameters of one boiling-frog conditioning campaign.

    Attributes
    ----------
    start_s, end_s : int
        Conditioning window; by convention `end_s` should be at or
        before the target `LMEpisode.start_s` (see
        `boiling_frog_scenario`, which enforces this automatically).
    actor_user, actor_host : str
        Same identity used for the later real attack -- the scenario
        models one already-compromised foothold being used both to
        condition the model and, later, to actually move laterally.
    target_host : str
        The host the attacker wants the detector to have already
        learned is "a normal destination for actor_host" by the time
        the real attack happens.
    events_per_day : float
        Mean conditioning events per day (Poisson rate). Kept low by
        default specifically so the aggregate per-window statistics a
        `lmsafe.drift` detector watches do not shift sharply -- see
        `lmsafe.policies.DriftTriggeredPolicy`'s docstring.
    label_as_malicious : bool
        See module docstring. Default False.
    seed : int
        RNG seed for the Poisson arrival process and jitter.
    """

    start_s: int
    end_s: int
    actor_user: str
    actor_host: str
    target_host: str
    events_per_day: float = 2.0
    label_as_malicious: bool = False
    seed: int = 0


def generate_conditioning_events(spec: ConditioningSpec) -> list[Event]:
    """Generate the conditioning event stream for one `ConditioningSpec`.

    Arrivals follow a homogeneous Poisson process at rate
    `events_per_day` over `[start_s, end_s]` -- a deliberately
    unremarkable, memoryless arrival pattern (no burstiness that would
    itself be a detectable signature).
    """
    if spec.end_s <= spec.start_s:
        return []
    rng = np.random.default_rng(spec.seed)
    duration_days = (spec.end_s - spec.start_s) / 86400.0
    n_events = rng.poisson(spec.events_per_day * duration_days)
    times = rng.integers(spec.start_s, spec.end_s, size=n_events) if n_events > 0 else []

    events = [
        Event(
            t=int(t),
            src_user=spec.actor_user,
            dst_user=spec.actor_user,
            src_comp=spec.actor_host,
            dst_comp=spec.target_host,
            auth_type="Kerberos",
            logon_type="LogOn",
            orientation="LogOn",
            success=True,
            is_malicious=spec.label_as_malicious,
            source="synthetic_conditioning",
        )
        for t in sorted(times)
    ]
    return events


def inject_conditioning(events: list[Event], spec: ConditioningSpec) -> list[Event]:
    """Merge a conditioning campaign into an existing, time-sorted event
    list and return a new, still time-sorted list.

    `events` is not modified in place (a new, sorted list is returned),
    so a caller can cheaply produce both an "attack-only" and an
    "attack-plus-conditioning" variant of the same base stream for a
    paired RQ2 comparison, e.g.::

        base = synth.generate(cfg, episodes=[episode])
        spec = boiling_frog_scenario(cfg, episode)
        conditioned = attack.inject_conditioning(base, spec)
        # compare detector/policy performance on `base` vs `conditioned`
    """
    conditioning_events = generate_conditioning_events(spec)
    if not conditioning_events:
        return list(events)
    merged = list(events) + conditioning_events
    merged.sort(key=lambda e: e.t)
    return merged


def boiling_frog_scenario(
    episode: LMEpisode,
    conditioning_days: float = 10.0,
    events_per_day: float = 2.0,
    gap_seconds: int = 3600,
    label_as_malicious: bool = False,
    seed: int = 0,
) -> ConditioningSpec:
    """Build a `ConditioningSpec` that targets the same actor and the
    first hop of a given `LMEpisode`, ending `gap_seconds` before the
    episode starts.

    This is the convenience constructor every RQ2 experiment config
    actually uses: it guarantees the conditioning campaign's
    `actor_host`/`actor_user` exactly match the subsequent real attack
    (the scenario's premise -- one compromised foothold used for both),
    and that conditioning never overlaps the attack itself.

    Parameters
    ----------
    episode : LMEpisode
        The real lateral-movement episode this conditioning campaign
        precedes. `target_host` is set to `episode.target_hosts[0]`
        (the attacker's first real hop) -- the host the attacker most
        wants the detector to have already learned is unremarkable.
    conditioning_days : float
        Length of the conditioning campaign.
    gap_seconds : int
        Quiet gap enforced between the end of conditioning and the
        start of the real episode, so the two phases are never
        ambiguous about which window they fall in.
    """
    end_s = episode.start_s - gap_seconds
    start_s = end_s - int(conditioning_days * 86400)
    return ConditioningSpec(
        start_s=max(start_s, 0),
        end_s=max(end_s, 0),
        actor_user=episode.actor_user,
        actor_host=episode.actor_host,
        target_host=episode.target_hosts[0],
        events_per_day=events_per_day,
        label_as_malicious=label_as_malicious,
        seed=seed,
    )


In [ ]:
%%writefile lmsafe/metrics.py
"""
Evaluation metrics.

Everything here consumes a list of `WindowScores` -- one entry per test
window, each holding that window's per-edge anomaly scores alongside its
ground-truth labels and event-count weights -- in the same time order
`lmsafe.runner` produced them in. Keeping the window structure (instead
of flattening immediately into one global array) is what makes two
things possible that a flat array cannot support correctly:

1. **A per-window alert budget.** "Rec@B" in this project means: a SOC
   analyst can review at most `B` alerts *per window*, not `B` alerts
   total across the whole multi-week test stream (which would be an
   unrealistic, hindsight-only budget). `recall_at_budget` below ranks
   each window's edges independently and takes each window's own top-B.

2. **Correct, time-aware bootstrap.** Authentication events inside one
   window, and consecutive windows inside one attack episode, are
   strongly correlated -- they are not i.i.d. draws. `block_bootstrap_ci`
   resamples contiguous *blocks of windows* (a standard moving-block
   bootstrap for time series) rather than individual edges, so its
   confidence intervals are not artificially narrow. `episode_bootstrap_ci`
   is the complementary tool for per-episode summary statistics (such as
   detection delay) where the natural unit of resampling is "one
   episode", not "one window" or "one edge".

Score convention (shared with `lmsafe.detectors`): **higher score =
more anomalous**, so every function here ranks/thresholds on
`scores` directly, with no sign flip.
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from typing import Callable, Optional, Sequence

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

from .data.windows import WindowGraph


@dataclass
class WindowScores:
    """One test window's scores, labels, and event-count weights,
    aligned by index to each other (and to the `WindowGraph.edges` they
    came from, though the `WindowGraph` itself is not kept here).

    Attributes
    ----------
    window_start : int
        Same clock as `Event.t` / `WindowGraph.window_start`.
    scores : np.ndarray
        Anomaly scores, higher = more anomalous.
    is_malicious : np.ndarray (bool)
        Ground truth. This is the *only* place in the whole package
        where it is legitimate to read `WindowGraph.edge_is_malicious`
        -- `lmsafe.runner` builds one `WindowScores` per window
        specifically to hand ground truth to `lmsafe.metrics` and
        nowhere else.
    weight : np.ndarray (int)
        Event count each row represents (`WindowGraph.edge_weight`) --
        used by the `weighted_*` metrics to recover event-level
        statistics from edge-deduplicated rows without having to
        re-materialize every individual event.
    """

    window_start: int
    scores: np.ndarray
    is_malicious: np.ndarray
    weight: np.ndarray

    @classmethod
    def from_window(cls, window: WindowGraph, scores: np.ndarray) -> "WindowScores":
        return cls(
            window_start=window.window_start,
            scores=np.asarray(scores, dtype=np.float64),
            is_malicious=np.asarray(window.edge_is_malicious, dtype=bool),
            weight=np.asarray(window.edge_weight, dtype=np.float64),
        )


def _concat(records: Sequence[WindowScores]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not records:
        return np.empty(0), np.empty(0, dtype=bool), np.empty(0)
    scores = np.concatenate([r.scores for r in records])
    labels = np.concatenate([r.is_malicious for r in records])
    weights = np.concatenate([r.weight for r in records])
    return scores, labels, weights


def weighted_auc(records: Sequence[WindowScores]) -> float:
    """Event-level ROC-AUC: every row's contribution is multiplied by
    its event count (`weight`), via `sklearn`'s native `sample_weight`
    support -- recovering the AUC you would get by scoring every
    original (non-deduplicated) event individually, without ever
    materializing that larger array.

    Returns `nan` if `records` contains only one class (no malicious
    edges at all, or -- pathologically on a tiny bootstrap block --
    no benign edges), since AUC is undefined in that case. Callers
    doing a bootstrap (see `block_bootstrap_ci`) should expect and
    filter `nan` draws rather than treat them as zero.
    """
    scores, labels, weights = _concat(records)
    if len(labels) == 0 or labels.all() or not labels.any():
        return float("nan")
    return float(roc_auc_score(labels, scores, sample_weight=weights))


def weighted_ap(records: Sequence[WindowScores]) -> float:
    """Event-level average precision (area under the precision-recall
    curve), analogous to `weighted_auc`. `nan` if there are no
    malicious events at all in `records`."""
    scores, labels, weights = _concat(records)
    if len(labels) == 0 or not labels.any():
        return float("nan")
    return float(average_precision_score(labels, scores, sample_weight=weights))


def recall_at_budget(
    records: Sequence[WindowScores],
    budget_per_window: int,
    *,
    weighted: bool = True,
) -> float:
    """Recall at a fixed **per-window** alert budget ("Rec@B").

    For every window, the `budget_per_window` highest-scoring rows are
    "alerts" (ties broken by the stable order `numpy.argsort` returns,
    which for a deterministic detector is itself deterministic).
    Recall is accumulated across the whole stream as
    ``caught / total_malicious``, where both are summed in *event*
    units if `weighted=True` (using each row's `weight`) or in *row*
    (deduplicated-edge) units if `weighted=False`.

    A window with zero malicious rows contributes to neither the
    numerator nor the denominator (it can only ever generate false
    positives, which this metric does not score -- see `weighted_ap`/
    `weighted_auc` for precision-sensitive metrics). Returns `nan` if
    no window in `records` contains any malicious row.
    """
    caught = 0.0
    total = 0.0
    for r in records:
        if not r.is_malicious.any():
            continue
        n = len(r.scores)
        b = min(budget_per_window, n)
        top = np.argsort(-r.scores)[:b]
        alerted = np.zeros(n, dtype=bool)
        alerted[top] = True
        w = r.weight if weighted else np.ones(n)
        total += float(w[r.is_malicious].sum())
        caught += float(w[r.is_malicious & alerted].sum())
    return caught / total if total > 0 else float("nan")


# ---------------------------------------------------------------------------
# Episode clustering and detection delay
# ---------------------------------------------------------------------------


def cluster_episodes(
    records: Sequence[WindowScores], gap_windows: int = 1
) -> list[list[tuple[int, int]]]:
    """Group malicious rows into attack episodes by temporal proximity.

    Two malicious *windows* (not individual rows) belong to the same
    episode if they are separated by at most `gap_windows` windows with
    no malicious activity at all. This is the same gap-based clustering
    this project used to inspect the real LANL red-team file by hand
    (there, a 6-hour gap separated 19 episodes and a 24-hour gap
    separated 8) -- generalized here to operate in units of windows so
    it works at any `window_seconds`.

    Returns
    -------
    list[list[tuple[int, int]]]
        One list per episode, each containing `(record_index, edge_index)`
        pairs identifying every malicious row in that episode, in the
        order they occur. `record_index` indexes into `records`.
    """
    malicious_positions = sorted(i for i, r in enumerate(records) if r.is_malicious.any())
    if not malicious_positions:
        return []

    groups: list[list[int]] = []
    current = [malicious_positions[0]]
    for pos in malicious_positions[1:]:
        if pos - current[-1] > gap_windows + 1:
            groups.append(current)
            current = []
        current.append(pos)
    groups.append(current)

    episodes: list[list[tuple[int, int]]] = []
    for group in groups:
        edges = [
            (i, j)
            for i in group
            for j in np.flatnonzero(records[i].is_malicious)
        ]
        episodes.append(edges)
    return episodes


def detection_delay_per_episode(
    records: Sequence[WindowScores],
    budget_per_window: int,
    episodes: Sequence[Sequence[tuple[int, int]]],
) -> list[Optional[int]]:
    """For each episode, the number of windows between the episode's
    first malicious window and the first window in which *any* of that
    episode's malicious rows is caught in the per-window top-B alerts.

    Returns `None` for an episode never caught within `records` (the
    caller should report "fraction of episodes never caught at all" as
    a distinct statistic -- see module docstring -- rather than silently
    dropping these; `episode_bootstrap_ci` expects `None`s to already be
    filtered out, by design, so that an unbounded "delay" does not
    distort a mean).
    """
    delays: list[Optional[int]] = []
    for edges in episodes:
        if not edges:
            delays.append(None)
            continue
        first_window = min(i for i, _ in edges)
        by_window: dict[int, list[int]] = defaultdict(list)
        for i, j in edges:
            by_window[i].append(j)

        caught_at = None
        for i in sorted(by_window):
            r = records[i]
            n = len(r.scores)
            b = min(budget_per_window, n)
            top = set(np.argsort(-r.scores)[:b].tolist())
            if any(j in top for j in by_window[i]):
                caught_at = i
                break
        delays.append(None if caught_at is None else caught_at - first_window)
    return delays


def episode_hit(
    records: Sequence[WindowScores],
    budget_per_window: int,
    episodes: Sequence[Sequence[tuple[int, int]]],
) -> list[bool]:
    """Per-episode binary indicator: was this episode caught at all
    (any malicious row inside the per-window top-B, at any point)?
    A convenient, bootstrap-able companion to `detection_delay_per_episode`
    for episodes that are never caught at all (whose delay is `None`).
    """
    return [d is not None for d in detection_delay_per_episode(records, budget_per_window, episodes)]


# ---------------------------------------------------------------------------
# Bootstrap confidence intervals
# ---------------------------------------------------------------------------


def block_bootstrap_ci(
    metric_fn: Callable[[Sequence[WindowScores]], float],
    records: Sequence[WindowScores],
    *,
    block_size: int = 24,
    n_boot: int = 500,
    ci: float = 0.95,
    seed: int = 0,
) -> tuple[float, float, float]:
    """Moving-block-bootstrap confidence interval for a window-level
    metric (`weighted_auc`, `weighted_ap`, or a `recall_at_budget`
    partial application).

    Resamples `len(records) // block_size` contiguous blocks of
    `block_size` consecutive windows, with replacement, concatenates
    them into one bootstrap replicate, and recomputes `metric_fn` on
    it -- preserving within-block time correlation instead of treating
    windows (or worse, individual edges) as independent. `block_size`
    should be chosen comfortably larger than the attack episodes you
    care about (e.g. >= the longest episode duration in windows) so a
    block only rarely splits an episode in half.

    Draws where `metric_fn` returns `nan` (e.g. a block with no
    malicious rows at all, for AUC/AP) are dropped before computing
    percentiles; `n_valid` fewer than `n_boot` is reported implicitly
    by a wider-than-expected interval and should be checked by the
    caller if `n_boot` is small.

    Returns
    -------
    (point_estimate, lo, hi)
        `point_estimate` is `metric_fn(records)` on the *original*
        (non-resampled) data; `lo`/`hi` are the `ci` percentile bounds
        of the bootstrap distribution.
    """
    point = metric_fn(records)
    n = len(records)
    if n == 0:
        return point, float("nan"), float("nan")
    block_size = min(block_size, n)
    n_blocks = max(1, n // block_size)
    rng = np.random.default_rng(seed)
    alpha = (1.0 - ci) / 2.0

    boot_vals = []
    max_start = max(n - block_size, 0)
    for _ in range(n_boot):
        resampled: list[WindowScores] = []
        for _ in range(n_blocks):
            start = int(rng.integers(0, max_start + 1))
            resampled.extend(records[start : start + block_size])
        val = metric_fn(resampled)
        if not np.isnan(val):
            boot_vals.append(val)

    if not boot_vals:
        return point, float("nan"), float("nan")
    lo, hi = np.percentile(boot_vals, [100 * alpha, 100 * (1 - alpha)])
    return point, float(lo), float(hi)


def episode_bootstrap_ci(
    values: Sequence[float],
    *,
    n_boot: int = 1000,
    ci: float = 0.95,
    seed: int = 0,
) -> tuple[float, float, float]:
    """Percentile bootstrap over a list of per-episode scalar values
    (e.g. `detection_delay_per_episode` output with `None`s already
    filtered out, or `episode_hit` cast to 0/1).

    This is the right tool specifically when the natural "unit of
    independence" is one episode -- as opposed to `block_bootstrap_ci`,
    which resamples contiguous time blocks and is for metrics (AUC, AP,
    Rec@B) computed over the full mix of benign and malicious rows.
    Mixing these up -- e.g. bootstrapping Rec@B by resampling episodes
    alone, ignoring the benign majority of the stream -- would
    understate the true uncertainty and is exactly the mistake this
    project's evaluation protocol is designed to avoid (see
    `lmsafe.config` module docstring and README "Why a fair split
    matters").
    """
    vals = np.asarray(list(values), dtype=np.float64)
    n = len(vals)
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    point = float(vals.mean())
    rng = np.random.default_rng(seed)
    alpha = (1.0 - ci) / 2.0
    boot_means = [
        float(vals[rng.integers(0, n, size=n)].mean()) for _ in range(n_boot)
    ]
    lo, hi = np.percentile(boot_means, [100 * alpha, 100 * (1 - alpha)])
    return point, float(lo), float(hi)


In [ ]:
%%writefile lmsafe/runner.py
"""
Experiment orchestration: data -> policy loop -> detector -> metrics,
for one "experiment cell" (one detector x one policy x one dataset
variant), plus a grid runner that sweeps many cells into one tidy table.

This module intentionally knows nothing about *where* `train_windows`/
`test_windows` came from -- LANL, OpTC, or `lmsafe.synth` -- which is the
entire point of the `Event`/`WindowGraph` boundary described in
`lmsafe.data.schema`. `scripts/run_experiment.py` is the thin,
dataset-aware layer on top of this module that a YAML config selects.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Optional, Sequence

import pandas as pd

from .data.windows import WindowGraph
from .detectors.base import Detector
from .metrics import (
    WindowScores,
    cluster_episodes,
    detection_delay_per_episode,
    episode_bootstrap_ci,
    episode_hit,
    recall_at_budget,
    weighted_ap,
    weighted_auc,
)
from .policies import UpdatePolicy


@dataclass
class ExperimentResult:
    """Everything one (detector, policy, dataset-variant) cell produced.

    Kept as a dataclass (not just a dict) so `run_grid`'s DataFrame has
    a documented, stable set of columns -- a typo in a dict key used to
    build a results table is a silent empty column, not an error.
    """

    detector_name: str
    policy_name: str
    scenario: str
    weighted_auc: float
    weighted_ap: float
    rec_at_budget: dict[int, float]
    episode_hit_rate: float
    mean_delay_windows: Optional[float]
    delay_ci_lo: Optional[float]
    delay_ci_hi: Optional[float]
    n_episodes: int
    n_test_windows: int
    diagnostics: dict = field(default_factory=dict)

    def to_row(self) -> dict:
        """Flatten into one dict suitable for a pandas row -- one
        column per alert budget (`rec_at_1`, `rec_at_5`, ...)."""
        row = {
            "detector": self.detector_name,
            "policy": self.policy_name,
            "scenario": self.scenario,
            "weighted_auc": self.weighted_auc,
            "weighted_ap": self.weighted_ap,
            "episode_hit_rate": self.episode_hit_rate,
            "mean_delay_windows": self.mean_delay_windows,
            "delay_ci_lo": self.delay_ci_lo,
            "delay_ci_hi": self.delay_ci_hi,
            "n_episodes": self.n_episodes,
            "n_test_windows": self.n_test_windows,
        }
        for b, r in self.rec_at_budget.items():
            row[f"rec_at_{b}"] = r
        for k, v in self.diagnostics.items():
            row[f"diag_{k}"] = v
        return row


def run_experiment_cell(
    train_windows: Sequence[WindowGraph],
    test_windows: Sequence[WindowGraph],
    detector: Detector,
    policy: UpdatePolicy,
    *,
    detector_name: str,
    scenario: str = "default",
    budgets: Sequence[int] = (1, 5, 10, 20),
    gap_windows_for_episodes: int = 2,
    delay_budget: Optional[int] = None,
    delay_n_boot: int = 500,
    delay_seed: int = 0,
) -> ExperimentResult:
    """Run one detector+policy combination over one test stream and
    compute every metric this project reports.

    Parameters
    ----------
    train_windows, test_windows : Sequence[WindowGraph]
        From `lmsafe.data.lanl.split_train_test` (fed through
        `lmsafe.data.windows.iter_windows`) or the equivalent synthetic
        split in `scripts/run_experiment.py`.
    detector : Detector
        A fresh (or pre-configured) detector instance; `policy.fit_initial`
        is called on it here, so pass an unfitted instance.
    policy : UpdatePolicy
    detector_name, scenario : str
        Free-form labels copied into the result/row, e.g.
        `scenario="drift_strength=0.5"` or `scenario="conditioned"` --
        whatever a given RQ script is sweeping over.
    budgets : Sequence[int]
        Per-window alert budgets to report Rec@B for.
    delay_budget : int, optional
        Which budget's alert set to use for `detection_delay_per_episode`
        / `episode_hit`. Defaults to the middle value of `budgets`.
    """
    policy.fit_initial(detector, list(train_windows))

    records: list[WindowScores] = []
    for w in test_windows:
        scores = detector.score(w)
        records.append(WindowScores.from_window(w, scores))
        detector = policy.observe(w, scores, detector)

    auc = weighted_auc(records)
    ap = weighted_ap(records)
    rec_at_budget = {b: recall_at_budget(records, b) for b in budgets}

    episodes = cluster_episodes(records, gap_windows=gap_windows_for_episodes)
    db = delay_budget if delay_budget is not None else sorted(budgets)[len(budgets) // 2]
    hits = episode_hit(records, db, episodes)
    delays = [d for d in detection_delay_per_episode(records, db, episodes) if d is not None]

    hit_rate = (sum(hits) / len(hits)) if hits else float("nan")
    if delays:
        mean_delay, lo, hi = episode_bootstrap_ci(delays, n_boot=delay_n_boot, seed=delay_seed)
    else:
        mean_delay, lo, hi = float("nan"), float("nan"), float("nan")

    diagnostics = dict(detector.state_summary())
    diagnostics.update(
        {f"policy_{k}": v for k, v in vars(policy).items() if isinstance(v, (int, float, str))}
    )

    return ExperimentResult(
        detector_name=detector_name,
        policy_name=policy.name,
        scenario=scenario,
        weighted_auc=auc,
        weighted_ap=ap,
        rec_at_budget=rec_at_budget,
        episode_hit_rate=hit_rate,
        mean_delay_windows=mean_delay,
        delay_ci_lo=lo,
        delay_ci_hi=hi,
        n_episodes=len(episodes),
        n_test_windows=len(test_windows),
        diagnostics=diagnostics,
    )


def run_grid(
    cells: Sequence[dict],
    *,
    train_windows_fn: Callable[[dict], Sequence[WindowGraph]],
    test_windows_fn: Callable[[dict], Sequence[WindowGraph]],
    **common_kwargs,
) -> pd.DataFrame:
    """Run a list of experiment cells and return one tidy `DataFrame`.

    Parameters
    ----------
    cells : Sequence[dict]
        Each dict must have keys `"detector"` (a `Detector` instance,
        unfitted), `"detector_name"` (str), `"policy"` (an
        `UpdatePolicy` instance), and `"scenario_key"` (any hashable,
        passed to `train_windows_fn`/`test_windows_fn` to select which
        data variant this cell runs on -- e.g. a `drift_strength` value
        or `"conditioned"` vs `"clean"`). Any extra keys are copied into
        `scenario` as a human-readable label if `"scenario"` itself is
        not given explicitly.
    train_windows_fn, test_windows_fn : Callable[[dict], Sequence[WindowGraph]]
        Given one cell's dict, return that scenario's train/test window
        lists. Memoize inside your own closure if several cells share
        the same `scenario_key` and regenerating the data is expensive
        -- `run_grid` itself does not cache.
    **common_kwargs
        Forwarded to every `run_experiment_cell` call (e.g. `budgets=`).

    Returns
    -------
    pd.DataFrame
        One row per cell, columns as in `ExperimentResult.to_row`.
    """
    rows = []
    for cell in cells:
        train_w = train_windows_fn(cell)
        test_w = test_windows_fn(cell)
        scenario = cell.get("scenario", str(cell.get("scenario_key", "default")))
        result = run_experiment_cell(
            train_w,
            test_w,
            detector=cell["detector"],
            policy=cell["policy"],
            detector_name=cell["detector_name"],
            scenario=scenario,
            **common_kwargs,
        )
        rows.append(result.to_row())
    return pd.DataFrame(rows)


In [ ]:
%%writefile lmsafe/factories.py
"""
Name -> object factories for detectors, policies, and drift detectors.

Exists so a YAML experiment config (`scripts/run_experiment.py`) or a
quick interactive session can build a detector/policy by name and a
dict of keyword arguments, instead of every caller importing every
concrete class directly. Every RQ script in `scripts/` also uses these,
so the set of valid names in one place (here) is the single source of
truth for what a config file is allowed to say.
"""
from __future__ import annotations

from typing import Any

from .detectors.base import Detector
from .detectors.count_baseline import CountBaselineDetector
from .detectors.feature_mlp import FeatureMLPDetector
from .detectors.rw_gnn import GraphFoundationDetector
from .drift import PageHinkley, SimplifiedADWIN
from .policies import (
    DriftTriggeredPolicy,
    FrozenPolicy,
    GatedPolicy,
    PeriodicPolicy,
    SlidingReplayPolicy,
    UpdatePolicy,
)

DETECTORS = {
    "count_baseline": CountBaselineDetector,
    "feature_mlp": FeatureMLPDetector,
    "graph_foundation": GraphFoundationDetector,
}

DRIFT_DETECTORS = {
    "page_hinkley": PageHinkley,
    "adwin": SimplifiedADWIN,
}


def make_detector(name: str, **kwargs: Any) -> Detector:
    try:
        cls = DETECTORS[name]
    except KeyError:
        raise ValueError(f"unknown detector {name!r}, expected one of {sorted(DETECTORS)}") from None
    return cls(**kwargs)


def make_drift_detector(name: str, **kwargs: Any):
    try:
        cls = DRIFT_DETECTORS[name]
    except KeyError:
        raise ValueError(
            f"unknown drift detector {name!r}, expected one of {sorted(DRIFT_DETECTORS)}"
        ) from None
    return cls(**kwargs)


def make_policy(name: str, **kwargs: Any) -> UpdatePolicy:
    """Build a policy by name. `drift_triggered` additionally requires
    `drift_detector_name` (+ optional `drift_detector_kwargs`) in
    `kwargs`, consumed here rather than passed through to
    `DriftTriggeredPolicy.__init__`, which expects an already-built
    drift-detector instance.
    """
    if name == "frozen":
        return FrozenPolicy(**kwargs)
    if name == "periodic":
        return PeriodicPolicy(**kwargs)
    if name == "sliding_replay":
        return SlidingReplayPolicy(**kwargs)
    if name == "drift_triggered":
        kwargs = dict(kwargs)
        dd_name = kwargs.pop("drift_detector_name", "page_hinkley")
        dd_kwargs = kwargs.pop("drift_detector_kwargs", {})
        drift_detector = make_drift_detector(dd_name, **dd_kwargs)
        return DriftTriggeredPolicy(drift_detector=drift_detector, **kwargs)
    if name == "gated":
        return GatedPolicy(**kwargs)
    raise ValueError(
        f"unknown policy {name!r}, expected one of "
        "['frozen', 'periodic', 'sliding_replay', 'drift_triggered', 'gated']"
    )


## 3) Install the package

In [ ]:
!pip install -e . --quiet
import lmsafe
print("lmsafe version:", lmsafe.__version__)

## 4) Write the scripts (`scripts/`)

In [ ]:
%%writefile scripts/fetch_lanl_window.py
#!/usr/bin/env python3
"""
Small CLI for quick, interactive LANL streaming checks -- the "peek at
the link and the data before committing to a multi-hour run" tool,
meant for a Colab cell. `scripts/run_experiment.py` (with a `lanl.yaml`
config) is what actually runs a full experiment; this script exists for
the smoke-test step before that, and for materializing a cached
window-graph file "offline" (once) that `run_experiment.py`'s
`lanl.cache_path` can then reuse.

    export LMSAFE_LANL_BASE_URL='<signed link>'   # or let it prompt you

    # 1) smoke test (seconds): first lines of a file + the red-team file
    python scripts/fetch_lanl_window.py peek --file redteam --n 5
    python scripts/fetch_lanl_window.py peek --file auth --n 5

    # 2) save the small red-team labels as CSV
    python scripts/fetch_lanl_window.py redteam --out redteam.csv

    # 3) stream+window a time range and pickle the result for
    #    run_experiment.py's lanl.cache_path to pick up directly
    python scripts/fetch_lanl_window.py build-cache \\
        --end-s 2592000 --out lmsafe_cache/lanl_dev_windows.pkl
"""
from __future__ import annotations

import argparse
import csv
import sys
import time
from pathlib import Path

from lmsafe.config import LanlConfig
from lmsafe.data import lanl, lanl_stream
from lmsafe.data.windows import iter_windows


def cmd_peek(a: argparse.Namespace) -> None:
    base_url = lanl_stream.resolve_base_url()
    url = lanl_stream.file_url(base_url, a.file)
    for i, ln in enumerate(lanl_stream.iter_lines(url)):
        print(ln.decode("utf-8", "replace"))
        if i + 1 >= a.n:
            break


def cmd_redteam(a: argparse.Namespace) -> None:
    base_url = lanl_stream.resolve_base_url()
    records = lanl.load_redteam_records(base_url)
    with open(a.out, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["time", "user", "src_comp", "dst_comp"])
        for r in records:
            w.writerow([r.t, r.user, r.src_comp, r.dst_comp])
    print(f"wrote {len(records)} red-team rows to {a.out} (the public release documents 749)")


#: Re-exported under this name for backward compatibility (and because
#: this module's own tests reference it directly); the real
#: implementation lives in `lmsafe.data.lanl` so `run_experiment.py`'s
#: `cache_path` checkpointing can share the exact same crash-safety
#: guarantee without duplicating it.
_atomic_pickle_dump = lanl.atomic_pickle_dump


def cmd_build_cache(a: argparse.Namespace) -> None:
    cfg = LanlConfig(
        end_s=a.end_s,
        start_s=a.start_s,
        train_cutoff_s=a.train_cutoff_s,
        only_remote_logon=not a.no_logon_filter,
    )
    base_url = lanl_stream.resolve_base_url()
    redteam = lanl.load_redteam_records(base_url)
    events_iter = lanl.stream_auth_events(cfg, base_url=base_url, redteam=redteam)
    train_events, test_events_iter = lanl.split_train_test(events_iter, cfg)

    train_w = list(iter_windows(train_events, cfg.window_seconds))

    out = Path(a.out)
    out.parent.mkdir(parents=True, exist_ok=True)

    # Stream-build the (potentially many-hour) test-window list with
    # periodic checkpoints, instead of only writing `out` once at the
    # very end. A multi-hour real-LANL run (58 days = 1B+ raw lines) has
    # a real chance of being killed by a Colab disconnect, a Ctrl+C, or
    # a network hiccup before it reaches the final line -- and
    # `iter_windows`/`stream_auth_events` are plain generators with
    # nothing written to disk until this function's caller used to do
    # it in one shot at the end. Without checkpoints, a run interrupted
    # at 90% progress leaves *zero* usable output (this is exactly what
    # was reported: a 58-day run reached day 50.3 of 58, got Ctrl+C'd,
    # and Google Drive came up `total 0` because nothing had ever been
    # written). Checkpointing every `checkpoint_every` windows means an
    # interrupted run still leaves a near-complete, directly-usable
    # cache file at `out` -- `run_experiment.py` cannot tell a
    # checkpoint from a finished cache, since both are just a
    # `(train_w, test_w)` pickle.
    test_w: list = []
    last_checkpoint_t = time.time()
    for w in iter_windows(test_events_iter, cfg.window_seconds):
        test_w.append(w)
        if len(test_w) % a.checkpoint_every == 0:
            _atomic_pickle_dump((train_w, test_w), out)
            print(
                f"[build-cache] checkpoint: {len(train_w)} train windows + "
                f"{len(test_w)} test windows so far -> {out} "
                f"({time.time() - last_checkpoint_t:.0f}s since last checkpoint)",
                file=sys.stderr,
                flush=True,
            )
            last_checkpoint_t = time.time()

    _atomic_pickle_dump((train_w, test_w), out)
    print(
        f"wrote {len(train_w)} train windows + {len(test_w)} test windows to {out} "
        f"-- point scripts/run_experiment.py's config at lanl.cache_path: {out}"
    )


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    p = sub.add_parser("peek", help="print the first N lines of a cyber1 file (link smoke test)")
    p.add_argument("--file", choices=lanl_stream.FILE_NAMES, default="auth")
    p.add_argument("--n", type=int, default=5)
    p.set_defaults(fn=cmd_peek)

    p = sub.add_parser("redteam", help="save the red-team ground truth as CSV")
    p.add_argument("--out", default="redteam.csv")
    p.set_defaults(fn=cmd_redteam)

    p = sub.add_parser("build-cache", help="stream+window a time range and pickle it for run_experiment.py")
    p.add_argument("--start-s", type=int, default=0)
    p.add_argument("--end-s", type=int, required=True)
    p.add_argument("--train-cutoff-s", type=int, default=147_600)
    p.add_argument("--no-logon-filter", action="store_true", help="disable the fair remote-LogOn filter (debugging only)")
    p.add_argument("--out", required=True)
    p.add_argument(
        "--checkpoint-every",
        type=int,
        default=24,
        help=(
            "write a partial (train_w, test_w) checkpoint to --out every N test "
            "windows (default 24, roughly once per simulated day at the default "
            "1-hour window length), so a multi-hour real-LANL run that gets "
            "interrupted (Ctrl+C, a Colab disconnect, a network drop) still "
            "leaves a usable, directly-loadable cache file instead of nothing"
        ),
    )
    p.set_defaults(fn=cmd_build_cache)

    a = ap.parse_args()
    a.fn(a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_experiment.py
#!/usr/bin/env python3
"""
Generic, config-driven experiment runner.

This is the one script meant to be pointed at the *real* LANL (or,
once `lmsafe.data.optc` is implemented, OpTC) data on Colab -- every
other script under `scripts/` is a self-contained synthetic
demonstration of one research question. A YAML config says which
dataset to use, which detector, and which policies to compare; this
script builds the data once, runs every policy against it, and writes
one results CSV.

Config schema
---------------
    source: synthetic | lanl          # which lmsafe.data reader to use

    detector:
      name: count_baseline | feature_mlp
      kwargs: {...}                    # forwarded to the detector class

    policies:
      - name: frozen | periodic | sliding_replay | drift_triggered | gated
        kwargs: {...}                  # forwarded to the policy class
                                        # (drift_triggered additionally
                                        # takes drift_detector_name:
                                        # page_hinkley|adwin and
                                        # drift_detector_kwargs)

    budgets: [1, 5, 10, 20]            # per-window alert budgets for Rec@B

    output:
      dir: results/my_run
      csv: results.csv

    # when source: synthetic -- see lmsafe.config.SynthConfig fields
    synth:
      n_users: 40
      ...

    # when source: lanl -- see lmsafe.config.LanlConfig fields
    lanl:
      end_s: 2592000                   # LANL_DEV_CUTOFF_S (30 days); use
                                        # LANL_FULL_CUTOFF_S (58 days) for
                                        # final paper numbers
      train_cutoff_s: 147600           # LANL_TRAIN_CUTOFF_S (41 hours)
      only_remote_logon: true
      filter_ntlm_only: false
      cache_path: lmsafe_cache/lanl_dev_windows.pkl
                                        # OPTIONAL but strongly recommended:
                                        # the streamed-and-windowed
                                        # (train, test) WindowGraph lists
                                        # are pickled here on first run and
                                        # loaded directly on every later
                                        # run, so a second detector/policy
                                        # sweep never re-streams the 7.2 GB
                                        # auth file (and keeps working even
                                        # after the signed LANL link has
                                        # expired). Written incrementally
                                        # (see checkpoint_every below), not
                                        # only once at the end, so an
                                        # interrupted multi-hour run still
                                        # leaves a usable partial file.
      checkpoint_every: 24              # OPTIONAL, only used with cache_path:
                                        # write a partial checkpoint every N
                                        # test windows (default 24, roughly
                                        # once per simulated day at the
                                        # default 1-hour window length).

Example configs are in `configs/`.

Usage
------
    python scripts/run_experiment.py configs/synthetic_quick.yaml
    python scripts/run_experiment.py configs/lanl_dev.yaml       # on Colab,
                                                                    # with
                                                                    # LMSAFE_LANL_BASE_URL set
"""
from __future__ import annotations

import argparse
import pickle
import sys
from pathlib import Path

import yaml

from lmsafe.config import LanlConfig, SynthConfig
from lmsafe.data import lanl
from lmsafe.data.windows import WindowGraph, iter_windows
from lmsafe.factories import make_detector, make_policy
from lmsafe.runner import run_experiment_cell


def build_synthetic_windows(cfg_dict: dict) -> tuple[list[WindowGraph], list[WindowGraph]]:
    from lmsafe import synth
    from lmsafe.config import LANL_TRAIN_CUTOFF_S

    # The training cutoff is always lmsafe.config.LANL_TRAIN_CUTOFF_S
    # (41 hours), reused here for synthetic data too so every scenario
    # in this project shares exactly one definition of "the benign
    # training period" -- see that constant's own docstring.
    cfg = SynthConfig(**cfg_dict)
    events = synth.generate(cfg)
    windows = list(iter_windows(events, cfg.window_seconds))
    train_w = [w for w in windows if w.window_start < LANL_TRAIN_CUTOFF_S]
    test_w = [w for w in windows if w.window_start >= LANL_TRAIN_CUTOFF_S]
    return train_w, test_w


def build_lanl_windows(cfg_dict: dict) -> tuple[list[WindowGraph], list[WindowGraph]]:
    cache_path = cfg_dict.pop("cache_path", None)
    # How often (in test windows) to write a crash-safe partial
    # checkpoint to `cache_path` while streaming -- see the loop below.
    # Only meaningful when `cache_path` is set, since there is nowhere
    # else to checkpoint to.
    checkpoint_every = cfg_dict.pop("checkpoint_every", 24)
    if cache_path and Path(cache_path).exists():
        print(f"[run_experiment] loading cached LANL windows from {cache_path}")
        with open(cache_path, "rb") as fh:
            cached_train_w, cached_test_w = pickle.load(fh)
        # This cache may be a *finished* run's output or an interrupted
        # run's partial checkpoint -- the pickle format is identical
        # either way (see the long comment below), so this is the only
        # place that can warn a human the loaded data might be short of
        # the config's own `end_s`. Confirmed necessary in practice: a
        # real 58-day run was Ctrl+C'd at day 40.64 after repeated
        # network reconnects, and silently reloading that partial cache
        # on a later run would otherwise look identical to a genuine,
        # complete 58-day run in every later log line.
        requested_days = cfg_dict.get("end_s", 0) / 86400
        if cached_test_w:
            covered_days = max(w.window_start + w.window_seconds for w in cached_test_w) / 86400
        else:
            covered_days = 0.0
        print(
            f"[run_experiment] cache contains {len(cached_train_w)} train windows + "
            f"{len(cached_test_w)} test windows, covering up to day {covered_days:.2f} "
            f"of the {requested_days:.1f} days this config asks for"
        )
        if requested_days and covered_days < requested_days - 0.05:
            print(
                f"[run_experiment] WARNING: this cache covers only {covered_days:.2f} of "
                f"{requested_days:.1f} requested days -- it looks like a partial checkpoint "
                f"from an interrupted run, not a finished one. Metrics computed from it are "
                f"real but describe a SHORTER evaluation window than this config's label "
                f"implies. To get the full window, delete this cache file and rerun (the "
                f"network stream itself cannot resume mid-file, so this re-reads from the "
                f"start); to knowingly keep the partial window, that is fine too -- just "
                f"don't report it as the full-window number."
            )
        return cached_train_w, cached_test_w

    cfg = LanlConfig(**{k: v for k, v in cfg_dict.items() if k != "cache_dir"})
    print(
        f"[run_experiment] streaming LANL auth events up to t={cfg.end_s:,}s "
        f"({cfg.end_s / 86400:.1f} days) -- this reads from the network, "
        "see lmsafe.data.lanl_stream for the signed-link setup"
    )
    base_url = lanl.lanl_stream.resolve_base_url()
    redteam = lanl.load_redteam_records(base_url)
    events_iter = lanl.stream_auth_events(cfg, base_url=base_url, redteam=redteam)
    train_events, test_events_iter = lanl.split_train_test(events_iter, cfg)

    train_w = list(iter_windows(train_events, cfg.window_seconds))

    # Stream-build the test-window list with periodic crash-safe
    # checkpoints to `cache_path`, instead of only writing it once at
    # the very end. A real LANL run (58 days = 1B+ raw lines) can take
    # hours, and a Colab disconnect, a Ctrl+C, or a network hiccup
    # before the final line used to leave *nothing* on disk at all --
    # the exact failure reported: a 58-day run reached day ~50 of 58,
    # was interrupted, and Google Drive came up `total 0` because the
    # single pickle.dump() at the end of this function never ran. Each
    # checkpoint is a complete, directly-loadable `(train_w, test_w)`
    # pickle -- the `if cache_path and Path(cache_path).exists()` branch
    # above cannot tell a checkpoint from a finished run, so a resumed
    # session picks up the most recent checkpoint transparently (at the
    # cost of re-covering the tail that wasn't yet checkpointed, since
    # this does not resume the network stream itself -- only the
    # output).
    test_w: list = []
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        for w in iter_windows(test_events_iter, cfg.window_seconds):
            test_w.append(w)
            if len(test_w) % checkpoint_every == 0:
                lanl.atomic_pickle_dump((train_w, test_w), Path(cache_path))
                print(
                    f"[run_experiment] checkpoint: {len(train_w)} train windows + "
                    f"{len(test_w)} test windows so far -> {cache_path}"
                )
        lanl.atomic_pickle_dump((train_w, test_w), Path(cache_path))
        print(f"[run_experiment] cached windows to {cache_path} for future runs")
    else:
        test_w = list(iter_windows(test_events_iter, cfg.window_seconds))

    return train_w, test_w


def main(config_path: Path) -> None:
    with open(config_path) as fh:
        config = yaml.safe_load(fh)

    source = config["source"]
    if source == "synthetic":
        train_w, test_w = build_synthetic_windows(dict(config.get("synth", {})))
    elif source == "lanl":
        train_w, test_w = build_lanl_windows(dict(config.get("lanl", {})))
    else:
        sys.exit(f"unknown source {source!r}, expected 'synthetic' or 'lanl'")

    print(f"[run_experiment] {len(train_w)} train windows, {len(test_w)} test windows")

    budgets = tuple(config.get("budgets", [1, 5, 10, 20]))
    out_dir = Path(config["output"]["dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / config["output"].get("csv", "results.csv")

    det_cfg = config["detector"]
    rows = []
    for pol_cfg in config["policies"]:
        detector = make_detector(det_cfg["name"], **det_cfg.get("kwargs", {}))
        policy = make_policy(pol_cfg["name"], **pol_cfg.get("kwargs", {}))
        result = run_experiment_cell(
            train_w,
            test_w,
            detector,
            policy,
            detector_name=det_cfg["name"],
            scenario=config.get("scenario_label", source),
            budgets=budgets,
        )
        print(
            f"[run_experiment] {policy.name:16s} AP={result.weighted_ap:.4f} "
            f"AUC={result.weighted_auc:.4f} "
            f"hit_rate={result.episode_hit_rate:.3f} "
            f"(n_episodes={result.n_episodes})"
        )
        rows.append(result.to_row())

    import pandas as pd

    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"[run_experiment] wrote {csv_path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("config", type=Path, help="path to a YAML experiment config")
    args = ap.parse_args()
    main(args.config)


In [ ]:
%%writefile scripts/run_rq1_drift_cost.py
#!/usr/bin/env python3
"""
RQ1 -- Drift cost: how much detection quality do different update
policies lose to *benign* drift alone, with no attacker conditioning at
all, and how fast do they recover?

Experiment design
-------------------
A single, fixed set of lateral-movement episodes (same actor hosts,
same target chains, same timing) is injected into four otherwise-
identical synthetic benign backgrounds that differ only in
`drift_strength` (0.0 = perfectly stationary, up to 1.0 = strong,
persistent population growth starting on `DRIFT_START_DAY`). Holding
the attack fixed and varying only the benign background isolates "cost
of drift" from "how hard this particular attack is to find" -- the
counterfactual RQ1 needs and that the real, single-recording LANL
dataset cannot offer on its own (see `lmsafe.synth` module docstring).

Every policy in `lmsafe.policies` is run against every drift level with
the `CountBaselineDetector` (fast enough for a full sweep on a 2-CPU
sandbox); `FeatureMLPDetector` is additionally spot-checked at the
strongest drift level only, as a sanity check that the finding is not
an artifact of the particular detector (a full sweep with it is left to
`scripts/run_experiment.py` + a YAML config, which can be pointed at
either detector for the real LANL/OpTC runs).

Usage
------
    python scripts/run_rq1_drift_cost.py [--out-dir results/rq1]

Outputs `<out-dir>/rq1_results.csv` and `<out-dir>/rq1_drift_cost.png`.
"""
from __future__ import annotations

import argparse
import copy
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from lmsafe import synth
from lmsafe.config import LANL_TRAIN_CUTOFF_S, SynthConfig
from lmsafe.data.windows import iter_windows
from lmsafe.detectors import CountBaselineDetector, FeatureMLPDetector
from lmsafe.policies import (
    DriftTriggeredPolicy,
    FrozenPolicy,
    GatedPolicy,
    PeriodicPolicy,
    SlidingReplayPolicy,
)
from lmsafe.drift import PageHinkley
from lmsafe.runner import run_experiment_cell

DRIFT_LEVELS = [0.0, 0.3, 0.6, 1.0]
DRIFT_START_DAY = 6
N_BENIGN_DAYS = 25
N_EPISODES = 5
EPISODE_CHAIN_LEN = 5
SEED = 42
BUDGETS = (1, 5, 10, 20)


def base_cfg(drift_strength: float) -> SynthConfig:
    return SynthConfig(
        n_users=40,
        n_hosts=25,
        n_benign_days=N_BENIGN_DAYS,
        base_edges_per_window=25,
        n_episodes=N_EPISODES,
        episode_chain_len=EPISODE_CHAIN_LEN,
        drift_start_day=DRIFT_START_DAY,
        drift_strength=drift_strength,
        seed=SEED,
    )


def fixed_episodes() -> list[synth.LMEpisode]:
    """Episodes placed once, on the drift_strength=0 population, and
    reused unchanged across every drift level (see module docstring)."""
    import numpy as np

    cfg0 = base_cfg(0.0)
    rng = np.random.default_rng(cfg0.seed)
    pop = synth.Population(cfg0, rng)
    return synth.default_episodes(cfg0, pop, rng)


def split(cfg: SynthConfig, episodes):
    events = synth.generate(cfg, episodes=episodes)
    windows = list(iter_windows(events, cfg.window_seconds))
    train_w = [w for w in windows if w.window_start < LANL_TRAIN_CUTOFF_S]
    test_w = [w for w in windows if w.window_start >= LANL_TRAIN_CUTOFF_S]
    return train_w, test_w


def policy_factory(name: str):
    if name == "frozen":
        return FrozenPolicy()
    if name == "periodic":
        return PeriodicPolicy(period=24)  # lookback defaults to period: disjoint, non-overlapping batches
    if name == "sliding_replay":
        return SlidingReplayPolicy(window_size=168, retrain_period=24)
    if name == "drift_triggered":
        return DriftTriggeredPolicy(PageHinkley(delta=0.01, threshold=8.0), lookback=168)
    if name == "gated":
        return GatedPolicy(
            score_admission_quantile=0.9,
            quarantine_windows=24,
            min_corroborating_src_hosts=3,
            retrain_period=24,
        )
    raise ValueError(name)


POLICY_NAMES = ["frozen", "periodic", "sliding_replay", "drift_triggered", "gated"]


def main(out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    episodes = fixed_episodes()
    print(f"[rq1] {len(episodes)} fixed lateral-movement episodes, reused at every drift level")

    rows = []
    for drift in DRIFT_LEVELS:
        cfg = base_cfg(drift)
        train_w, test_w = split(cfg, copy.deepcopy(episodes))
        print(f"[rq1] drift_strength={drift}: {len(train_w)} train / {len(test_w)} test windows")
        for pname in POLICY_NAMES:
            policy = policy_factory(pname)
            result = run_experiment_cell(
                train_w,
                test_w,
                CountBaselineDetector(),
                policy,
                detector_name="count_baseline",
                scenario=f"drift_strength={drift}",
                budgets=BUDGETS,
            )
            row = result.to_row()
            row["drift_strength"] = drift
            rows.append(row)
            print(
                f"[rq1]   {pname:16s} AP={result.weighted_ap:.4f} "
                f"Rec@10={result.rec_at_budget[10]:.3f} "
                f"hit_rate={result.episode_hit_rate:.2f}"
            )

    # Spot-check with the torch detector at the strongest drift level only.
    strong_cfg = base_cfg(DRIFT_LEVELS[-1])
    train_w, test_w = split(strong_cfg, copy.deepcopy(episodes))
    for pname in ("frozen", "gated"):
        result = run_experiment_cell(
            train_w,
            test_w,
            FeatureMLPDetector(epochs_per_fit=2, hidden_dim=12),
            policy_factory(pname),
            detector_name="feature_mlp",
            scenario=f"drift_strength={DRIFT_LEVELS[-1]}",
            budgets=BUDGETS,
        )
        row = result.to_row()
        row["drift_strength"] = DRIFT_LEVELS[-1]
        rows.append(row)
        print(f"[rq1] [feature_mlp spot-check] {pname:16s} AP={result.weighted_ap:.4f}")

    df = pd.DataFrame(rows)
    csv_path = out_dir / "rq1_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"[rq1] wrote {csv_path}")

    make_figure(df, out_dir / "rq1_drift_cost.png")


def make_figure(df: pd.DataFrame, path: Path) -> None:
    sub = df[df["detector"] == "count_baseline"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    for pname in POLICY_NAMES:
        s = sub[sub["policy"] == pname].sort_values("drift_strength")
        axes[0].plot(s["drift_strength"], s["weighted_ap"], marker="o", label=pname)
        axes[1].plot(s["drift_strength"], s["rec_at_10"], marker="o", label=pname)

    axes[0].set_xlabel("benign drift strength")
    axes[0].set_ylabel("weighted AP")
    axes[0].set_title("RQ1: average precision vs. drift")
    axes[1].set_xlabel("benign drift strength")
    axes[1].set_ylabel("Rec@10 (per-window alert budget)")
    axes[1].set_title("RQ1: recall@budget vs. drift")
    axes[1].legend(fontsize=8, loc="best")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    print(f"[rq1] wrote {path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out-dir", type=Path, default=Path("results/rq1"))
    args = ap.parse_args()
    main(args.out_dir)


In [ ]:
%%writefile scripts/run_rq2_poisoning.py
#!/usr/bin/env python3
"""
RQ2 -- Update poisoning: how much detection quality do the naive update
policies lose when an adaptive attacker injects conditioning edges
before attacking, compared to the same policies with no conditioning at
all?

Experiment design
-------------------
One benign background (fixed `drift_strength`, fixed seed) with one
"target" lateral-movement episode. Two variants of the event stream are
built from the *same* base events and the *same* episode:

    clean        -- the episode alone, no conditioning.
    conditioned  -- `lmsafe.attack.boiling_frog_scenario` adds a slow,
                     low-volume trickle from the episode's own actor
                     host to its first target host, for
                     `CONDITIONING_DAYS` days immediately before the
                     episode.

Every policy is run on both variants; the quantity of interest is the
*drop* in detection quality on the target episode specifically (not the
aggregate AUC/AP across the whole stream, which is dominated by the
other, unconditioned episodes also present as distractors) between
`clean` and `conditioned`. A policy robust to this attack should show
little to no drop; a policy that blindly incorporates everything it
sees should show a large one.

`GatedPolicy` is included in this comparison deliberately: RQ2 is where
its score-gated admission and multi-host-corroboration mechanisms
either do or do not earn their keep against the other four baselines
run identically. RQ3's script then asks the complementary question (can
Gated also recover RQ1's benign-drift performance at the same time).

Usage
------
    python scripts/run_rq2_poisoning.py [--out-dir results/rq2]
"""
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmsafe import attack, synth
from lmsafe.config import LANL_TRAIN_CUTOFF_S, SynthConfig
from lmsafe.data.windows import iter_windows
from lmsafe.detectors import CountBaselineDetector
from lmsafe.policies import (
    DriftTriggeredPolicy,
    FrozenPolicy,
    GatedPolicy,
    PeriodicPolicy,
    SlidingReplayPolicy,
)
from lmsafe.drift import PageHinkley

N_BENIGN_DAYS = 30
N_DISTRACTOR_EPISODES = 3     # other, unconditioned episodes present as realistic background
DRIFT_STRENGTH = 0.2
DRIFT_START_DAY = 6
CONDITIONING_DAYS = 10
CONDITIONING_EVENTS_PER_DAY = 2.0
SEED = 123
TARGET_BUDGET = 10  # per-window alert budget used to score the target episode specifically


def policy_factory(name: str):
    if name == "frozen":
        return FrozenPolicy()
    if name == "periodic":
        return PeriodicPolicy(period=24)
    if name == "sliding_replay":
        return SlidingReplayPolicy(window_size=168, retrain_period=24)
    if name == "drift_triggered":
        return DriftTriggeredPolicy(PageHinkley(delta=0.01, threshold=8.0), lookback=168)
    if name == "gated":
        return GatedPolicy(
            score_admission_quantile=0.9,
            quarantine_windows=24,
            min_corroborating_src_hosts=3,
            retrain_period=24,
        )
    raise ValueError(name)


POLICY_NAMES = ["frozen", "periodic", "sliding_replay", "drift_triggered", "gated"]


def build_scenario():
    cfg = SynthConfig(
        n_users=40,
        n_hosts=25,
        n_benign_days=N_BENIGN_DAYS,
        base_edges_per_window=25,
        n_episodes=N_DISTRACTOR_EPISODES,
        episode_chain_len=5,
        drift_start_day=DRIFT_START_DAY,
        drift_strength=DRIFT_STRENGTH,
        seed=SEED,
    )
    rng = np.random.default_rng(cfg.seed)
    pop = synth.Population(cfg, rng)
    distractor_episodes = synth.default_episodes(cfg, pop, rng)

    # The RQ2 *target* episode: placed late enough that a full
    # CONDITIONING_DAYS campaign still starts safely after the 41-hour
    # training cutoff, using a fresh actor not reused by any distractor.
    target_start_day = N_BENIGN_DAYS - 4
    target = synth.LMEpisode(
        start_s=int(target_start_day * 86400),
        actor_user="U_target@DOM1",
        actor_host="C_target_actor",
        target_hosts=["C_target_1", "C_target_2", "C_target_3", "C_target_4"],
        hop_seconds=30,
    )
    all_episodes = distractor_episodes + [target]
    base_events = synth.generate(cfg, episodes=all_episodes)

    spec = attack.boiling_frog_scenario(
        target,
        conditioning_days=CONDITIONING_DAYS,
        events_per_day=CONDITIONING_EVENTS_PER_DAY,
        seed=SEED,
    )
    conditioned_events = attack.inject_conditioning(base_events, spec)
    return cfg, base_events, conditioned_events, target


def to_windows(events, cfg):
    windows = list(iter_windows(events, cfg.window_seconds))
    train_w = [w for w in windows if w.window_start < LANL_TRAIN_CUTOFF_S]
    test_w = [w for w in windows if w.window_start >= LANL_TRAIN_CUTOFF_S]
    return train_w, test_w


def target_hop_metrics(test_w, detector, policy, target: synth.LMEpisode) -> dict:
    """Run one (detector, policy) over `test_w` and report how
    suspicious the detector finds *specifically* the conditioned hop --
    `target.actor_host -> target.target_hosts[0]` -- in the window the
    real attack happens.

    This -- not "was the whole episode ever caught" -- is the right
    RQ2 measurement: `boiling_frog_scenario` only conditions the first
    hop's destination. Hops 2+ (`target_hosts[1:]`) are, by
    construction, hosts the actor has never touched before regardless
    of conditioning, so they are almost always individually flagged as
    novel and would make an episode-level "caught anywhere" check
    insensitive to the attack this experiment is actually studying.
    Comparing the conditioned hop's own score/rank between the `clean`
    and `conditioned` variants isolates exactly the effect RQ2 asks
    about.

    `detector`/`policy` must already have had `policy.fit_initial`
    called on them by the caller -- this function only runs the
    test-window loop. Calling `fit_initial` again here, even with an
    empty window list, would silently wipe the real 41-hour benign fit
    (every concrete `Detector.fit(..., warm_start=False)` resets to an
    untrained state first) right before scoring.
    """
    actor_host = target.actor_host
    first_hop_dst = target.target_hosts[0]

    hop_score: Optional[float] = None
    hop_rank_frac: Optional[float] = None
    hop_in_budget: Optional[bool] = None

    for w in test_w:
        s = detector.score(w)
        if w.window_start <= target.start_s < w.window_start + w.window_seconds:
            for idx, (a, b) in enumerate(w.edges):
                if w.nodes[a] == actor_host and w.nodes[b] == first_hop_dst:
                    order = np.argsort(-s)
                    rank = int(np.where(order == idx)[0][0])
                    hop_score = float(s[idx])
                    hop_rank_frac = rank / max(len(s) - 1, 1)
                    hop_in_budget = rank < TARGET_BUDGET
                    break
        detector = policy.observe(w, s, detector)

    return {
        "hop_score": hop_score,
        "hop_rank_frac": hop_rank_frac,
        "hop_in_budget": hop_in_budget,
    }


def main(out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg, base_events, conditioned_events, target = build_scenario()
    print(f"[rq2] target episode at day {target.start_s / 86400:.1f}, "
          f"actor={target.actor_host}, first_target={target.target_hosts[0]}")

    rows = []
    for variant_name, events in (("clean", base_events), ("conditioned", conditioned_events)):
        train_w, test_w = to_windows(events, cfg)
        for pname in POLICY_NAMES:
            detector = CountBaselineDetector()
            policy = policy_factory(pname)
            policy.fit_initial(detector, train_w)
            m = target_hop_metrics(test_w, detector, policy, target)
            rows.append({"variant": variant_name, "policy": pname, **m})
            print(
                f"[rq2] {variant_name:11s} {pname:16s} "
                f"score={m['hop_score']:.4f} rank_frac={m['hop_rank_frac']:.4f} "
                f"in_budget={m['hop_in_budget']}"
            )

    df = pd.DataFrame(rows)
    csv_path = out_dir / "rq2_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"[rq2] wrote {csv_path}")
    make_figure(df, out_dir / "rq2_poisoning.png")


def make_figure(df: pd.DataFrame, path: Path) -> None:
    piv = df.pivot(index="policy", columns="variant", values="hop_rank_frac").reindex(POLICY_NAMES)
    piv = piv.astype(float)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(piv.index))
    width = 0.35
    ax.bar(x - width / 2, piv["clean"], width, label="clean (no conditioning)")
    ax.bar(x + width / 2, piv["conditioned"], width, label="conditioned (boiling-frog)")
    ax.set_xticks(x)
    ax.set_xticklabels(piv.index, rotation=20, ha="right")
    ax.set_ylabel("conditioned hop's anomaly rank (fraction, 0=most suspicious)")
    ax.set_title("RQ2: effect of update poisoning on the conditioned hop's rank")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    print(f"[rq2] wrote {path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out-dir", type=Path, default=Path("results/rq2"))
    args = ap.parse_args()
    main(args.out_dir)


In [ ]:
%%writefile scripts/run_rq3_gated_defense.py
#!/usr/bin/env python3
"""
RQ3 -- Gated defense: does the proposed `GatedPolicy` recover RQ1's
benign-drift adaptation benefit *without* re-opening RQ2's update-
poisoning gap, at the same fixed alert budget, when both pressures are
present at the same time?

Experiment design
-------------------
A 2x2 design, crossing:

    drift_strength in {NONE, STRONG}    -- RQ1's axis
    conditioning   in {off, on}         -- RQ2's axis

on the same benign background and the same target episode (the RQ2
scenario, reused verbatim via `build_rq2_scenario`-equivalent logic),
for every policy. Two outcomes are reported per cell:

    aggregate_ap, aggregate_rec_at_10   -- whole-stream detection
                                            quality (the RQ1 lens: does
                                            this policy keep up with
                                            drift at all, including on
                                            the *distractor* episodes
                                            that have nothing to do
                                            with the conditioning
                                            attack?)
    hop_rank_frac                       -- the conditioned hop's own
                                            anomaly rank (the RQ2 lens:
                                            did conditioning fool this
                                            policy on the attack it was
                                            built for?)

A policy that only ever looked good on RQ1 (periodic/sliding/drift-
triggered) should show a good `aggregate_rec_at_10` under drift but a
high `hop_rank_frac` jump under conditioning. `FrozenPolicy` should show
the opposite trade: immune to conditioning (`hop_rank_frac` never
moves) but the worst `aggregate_rec_at_10` under strong drift. RQ3's
claim is that `GatedPolicy` should land close to the best adaptive
policy on `aggregate_rec_at_10` under drift, while staying close to
Frozen's near-zero `hop_rank_frac` shift under conditioning -- i.e. it
should not have to choose.

Usage
------
    python scripts/run_rq3_gated_defense.py [--out-dir results/rq3]
"""
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmsafe import attack, synth
from lmsafe.config import LANL_TRAIN_CUTOFF_S, SynthConfig
from lmsafe.data.windows import iter_windows
from lmsafe.detectors import CountBaselineDetector
from lmsafe.drift import PageHinkley
from lmsafe.metrics import WindowScores, recall_at_budget, weighted_ap
from lmsafe.policies import (
    DriftTriggeredPolicy,
    FrozenPolicy,
    GatedPolicy,
    PeriodicPolicy,
    SlidingReplayPolicy,
)

N_BENIGN_DAYS = 30
N_DISTRACTOR_EPISODES = 3
DRIFT_START_DAY = 6
DRIFT_LEVELS = {"none": 0.0, "strong": 0.8}
CONDITIONING_DAYS = 10
CONDITIONING_EVENTS_PER_DAY = 2.0
SEED = 777
TARGET_BUDGET = 10


def policy_factory(name: str):
    if name == "frozen":
        return FrozenPolicy()
    if name == "periodic":
        return PeriodicPolicy(period=24)
    if name == "sliding_replay":
        return SlidingReplayPolicy(window_size=168, retrain_period=24)
    if name == "drift_triggered":
        return DriftTriggeredPolicy(PageHinkley(delta=0.01, threshold=8.0), lookback=168)
    if name == "gated":
        return GatedPolicy(
            score_admission_quantile=0.9,
            quarantine_windows=24,
            min_corroborating_src_hosts=3,
            retrain_period=24,
        )
    raise ValueError(name)


POLICY_NAMES = ["frozen", "periodic", "sliding_replay", "drift_triggered", "gated"]


def build_scenario(drift_strength: float):
    cfg = SynthConfig(
        n_users=40,
        n_hosts=25,
        n_benign_days=N_BENIGN_DAYS,
        base_edges_per_window=25,
        n_episodes=N_DISTRACTOR_EPISODES,
        episode_chain_len=5,
        drift_start_day=DRIFT_START_DAY,
        drift_strength=drift_strength,
        seed=SEED,
    )
    rng = np.random.default_rng(cfg.seed)
    pop = synth.Population(cfg, rng)
    distractor_episodes = synth.default_episodes(cfg, pop, rng)

    target_start_day = N_BENIGN_DAYS - 4
    target = synth.LMEpisode(
        start_s=int(target_start_day * 86400),
        actor_user="U_target@DOM1",
        actor_host="C_target_actor",
        target_hosts=["C_target_1", "C_target_2", "C_target_3", "C_target_4"],
        hop_seconds=30,
    )
    all_episodes = distractor_episodes + [target]
    base_events = synth.generate(cfg, episodes=all_episodes)

    spec = attack.boiling_frog_scenario(
        target,
        conditioning_days=CONDITIONING_DAYS,
        events_per_day=CONDITIONING_EVENTS_PER_DAY,
        seed=SEED,
    )
    conditioned_events = attack.inject_conditioning(base_events, spec)
    return cfg, base_events, conditioned_events, target


def to_windows(events, cfg):
    windows = list(iter_windows(events, cfg.window_seconds))
    train_w = [w for w in windows if w.window_start < LANL_TRAIN_CUTOFF_S]
    test_w = [w for w in windows if w.window_start >= LANL_TRAIN_CUTOFF_S]
    return train_w, test_w


def run_cell(train_w, test_w, policy, target: synth.LMEpisode) -> dict:
    """Run one (detector, policy) over one scenario's test stream and
    report both the RQ1-lens aggregate metrics and the RQ2-lens
    conditioned-hop metric in a single pass (one call to
    `detector.score` per window, as `lmsafe.runner` also guarantees)."""
    detector = CountBaselineDetector()
    policy.fit_initial(detector, train_w)

    actor_host, first_hop_dst = target.actor_host, target.target_hosts[0]
    hop_score: Optional[float] = None
    hop_rank_frac: Optional[float] = None

    records = []
    for w in test_w:
        s = detector.score(w)
        records.append(WindowScores.from_window(w, s))
        if w.window_start <= target.start_s < w.window_start + w.window_seconds:
            for idx, (a, b) in enumerate(w.edges):
                if w.nodes[a] == actor_host and w.nodes[b] == first_hop_dst:
                    order = np.argsort(-s)
                    rank = int(np.where(order == idx)[0][0])
                    hop_score = float(s[idx])
                    hop_rank_frac = rank / max(len(s) - 1, 1)
                    break
        detector = policy.observe(w, s, detector)

    return {
        "aggregate_ap": weighted_ap(records),
        "aggregate_rec_at_10": recall_at_budget(records, TARGET_BUDGET),
        "hop_score": hop_score,
        "hop_rank_frac": hop_rank_frac,
    }


def main(out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for drift_label, drift_strength in DRIFT_LEVELS.items():
        cfg, base_events, conditioned_events, target = build_scenario(drift_strength)
        for conditioning_label, events in (("off", base_events), ("on", conditioned_events)):
            train_w, test_w = to_windows(events, cfg)
            for pname in POLICY_NAMES:
                policy = policy_factory(pname)
                m = run_cell(train_w, test_w, policy, target)
                rows.append(
                    {
                        "drift": drift_label,
                        "conditioning": conditioning_label,
                        "policy": pname,
                        **m,
                    }
                )
                print(
                    f"[rq3] drift={drift_label:6s} conditioning={conditioning_label:3s} "
                    f"{pname:16s} AP={m['aggregate_ap']:.4f} Rec@10={m['aggregate_rec_at_10']:.3f} "
                    f"hop_rank_frac={m['hop_rank_frac']:.3f}"
                )

    df = pd.DataFrame(rows)
    csv_path = out_dir / "rq3_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"[rq3] wrote {csv_path}")
    make_figure(df, out_dir / "rq3_gated_defense.png")


def make_figure(df: pd.DataFrame, path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Panel 1 (RQ1 lens): Rec@10 under strong drift, conditioning off.
    sub = df[(df.drift == "strong") & (df.conditioning == "off")].set_index("policy").reindex(POLICY_NAMES)
    axes[0].bar(sub.index, sub["aggregate_rec_at_10"])
    axes[0].set_title("RQ1 lens: Rec@10 under strong drift\n(no conditioning)")
    axes[0].set_ylabel("Rec@10")
    axes[0].tick_params(axis="x", rotation=20)

    # Panel 2 (RQ2 lens): hop rank shift (conditioning on - off), drift=none.
    off = df[(df.drift == "none") & (df.conditioning == "off")].set_index("policy")["hop_rank_frac"]
    on = df[(df.drift == "none") & (df.conditioning == "on")].set_index("policy")["hop_rank_frac"]
    shift = (on - off).reindex(POLICY_NAMES)
    axes[1].bar(shift.index, shift.values, color="darkorange")
    axes[1].set_title("RQ2 lens: conditioned-hop rank shift\n(higher = more fooled)")
    axes[1].set_ylabel("hop_rank_frac(on) - hop_rank_frac(off)")
    axes[1].tick_params(axis="x", rotation=20)

    fig.suptitle("RQ3: can GatedPolicy win on both axes at once?")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    print(f"[rq3] wrote {path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out-dir", type=Path, default=Path("results/rq3"))
    args = ap.parse_args()
    main(args.out_dir)


In [ ]:
%%writefile scripts/pretrain_foundation_model.py
#!/usr/bin/env python3
"""
Pretrain lmsafe.detectors.rw_gnn.GraphFoundationDetector's encoder +
edge-scorer weights on one network's benign windows, and save a
checkpoint that scripts/run_cross_network_transfer.py (or any other
script) can hand to a fresh `GraphFoundationDetector` via
`pretrained_state_dict=` for zero-shot or fine-tuned use on a
*different* network.

This is the "foundation model" step, scoped exactly as described in
lmsafe/detectors/rw_gnn.py's module docstring: a small, CPU-trainable
self-supervised link-prediction pretraining run, not a claim to
reproduce a large published architecture.

Sources
--------
--source synthetic
    Pretrain on lmsafe.synth's benign generator -- fast, no real data
    required, useful for development/dry runs and for the notebook's
    offline verification path.
--source lanl-cache --cache-path <path>
    Pretrain on a `(train_w, test_w)` pickle produced by
    `fetch_lanl_window.py build-cache` or `run_experiment.py`'s own
    `lanl.cache_path`. By default only `train_w` (the clean 41-hour
    training split) is used, specifically so the resulting checkpoint
    stays safe to later evaluate against that same network's *test*
    windows elsewhere without having pretrained on them first. Pass
    --include-test-windows to opt into pretraining on the full stream
    instead (only appropriate when this checkpoint will only ever be
    evaluated on a *different* network, never on this one's own test
    split).

Usage
------
    python scripts/pretrain_foundation_model.py --source synthetic \\
        --out checkpoints/foundation_synthetic.pt

    python scripts/pretrain_foundation_model.py --source lanl-cache \\
        --cache-path lmsafe_cache/lanl_full_windows.pkl \\
        --out checkpoints/foundation_lanl.pt
"""
from __future__ import annotations

import argparse
import pickle
import sys
import time
from pathlib import Path

import torch

from lmsafe import synth
from lmsafe.config import SynthConfig
from lmsafe.data.windows import WindowGraph, iter_windows
from lmsafe.detectors.rw_gnn import GraphFoundationDetector


def _synthetic_windows(n_windows_target: int, seed: int) -> list[WindowGraph]:
    cfg = SynthConfig(
        n_users=200, n_hosts=150, n_benign_days=max(4, n_windows_target // 24 + 1),
        base_edges_per_window=40, n_episodes=0, seed=seed,
    )
    events = synth.generate(cfg, episodes=[])
    windows = list(iter_windows(events, cfg.window_seconds))
    return windows[:n_windows_target] if n_windows_target else windows


def _lanl_cache_windows(cache_path: Path, include_test_windows: bool) -> list[WindowGraph]:
    with open(cache_path, "rb") as fh:
        train_w, test_w = pickle.load(fh)
    windows = list(train_w)
    if include_test_windows:
        windows += list(test_w)
    return windows


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--source", choices=["synthetic", "lanl-cache"], required=True)
    p.add_argument("--cache-path", help="required for --source lanl-cache")
    p.add_argument("--include-test-windows", action="store_true", help="see module docstring -- only safe if this checkpoint will never be evaluated on this same network's own test split")
    p.add_argument("--n-synthetic-windows", type=int, default=300)
    p.add_argument("--hidden-dim", type=int, default=16)
    p.add_argument("--epochs-per-fit", type=int, default=3)
    p.add_argument("--lr", type=float, default=1e-2)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--progress-every", type=int, default=100)
    p.add_argument("--out", required=True)
    a = p.parse_args()

    if a.source == "synthetic":
        windows = _synthetic_windows(a.n_synthetic_windows, a.seed)
    else:
        if not a.cache_path:
            p.error("--cache-path is required for --source lanl-cache")
        windows = _lanl_cache_windows(Path(a.cache_path), a.include_test_windows)

    if not windows:
        print("[pretrain] no windows to pretrain on -- nothing to do", file=sys.stderr)
        sys.exit(1)

    print(f"[pretrain] pretraining GraphFoundationDetector on {len(windows):,} windows "
          f"(source={a.source}, hidden_dim={a.hidden_dim})")

    det = GraphFoundationDetector(
        hidden_dim=a.hidden_dim, epochs_per_fit=a.epochs_per_fit, lr=a.lr, seed=a.seed,
    )

    t0 = time.time()
    # Fit window-by-window (rather than one big `fit(windows)` call) so
    # progress can be reported on a long real-data run -- the detector's
    # own `fit()` already accepts a full sequence, but chunking it here
    # costs nothing and gives visibility matching this project's other
    # long-running scripts (see lmsafe.data.lanl.stream_auth_events's
    # own progress-reporting rationale).
    for i, w in enumerate(windows, start=1):
        det.fit([w], warm_start=True)
        if i % a.progress_every == 0:
            print(f"[pretrain] fit {i:,}/{len(windows):,} windows, "
                  f"{det.state_summary()}, {time.time() - t0:.0f}s elapsed",
                  file=sys.stderr, flush=True)

    out = Path(a.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    torch.save(det.export_state_dict(), out)
    print(f"[pretrain] done in {time.time() - t0:.0f}s -- wrote checkpoint to {out}")
    print(f"[pretrain] final state: {det.state_summary()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_cross_network_transfer.py
#!/usr/bin/env python3
"""
Cross-network transfer: does pretraining
`lmsafe.detectors.rw_gnn.GraphFoundationDetector` on one network's
benign traffic ("network A") help it detect lateral movement on a
*different* network ("network B") whose host population is completely
disjoint, compared to training on network B alone?

This is the experiment the original proposal title's "Cross-Network
Transfer" clause refers to. The default run here uses `lmsafe.synth`
for both networks (fast, offline, reproducible) with network B's host
and user *names* deliberately relabeled into an OpTC-style scheme
("SysClient0001" instead of LANL-style "C1") so the transfer is real,
not an artifact of two synthetic runs happening to share a naming
convention. Pass --network-a-cache-path / --network-b-cache-path to
point at real cached (train_w, test_w) pickles instead (e.g. a LANL
cache for A once an OpTC cache exists for B, or vice versa) -- the
evaluation logic below is dataset-agnostic by construction (same
WindowGraph boundary every reader in lmsafe.data produces).

Three regimes, evaluated on network B's test windows
-------------------------------------------------------
scratch
    A fresh `GraphFoundationDetector`, fit only on B's own training
    split (no exposure to A at all). The "no transfer" baseline.
zero_shot
    Pretrained on A, then applied to B's test stream with **no** fit on
    B's training split whatsoever -- the strongest possible transfer
    claim, if it works at all.
finetune
    Pretrained on A, then fine-tuned (`fit(..., warm_start=True)`) on
    B's own (small) training split before evaluation -- the realistic
    middle ground most deployments would actually use.

To isolate "what did the starting point buy you" from "how adaptively
does each regime behave during the test stream" (a separate, already-
covered question -- see RQ1), every regime is **frozen** after its one
initial fit: during the test stream each window is scored, then folded
into the detector's running host-stats via `observe()` (never a
gradient step -- see that method's docstring), for every regime alike.

Usage
------
    python scripts/run_cross_network_transfer.py [--out-dir results/cross_network_transfer]
"""
from __future__ import annotations

import argparse
import pickle
from pathlib import Path
from typing import Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from lmsafe import synth
from lmsafe.config import LANL_TRAIN_CUTOFF_S, LanlConfig, SynthConfig
from lmsafe.data import lanl, optc
from lmsafe.data.windows import WindowGraph, iter_windows
from lmsafe.detectors.rw_gnn import GraphFoundationDetector
from lmsafe.metrics import (
    WindowScores,
    cluster_episodes,
    episode_hit,
    recall_at_budget,
    weighted_ap,
    weighted_auc,
)

BUDGET = 10


def _relabel(windows: list[WindowGraph], prefix: str) -> list[WindowGraph]:
    """Return a copy of `windows` with every host name replaced by a
    deterministic `{prefix}NNNN`-style name, so network B is provably
    disjoint from network A's naming scheme (never just "a different
    random seed that happens to also look like 'C123'") -- the same
    technique used in tests/test_rw_gnn.py to make the inductive-
    transfer claim mean something concrete."""
    rename: dict[str, str] = {}

    def relabel_one(host: str) -> str:
        if host not in rename:
            rename[host] = f"{prefix}{len(rename):04d}"
        return rename[host]

    out = []
    for w in windows:
        new_nodes = [relabel_one(h) for h in w.nodes]
        out.append(
            WindowGraph(
                window_start=w.window_start,
                window_seconds=w.window_seconds,
                nodes=new_nodes,
                node_index={h: i for i, h in enumerate(new_nodes)},
                edges=list(w.edges),
                edge_weight=list(w.edge_weight),
                edge_is_malicious=list(w.edge_is_malicious),
                n_events=w.n_events,
            )
        )
    return out


def _load_cache_windows(cache_path: Path) -> tuple[list[WindowGraph], list[WindowGraph]]:
    with open(cache_path, "rb") as fh:
        train_w, test_w = pickle.load(fh)
    return list(train_w), list(test_w)


def _build_synthetic_network_a(seed: int) -> list[WindowGraph]:
    """Network A is pretraining material only -- benign traffic, no
    injected episodes, never itself evaluated."""
    cfg = SynthConfig(n_users=150, n_hosts=120, n_benign_days=12, base_edges_per_window=35, n_episodes=0, seed=seed)
    events = synth.generate(cfg, episodes=[])
    windows = list(iter_windows(events, cfg.window_seconds))
    return _relabel(windows, "C")  # LANL-style, explicit for symmetry with B's relabeling


def _build_synthetic_network_b(seed: int) -> tuple[list[WindowGraph], list[WindowGraph]]:
    """Network B is the transfer target: benign traffic plus a handful
    of injected lateral-movement episodes in the test portion, so
    transfer quality is actually measurable (AUC/AP/episode-hit-rate
    against a real, known-positive signal), with an OpTC-shaped naming
    scheme fully disjoint from network A's."""
    cfg = SynthConfig(n_users=150, n_hosts=120, n_benign_days=12, base_edges_per_window=35, n_episodes=6, seed=seed)
    events = synth.generate(cfg)
    # Same 41-hour train/test boundary used throughout this project
    # (LANL_TRAIN_CUTOFF_S) -- "B's own small training split" means the
    # same thing here as everywhere else, not an arbitrarily tinier cut
    # that would make the `scratch` baseline artificially starved.
    lanl_cfg = LanlConfig(train_cutoff_s=LANL_TRAIN_CUTOFF_S, end_s=cfg.n_benign_days * 86400)
    train_events, test_events_iter = lanl.split_train_test(iter(events), lanl_cfg)
    train_w = list(iter_windows(train_events, cfg.window_seconds))
    test_w = list(iter_windows(test_events_iter, cfg.window_seconds))
    return _relabel(train_w, "SysClient"), _relabel(test_w, "SysClient")


def _build_real_optc_network_b(
    paths: list[str], train_cutoff_s: int, window_seconds: int, only_internal_internal: bool
) -> tuple[list[WindowGraph], list[WindowGraph]]:
    """Build network B's train/test windows from real OpTC ecar-bro FLOW
    files via `lmsafe.data.optc.stream_optc_events`. No relabeling here
    -- real OpTC hostnames ("SysClient0351.systemia.com", ...) are
    already a completely different population from network A's LANL-
    style ("C123") names, so the cross-network-transfer claim is real by
    construction, not something this script needs to manufacture.

    Important honesty note: a single `ecar-bro/short` partition file (or
    a few) may contain *zero* internal-to-internal flows at all -- the
    real sample this reader was built against was entirely outbound
    traffic to external IPs (benign beaconing plus one C2-looking
    sequence), which `only_internal_internal=True` correctly drops in
    full. If that happens here, `main()` below reports it plainly rather
    than silently producing an empty, uninterpretable result -- try
    files from `ecar-bro/evaluation` (the actual red-team engagement
    day, across more than one host range) for files likely to contain
    genuine lateral-movement-relevant internal traffic.
    """
    cfg = optc.OptcConfig(
        window_seconds=window_seconds,
        train_cutoff_s=train_cutoff_s,
        only_internal_internal=only_internal_internal,
    )
    events = optc.stream_optc_events(paths, cfg)
    if not events:
        return [], []
    train_events, test_events_iter = lanl.split_train_test(iter(events), cfg)
    train_w = list(iter_windows(train_events, cfg.window_seconds))
    test_w = list(iter_windows(test_events_iter, cfg.window_seconds))
    return train_w, test_w


def _evaluate_frozen(det: GraphFoundationDetector, test_windows: list[WindowGraph]) -> list[WindowScores]:
    """Score every test window with weights frozen at whatever the
    caller already fit; fold each window into the running host-stats
    via `observe()` afterward (never a gradient step) so later windows
    benefit from having "seen" earlier hosts -- see the module
    docstring's "Three regimes" section for why this applies uniformly
    to every regime."""
    records = []
    for w in test_windows:
        if not w.edges:
            continue
        scores = det.score(w)
        records.append(WindowScores.from_window(w, scores))
        det.observe(w)
    return records


def _summarize(name: str, records: list[WindowScores]) -> dict:
    episodes = cluster_episodes(records, gap_windows=1)
    hits = episode_hit(records, BUDGET, episodes)
    return {
        "regime": name,
        "weighted_auc": weighted_auc(records),
        "weighted_ap": weighted_ap(records),
        f"rec_at_{BUDGET}": recall_at_budget(records, BUDGET),
        "episode_hit_rate": (sum(hits) / len(hits)) if hits else float("nan"),
        "n_episodes": len(episodes),
        "n_test_windows": len(records),
    }


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--network-a-cache-path", help="path to a (train_w, test_w) pickle for network A; default: synthetic")
    p.add_argument("--network-b-cache-path", help="path to a (train_w, test_w) pickle for network B; default: synthetic")
    p.add_argument(
        "--network-b-optc-paths", nargs="+",
        help="one or more real OpTC ecar-bro FLOW files (.json or .json.gz) to use as network B instead "
             "of a cache or synthetic data",
    )
    p.add_argument("--network-b-optc-train-cutoff-s", type=int, default=3600,
                    help="train/test boundary in seconds from the first event in the given OpTC files "
                         "(default 1h -- OpTC samples span hours/days, not LANL's months)")
    p.add_argument("--network-b-optc-keep-external", action="store_true",
                    help="keep edges to unresolved (external) IPs as pseudo-hosts instead of dropping them "
                         "-- for inspecting what a file contains, not for a real transfer result")
    p.add_argument("--hidden-dim", type=int, default=16)
    p.add_argument("--epochs-per-fit", type=int, default=5)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out-dir", default="results/cross_network_transfer")
    a = p.parse_args()

    if a.network_a_cache_path:
        windows_a_train, windows_a_test = _load_cache_windows(Path(a.network_a_cache_path))
        windows_a = windows_a_train + windows_a_test
    else:
        windows_a = _build_synthetic_network_a(a.seed)

    if a.network_b_optc_paths:
        b_train, b_test = _build_real_optc_network_b(
            a.network_b_optc_paths, a.network_b_optc_train_cutoff_s, 3600,
            only_internal_internal=not a.network_b_optc_keep_external,
        )
        if not b_train and not b_test:
            print(
                "[cross_network_transfer] the given OpTC file(s) produced ZERO usable "
                "internal-to-internal windows for network B -- this is expected for a file "
                "that only contains outbound traffic to external IPs (e.g. the real "
                "ecar-bro/short sample this reader was built against: benign cert-check "
                "beaconing plus one C2-looking sequence, nothing host-to-host internal). "
                "Try a file from ecar-bro/evaluation instead, or pass "
                "--network-b-optc-keep-external to at least smoke-test the pipeline with "
                "external-IP pseudo-hosts (not a meaningful transfer result)."
            )
            return
    elif a.network_b_cache_path:
        b_train, b_test = _load_cache_windows(Path(a.network_b_cache_path))
    else:
        b_train, b_test = _build_synthetic_network_b(a.seed + 1)

    print(f"[cross_network_transfer] network A: {len(windows_a):,} windows (pretraining only)")
    print(f"[cross_network_transfer] network B: {len(b_train):,} train + {len(b_test):,} test windows")
    if a.network_b_optc_paths:
        print(
            "[cross_network_transfer] NOTE: lmsafe.data.optc does not yet join against any "
            "red-team label file (see its module docstring), so every metric below involving "
            "a malicious label (AUC, AP, Rec@B, episode_hit_rate) will read as NaN / 0 "
            "episodes on real OpTC data -- this run still exercises the full real transfer "
            "mechanism (pretrain on A, apply to real B), just without a ground-truth score yet."
        )

    # -- pretrain once on A ------------------------------------------------
    pretrainer = GraphFoundationDetector(hidden_dim=a.hidden_dim, epochs_per_fit=a.epochs_per_fit, seed=a.seed)
    pretrainer.fit(windows_a, warm_start=False)
    checkpoint = pretrainer.export_state_dict()
    print(f"[cross_network_transfer] pretrained on A: {pretrainer.state_summary()}")

    rows = []

    # -- scratch: no exposure to A at all -----------------------------------
    scratch = GraphFoundationDetector(hidden_dim=a.hidden_dim, epochs_per_fit=a.epochs_per_fit, seed=a.seed)
    scratch.fit(b_train, warm_start=False)
    rows.append(_summarize("scratch", _evaluate_frozen(scratch, b_test)))

    # -- zero_shot: pretrained on A, no fit on B's train split at all -------
    zero_shot = GraphFoundationDetector(hidden_dim=a.hidden_dim, epochs_per_fit=a.epochs_per_fit, seed=a.seed, pretrained_state_dict=checkpoint)
    rows.append(_summarize("zero_shot", _evaluate_frozen(zero_shot, b_test)))

    # -- finetune: pretrained on A, then fine-tuned on B's train split ------
    finetune = GraphFoundationDetector(hidden_dim=a.hidden_dim, epochs_per_fit=a.epochs_per_fit, seed=a.seed, pretrained_state_dict=checkpoint)
    finetune.fit(b_train, warm_start=True)
    rows.append(_summarize("finetune", _evaluate_frozen(finetune, b_test)))

    df = pd.DataFrame(rows)
    out_dir = Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "cross_network_transfer_results.csv"
    df.to_csv(csv_path, index=False)
    print(df.to_string(index=False))
    print(f"[cross_network_transfer] wrote {csv_path}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, metric in zip(axes, ["weighted_auc", "weighted_ap", "episode_hit_rate"]):
        ax.bar(df["regime"], df[metric])
        ax.set_title(metric)
        ax.set_ylim(bottom=0)
    fig.suptitle("Cross-network transfer: network A (pretrain) -> network B (evaluate)")
    fig.tight_layout()
    png_path = out_dir / "cross_network_transfer.png"
    fig.savefig(png_path, dpi=120)
    print(f"[cross_network_transfer] wrote {png_path}")


if __name__ == "__main__":
    main()


## 5) Write the config files (`configs/`)

In [ ]:
%%writefile configs/synthetic_quick.yaml
# Quick synthetic smoke test -- runs in seconds, no network access.
# Use this to sanity-check the whole pipeline (or a code change to it)
# before pointing scripts/run_experiment.py at the real LANL data.
source: synthetic
scenario_label: synthetic_quick

detector:
  name: count_baseline
  kwargs:
    alpha: 0.5

policies:
  - name: frozen
    kwargs: {}
  - name: periodic
    kwargs:
      period: 24
  - name: sliding_replay
    kwargs:
      window_size: 168
      retrain_period: 24
  - name: drift_triggered
    kwargs:
      drift_detector_name: page_hinkley
      drift_detector_kwargs:
        delta: 0.01
        threshold: 8.0
      lookback: 168
  - name: gated
    kwargs:
      score_admission_quantile: 0.9
      quarantine_windows: 24
      min_corroborating_src_hosts: 3
      retrain_period: 24

budgets: [1, 5, 10, 20]

output:
  dir: results/synthetic_quick
  csv: results.csv

synth:
  n_users: 30
  n_hosts: 20
  n_benign_days: 15
  base_edges_per_window: 20
  n_episodes: 4
  episode_chain_len: 4
  drift_start_day: 6
  drift_strength: 0.3
  seed: 1


In [ ]:
%%writefile configs/lanl_dev.yaml
# Real LANL data, development cutoff (30 days).
#
# This project's own inspection of the public redteam.txt.gz (done
# 2026-11-13) found all 749 labeled red-team events before day 29.6 and
# none after day 30 -- so a 30-day cut already contains every known
# attack while reading roughly half of the 7.2 GB auth stream. Use
# lanl_full.yaml for the final paper numbers, which read the full
# 58-day release to match the LMDEval fair-evaluation protocol exactly.
#
# Before running: export LMSAFE_LANL_BASE_URL='<signed link from
# https://csr.lanl.gov/data/cyber1/>' -- see lmsafe/data/lanl_stream.py.
# The link is read at run time and is never written to this file.
source: lanl
scenario_label: lanl_dev_30d

detector:
  name: count_baseline
  kwargs:
    alpha: 0.5

policies:
  - name: frozen
    kwargs: {}
  - name: periodic
    kwargs:
      period: 24
  - name: sliding_replay
    kwargs:
      window_size: 168
      retrain_period: 24
  - name: drift_triggered
    kwargs:
      drift_detector_name: page_hinkley
      drift_detector_kwargs:
        delta: 0.01
        threshold: 8.0
      lookback: 168
  - name: gated
    kwargs:
      score_admission_quantile: 0.9
      quarantine_windows: 24
      min_corroborating_src_hosts: 3
      retrain_period: 24

budgets: [1, 5, 10, 20, 50]

output:
  dir: results/lanl_dev
  csv: results.csv

lanl:
  end_s: 2592000          # LANL_DEV_CUTOFF_S, 30 days
  train_cutoff_s: 147600  # LANL_TRAIN_CUTOFF_S, 41 hours
  only_remote_logon: true
  filter_ntlm_only: false
  cache_path: lmsafe_cache/lanl_dev_windows.pkl
  checkpoint_every: 24     # partial checkpoint every 24 test windows (~1 sim. day)


In [ ]:
%%writefile configs/lanl_full.yaml
# Real LANL data, full 58-day release -- for final paper numbers,
# directly comparable to the LMDEval fair-evaluation protocol this
# project's preprocessing follows. Reads the complete 7.2 GB auth
# stream (still never downloaded to disk -- see
# lmsafe/data/lanl_stream.py); budget plenty of wall-clock time for
# this on Colab, and set `cache_path` so you only pay that cost once.
#
# Before running: export LMSAFE_LANL_BASE_URL='<signed link from
# https://csr.lanl.gov/data/cyber1/>'.
source: lanl
scenario_label: lanl_full_58d

detector:
  name: count_baseline
  kwargs:
    alpha: 0.5

policies:
  - name: frozen
    kwargs: {}
  - name: periodic
    kwargs:
      period: 24
  - name: sliding_replay
    kwargs:
      window_size: 168
      retrain_period: 24
  - name: drift_triggered
    kwargs:
      drift_detector_name: page_hinkley
      drift_detector_kwargs:
        delta: 0.01
        threshold: 8.0
      lookback: 168
  - name: gated
    kwargs:
      score_admission_quantile: 0.9
      quarantine_windows: 24
      min_corroborating_src_hosts: 3
      retrain_period: 24

budgets: [1, 5, 10, 20, 50]

output:
  dir: results/lanl_full
  csv: results.csv

lanl:
  end_s: 5011200          # LANL_FULL_CUTOFF_S, 58 days
  train_cutoff_s: 147600  # LANL_TRAIN_CUTOFF_S, 41 hours
  only_remote_logon: true
  filter_ntlm_only: false
  cache_path: lmsafe_cache/lanl_full_windows.pkl
  checkpoint_every: 24     # partial checkpoint every 24 test windows (~1 sim. day) --
                           # this is the run most likely to be interrupted
                           # (hours of wall time), so a crash-safe cache
                           # matters most here


## 6) Write the tests (`tests/`) and run them

132 automated tests covering: the `Event`/`WindowGraph` schema, the LANL
fairness protocol (checked against the real `redteam.txt.gz` file -- 749
rows -- *and* against real/documented raw `auth.txt.gz` sample lines
parsed end-to-end, which is what caught a real shipped bug: `logon_type`
vs `orientation` were being compared against the wrong field, silently
rejecting 100% of real events), the OpTC FLOW reader (checked against a
real `ecar-bro/short` sample line verbatim, the full 241-line real sample
file bundled at `tests/fixtures/optc_ecarbro_short_sample.json` end to
end, plus the internal-to-internal host-resolution path that real sample
didn't happen to exercise -- see below), the synthetic generator, all
three detectors (including the new
`GraphFoundationDetector` -- see below), both drift detectors, all five
update policies, the attack module, every metrics function, the runner,
and a static check that no detector or policy ever reads the ground-truth
label.

A decode-free raw-bytes prefilter in `stream_auth_events` now skips
utf-8 decoding and `Event` construction entirely for a raw line the
fair protocol would discard anyway (same-host / non-"LogOn" events --
the majority of real rows). Two tests prove this fast path never
disagrees with the slow, authoritative path, including on deliberately
invalid-UTF-8 hostname bytes. A network-free synthetic microbenchmark
at the real ~37% keep ratio measured roughly a 2x speedup (~51% less
CPU time) from this change alone -- real wall-clock speedup on an
actual LANL run will be somewhat smaller to the extent the run is
network/decompression-bound rather than CPU-bound.

Two test files guard a second real bug found in the field: both
`fetch_lanl_window.py build-cache` *and* `run_experiment.py`'s own
`cache_path` streaming used to pickle `(train_w, test_w)` only once, at
the very end of a multi-hour real-LANL run -- so a run interrupted by
Ctrl+C, a Colab disconnect, or a network drop (which is exactly what
happened on a real 58-day run that reached ~87% progress) left *nothing*
on disk. Both paths now write a crash-safe, atomic partial checkpoint
every `checkpoint_every` test windows (default 24, about once per
simulated day), so an interrupted run still leaves a near-complete,
directly-usable cache file instead of losing hours of streaming.

That checkpoint mechanism has a sharp edge of its own, also found in
the field on a real run (six network reconnects, stopped by hand at day
40.64 of 58): the pickle format can't tell a *finished* run apart from
an *interrupted* one, so simply reusing `cache_path` on a later run
silently treats a partial checkpoint as if it were the complete result.
`build_lanl_windows` now prints how many days the loaded cache actually
covers against how many the config asked for, with a loud `WARNING`
when they disagree -- two tests (`test_loading_a_partial_checkpoint_
prints_a_short_coverage_warning`, `test_loading_a_complete_cache_
prints_no_warning`) pin this down against both a short and a complete
cache.

Running this here in Colab before any real experiment confirms the
runtime environment is sound.

In [ ]:
%%writefile tests/conftest.py
"""Shared pytest fixtures."""
from pathlib import Path

import pytest

FIXTURES_DIR = Path(__file__).parent / "fixtures"


@pytest.fixture
def redteam_fixture_path() -> Path:
    """The real LANL redteam.txt.gz (749 rows), used so tests exercise
    the actual public label file's shape, not just a hand-built stand-in."""
    path = FIXTURES_DIR / "redteam.txt.gz"
    assert path.exists(), f"missing test fixture: {path}"
    return path


In [ ]:
%%writefile tests/_doubles.py
"""Tiny test doubles shared across policy/runner tests."""
from __future__ import annotations

import numpy as np

from lmsafe.detectors.base import Detector


class RecordingDetector(Detector):
    """A `Detector` that does no real scoring -- it just records every
    `fit` call (how many windows, whether warm_start) so a policy's
    update *schedule* can be tested precisely, independent of any real
    detector's own statistical behavior.

    `score` returns a fixed, caller-supplied array (or zeros) so tests
    can also control exactly what a policy's score-gated logic sees.
    """

    def __init__(self, score_fn=None):
        self.fit_calls: list[tuple[int, bool]] = []  # (n_windows, warm_start)
        self._score_fn = score_fn

    def fit(self, windows, *, warm_start: bool = True) -> None:
        self.fit_calls.append((len(list(windows)), warm_start))

    def score(self, window) -> np.ndarray:
        if self._score_fn is not None:
            return self._score_fn(window)
        return np.zeros(len(window.edges), dtype=np.float64)

    def clone_fresh(self) -> "RecordingDetector":
        return RecordingDetector(self._score_fn)


In [ ]:
%%writefile tests/test_schema.py
from lmsafe.data.schema import Event, RedTeamRecord, event_from_csv_row, event_to_csv_row


def test_event_edge_and_remote_logon():
    e = Event(t=1, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2", orientation="LogOn")
    assert e.edge == ("C1", "C2")
    assert e.is_remote_logon()


def test_same_host_logon_is_not_remote():
    e = Event(t=1, src_user="a", dst_user="a", src_comp="C1", dst_comp="C1", orientation="LogOn")
    assert not e.is_remote_logon()


def test_logoff_is_not_remote_logon():
    e = Event(t=1, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2", orientation="LogOff")
    assert not e.is_remote_logon()


def test_logon_type_alone_does_not_make_it_a_remote_logon():
    # Regression guard for the exact bug this project shipped: LANL's
    # `logon_type` column holds the Windows auth *mechanism*
    # ("Network", "Batch", "Interactive", ...) and is never literally
    # "LogOn" -- that marker lives in `orientation`. An Event with
    # logon_type="LogOn" but no matching orientation must NOT be
    # treated as a remote logon, and real-shaped logon_type values
    # (e.g. "Network") with the correct orientation must still pass.
    wrong_field = Event(
        t=1, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2",
        logon_type="LogOn", orientation="",
    )
    assert not wrong_field.is_remote_logon()

    real_shaped = Event(
        t=1, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2",
        logon_type="Network", orientation="LogOn",
    )
    assert real_shaped.is_remote_logon()


def test_csv_roundtrip():
    e = Event(
        t=42,
        src_user="U1@DOM1",
        dst_user="U2@DOM1",
        src_comp="C1",
        dst_comp="C2",
        auth_type="Kerberos",
        logon_type="LogOn",
        orientation="LogOn",
        success=True,
        is_malicious=True,
        source="synthetic",
    )
    row = event_to_csv_row(e)
    e2 = event_from_csv_row({k: str(v) for k, v in row.items()})
    assert e2 == e


def test_redteam_record_key_uses_dst_user_not_src_user():
    # Regression guard for the exact join-key bug this project's own
    # literature review flagged as easy to get backwards (see
    # RedTeamRecord.key docstring): the key must be built from the
    # record's own fields in (t, user, src_comp, dst_comp) order, with
    # "user" matched against an Event's dst_user elsewhere, not here.
    r = RedTeamRecord(t=100, user="U1@DOM1", src_comp="C1", dst_comp="C2")
    assert r.key() == (100, "U1@DOM1", "C1", "C2")


In [ ]:
%%writefile tests/test_lanl_labels.py
"""Tests for the fair LANL extraction protocol (lmsafe.data.lanl),
exercised entirely offline: no network access, using the real
redteam.txt.gz fixture for the label-loading test and hand-built
Events for the filtering/labeling logic."""
import io

from lmsafe.config import LanlConfig
from lmsafe.data import lanl
from lmsafe.data.schema import Event, RedTeamRecord


def make_cfg(**overrides) -> LanlConfig:
    return LanlConfig(**overrides)


def test_load_real_redteam_fixture(redteam_fixture_path):
    records = lanl.load_redteam_records(local_path=redteam_fixture_path)
    assert len(records) == 749
    assert records[0] == RedTeamRecord(t=150_885, user="U620@DOM1", src_comp="C17693", dst_comp="C1003")
    assert records[-1] == RedTeamRecord(t=2_557_047, user="U737@DOM1", src_comp="C19932", dst_comp="C108")
    # The project's own inspection (2026-11-13): no event before 41h.
    assert min(r.t for r in records) >= 41 * 3600


def test_remote_logon_positive_match_is_labeled_malicious():
    cfg = make_cfg()
    keys = {RedTeamRecord(t=100, user="U1@DOM1", src_comp="C1", dst_comp="C2").key()}
    ev = Event(t=100, src_user="X", dst_user="U1@DOM1", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOn")
    out = lanl.apply_fair_protocol(ev, keys, cfg)
    assert out is not None
    assert out.is_malicious


def test_same_host_logon_is_dropped_even_if_it_would_match():
    cfg = make_cfg()
    # Same (t, user, src, dst) as a red-team key, but src==dst -- the
    # fair protocol's remote-LogOn filter must drop it before the
    # label join ever runs.
    keys = {RedTeamRecord(t=100, user="U1@DOM1", src_comp="C3", dst_comp="C3").key()}
    ev = Event(t=100, src_user="X", dst_user="U1@DOM1", src_comp="C3", dst_comp="C3", logon_type="Network", orientation="LogOn")
    assert lanl.apply_fair_protocol(ev, keys, cfg) is None


def test_logoff_event_is_dropped():
    cfg = make_cfg()
    ev = Event(t=1, src_user="a", dst_user="b", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOff")
    assert lanl.apply_fair_protocol(ev, set(), cfg) is None


def test_benign_remote_logon_not_in_redteam_is_kept_unlabeled():
    cfg = make_cfg()
    ev = Event(t=5, src_user="a", dst_user="b", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOn")
    out = lanl.apply_fair_protocol(ev, set(), cfg)
    assert out is not None
    assert not out.is_malicious


def test_ntlm_only_filter_is_off_by_default():
    cfg = make_cfg()
    assert cfg.filter_ntlm_only is False
    ev = Event(t=1, src_user="a", dst_user="b", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOn", auth_type="Kerberos")
    assert lanl.apply_fair_protocol(ev, set(), cfg) is not None


def test_ntlm_only_filter_can_be_opted_into_for_non_fair_comparison():
    cfg = make_cfg(filter_ntlm_only=True)
    kerberos_ev = Event(t=1, src_user="a", dst_user="b", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOn", auth_type="Kerberos")
    ntlm_ev = Event(t=1, src_user="a", dst_user="b", src_comp="C1", dst_comp="C2", logon_type="Network", orientation="LogOn", auth_type="NTLM")
    assert lanl.apply_fair_protocol(kerberos_ev, set(), cfg) is None
    assert lanl.apply_fair_protocol(ntlm_ev, set(), cfg) is not None


def test_real_lanl_sample_line_same_host_is_dropped():
    # The exact example line from LANL's own published auth.txt.gz
    # schema documentation. Columns: t, src_user, dst_user, src_comp,
    # dst_comp, auth_type, logon_type, orientation, success. src_comp
    # == dst_comp here, so it must be dropped despite orientation
    # being "LogOn".
    raw = b"1,C625$@DOM1,U147@DOM1,C625,C625,Negotiate,Batch,LogOn,Success"
    ev = lanl._auth_fields_to_event(raw.split(b","))
    assert ev is not None
    assert ev.logon_type == "Batch"
    assert ev.orientation == "LogOn"
    assert not ev.is_remote_logon()
    assert lanl.apply_fair_protocol(ev, set(), make_cfg()) is None


def test_real_lanl_peek_line_remote_logon_is_kept():
    # A real line from the user's own auth.txt.gz peek output, with
    # logon_type="Network" (never "LogOn") and orientation="LogOn",
    # src_comp != dst_comp. This is the exact shape of row that the
    # logon_type-vs-orientation mix-up bug rejected 100% of the time
    # across both real LANL runs (kept=0 over 1.5B+ lines) -- this test
    # fails under the old `self.logon_type == "LogOn"` check and must
    # keep passing under the fixed `self.orientation == "LogOn"` check.
    raw = b"1,ANONYMOUS LOGON@C586,ANONYMOUS LOGON@C586,C1250,C586,NTLM,Network,LogOn,Success"
    ev = lanl._auth_fields_to_event(raw.split(b","))
    assert ev is not None
    assert ev.logon_type == "Network"
    assert ev.orientation == "LogOn"
    assert ev.is_remote_logon()
    out = lanl.apply_fair_protocol(ev, set(), make_cfg())
    assert out is not None
    assert not out.is_malicious


def _ev(t):
    return Event(t=t, src_user="a", dst_user="a", src_comp="C1", dst_comp="C1", source="lanl")


def test_split_train_test_boundary_is_strict_less_than():
    cfg = make_cfg(train_cutoff_s=1000)
    events = [_ev(0), _ev(999), _ev(1000), _ev(1001)]
    train, test_iter = lanl.split_train_test(iter(events), cfg)
    test = list(test_iter)
    assert [e.t for e in train] == [0, 999]
    assert [e.t for e in test] == [1000, 1001]


def test_split_train_test_all_train():
    cfg = make_cfg(train_cutoff_s=1000)
    events = [_ev(0), _ev(1)]
    train, test_iter = lanl.split_train_test(iter(events), cfg)
    assert [e.t for e in train] == [0, 1]
    assert list(test_iter) == []


def test_split_train_test_all_test():
    cfg = make_cfg(train_cutoff_s=1000)
    events = [_ev(1000), _ev(2000)]
    train, test_iter = lanl.split_train_test(iter(events), cfg)
    assert train == []
    assert [e.t for e in test_iter] == [1000, 2000]


def test_split_train_test_empty_input():
    cfg = make_cfg(train_cutoff_s=1000)
    train, test_iter = lanl.split_train_test(iter([]), cfg)
    assert train == []
    assert list(test_iter) == []


def _raw_auth_line(t: int, *, remote: bool) -> bytes:
    """One synthetic raw auth.txt.gz line. `remote=False` builds a
    same-host LogOff line, the kind `apply_fair_protocol` always drops
    -- used below to simulate a stream dominated by filtered-out rows,
    like the real file is."""
    if remote:
        return f"{t},a@DOM1,b@DOM1,C1,C2,Kerberos,Network,LogOn,Success".encode()
    return f"{t},a@DOM1,a@DOM1,C1,C1,?,Network,LogOff,Success".encode()


def test_stream_auth_events_progress_fires_on_raw_lines_not_kept_lines(monkeypatch):
    """Regression test for a real bug found while running this against
    the live LANL stream: the progress-print check used to sit after
    several `continue`s that skip most raw lines (same-host / non-LogOn
    events, which are the majority of real auth.txt.gz traffic). When
    the raw line count crossed a `progress_every` boundary on exactly
    one of those filtered lines, that round's print was silently
    skipped rather than merely delayed -- over a real multi-hour stream
    this produced long, unpredictable stretches with no progress output
    at all even though the stream was reading fine, indistinguishable
    from a hang.

    This test builds a stream that is ~90% filtered-out same-host
    LogOffs by design, specifically so that several `progress_every`
    boundaries land on a filtered line, and checks the print still
    fires exactly once per `progress_every` raw lines regardless.
    """
    progress_every = 10
    total_raw_lines = 55  # crosses the boundary 5 times (10,20,...,50)

    lines = []
    for i in range(total_raw_lines):
        t = i + 1
        # Every 10th raw line (indices 9, 19, 29, ...) is exactly the
        # one that will cross a progress_every boundary (counts are
        # 1-indexed inside stream_auth_events) -- make that specific
        # line a filtered-out same-host LogOff, the worst case for the
        # old, buggy placement of the progress check.
        remote = (i + 1) % progress_every != 0
        lines.append(_raw_auth_line(t, remote=remote))

    def fake_iter_lines(url):
        return iter(lines)

    monkeypatch.setattr(lanl.lanl_stream, "iter_lines", fake_iter_lines)

    cfg = make_cfg(end_s=10_000, train_cutoff_s=0)
    stderr = io.StringIO()
    events = list(
        lanl.stream_auth_events(
            cfg,
            base_url="https://example.invalid/cyber1/",
            redteam=[],
            progress_every=progress_every,
            progress_stream=stderr,
        )
    )

    # Every 10th line was deliberately a same-host LogOff -> dropped by
    # apply_fair_protocol, so kept count is less than raw count, which
    # is exactly the condition that used to hide the progress print.
    assert len(events) < total_raw_lines

    progress_lines = [ln for ln in stderr.getvalue().splitlines() if ln.startswith("[lanl] read")]
    assert len(progress_lines) == total_raw_lines // progress_every
    assert "read 10 lines" in progress_lines[0]
    assert "read 50 lines" in progress_lines[-1]


def _slow_path_keeps(raw: list, cfg: LanlConfig) -> bool:
    """The authoritative, decode-first answer: build the real `Event`
    and ask `apply_fair_protocol` whether it survives (ignoring
    red-team labeling, which is orthogonal to the fair-protocol filter
    this is checking)."""
    parsed = lanl._auth_fields_to_event(raw)
    assert parsed is not None
    return lanl.apply_fair_protocol(parsed, set(), cfg) is not None


def test_raw_prefilter_agrees_with_apply_fair_protocol_on_real_shaped_lines():
    # `stream_auth_events`'s performance optimization: skip utf-8 decode
    # + Event construction entirely for a line the fair protocol would
    # discard anyway. This test is the proof-by-example that the fast,
    # decode-free `_raw_line_passes_fair_prefilter` never disagrees with
    # the slow, authoritative, fully-decoded `apply_fair_protocol` path --
    # across remote/same-host, LogOn/LogOff, and NTLM/Kerberos
    # combinations, with both filters on and off.
    cases = [
        # (src_comp, dst_comp, auth_type, orientation)
        (b"C1", b"C2", b"NTLM", b"LogOn"),      # remote LogOn -> kept
        (b"C1", b"C1", b"NTLM", b"LogOn"),      # same-host -> dropped
        (b"C1", b"C2", b"NTLM", b"LogOff"),     # LogOff -> dropped
        (b"C1", b"C2", b"Kerberos", b"LogOn"),  # remote, non-NTLM
        (b"C1", b"C1", b"Kerberos", b"LogOff"), # same-host AND LogOff
    ]
    for src, dst, auth_t, orient in cases:
        raw = [b"100", b"u@D", b"u@D", src, dst, auth_t, b"Network", orient, b"Success"]
        for cfg in (
            make_cfg(),  # only_remote_logon=True (default), filter_ntlm_only=False
            make_cfg(filter_ntlm_only=True),
            make_cfg(only_remote_logon=False),
            make_cfg(only_remote_logon=False, filter_ntlm_only=True),
        ):
            fast = lanl._raw_line_passes_fair_prefilter(raw, cfg)
            slow = _slow_path_keeps(raw, cfg)
            assert fast == slow, (
                f"fast/slow path disagreement for src={src} dst={dst} "
                f"auth_t={auth_t} orient={orient} cfg={cfg}: fast={fast} slow={slow}"
            )


def test_raw_prefilter_never_falsely_rejects_on_invalid_utf8_hostnames():
    # The one theoretical edge case the prefilter's docstring calls out:
    # two *different*, both-invalid-utf8 byte sequences could in
    # principle decode (errors="replace") to the *same* string, making
    # a byte-level "different hosts" check a safe over-approximation of
    # the real, decoded-string check -- never a false reject. Exercise
    # that directly with genuinely invalid UTF-8 bytes in the host
    # fields: the fast path must never say "drop" when the slow path
    # would keep it.
    weird_hosts = [b"\xff", b"\xfe", b"C1", b"\xff\xfe"]
    for src in weird_hosts:
        for dst in weird_hosts:
            raw = [b"100", b"u@D", b"u@D", src, dst, b"NTLM", b"Network", b"LogOn", b"Success"]
            cfg = make_cfg()
            fast = lanl._raw_line_passes_fair_prefilter(raw, cfg)
            slow = _slow_path_keeps(raw, cfg)
            if slow:
                assert fast, (
                    f"fast path falsely rejected a line the slow path would keep: "
                    f"src={src!r} dst={dst!r}"
                )


In [ ]:
%%writefile tests/test_windows.py
from lmsafe.data.schema import Event
from lmsafe.data.windows import filter_edges, iter_windows


def _ev(t, src, dst, mal=False):
    return Event(t=t, src_user="u", dst_user="u", src_comp=src, dst_comp=dst, is_malicious=mal, source="x")


def test_iter_windows_groups_and_dedupes_edges():
    events = [
        _ev(0, "C1", "C2"),
        _ev(10, "C1", "C2"),
        _ev(20, "C2", "C3", mal=True),
        _ev(3700, "C3", "C4"),  # next hour
    ]
    windows = list(iter_windows(events, window_seconds=3600))
    assert len(windows) == 2

    w0 = windows[0]
    assert w0.window_start == 0
    assert w0.nodes == ["C1", "C2", "C3"]
    assert w0.edges == [(0, 1), (1, 2)]
    assert w0.edge_weight == [2, 1]
    assert w0.edge_is_malicious == [False, True]
    assert w0.n_events == 3

    w1 = windows[1]
    assert w1.window_start == 3600
    assert w1.nodes == ["C3", "C4"]


def test_empty_windows_are_skipped_not_yielded_empty():
    events = [_ev(0, "C1", "C2"), _ev(7300, "C1", "C2")]  # two windows apart, middle window empty
    windows = list(iter_windows(events, window_seconds=3600))
    assert [w.window_start for w in windows] == [0, 7200]


def test_iter_windows_on_empty_stream():
    assert list(iter_windows([], window_seconds=3600)) == []


def test_align_start_shifts_window_boundaries():
    events = [_ev(50, "C1", "C2"), _ev(150, "C1", "C2")]
    windows = list(iter_windows(events, window_seconds=100, align_start=25))
    # boundaries at 25, 125, 225, ... -> t=50 in [25,125), t=150 in [125,225)
    assert [w.window_start for w in windows] == [25, 125]


def test_filter_edges_keeps_only_selected_edges_and_remaps_nodes():
    events = [_ev(0, "C1", "C2"), _ev(0, "C3", "C4", mal=True)]
    w = next(iter(iter_windows(events, window_seconds=3600)))
    assert w.nodes == ["C1", "C2", "C3", "C4"]
    sub = filter_edges(w, [1])  # keep only the C3->C4 edge
    assert sub.nodes == ["C3", "C4"]
    assert sub.edges == [(0, 1)]
    assert sub.edge_is_malicious == [True]
    assert sub.n_events == 1
    assert sub.window_start == w.window_start


def test_filter_edges_empty_selection():
    events = [_ev(0, "C1", "C2")]
    w = next(iter(iter_windows(events, window_seconds=3600)))
    sub = filter_edges(w, [])
    assert sub.nodes == []
    assert sub.edges == []
    assert sub.n_events == 0


In [ ]:
%%writefile tests/test_synth.py
from lmsafe.config import LANL_TRAIN_CUTOFF_S, SynthConfig
from lmsafe import synth


def small_cfg(**overrides) -> SynthConfig:
    base = dict(
        n_users=15,
        n_hosts=12,
        n_benign_days=6,
        base_edges_per_window=10,
        n_episodes=3,
        episode_chain_len=4,
        seed=1,
    )
    base.update(overrides)
    return SynthConfig(**base)


def test_generate_is_sorted_and_deterministic_for_fixed_seed():
    cfg = small_cfg()
    a = synth.generate(cfg)
    b = synth.generate(cfg)
    assert [e.t for e in a] == sorted(e.t for e in a)
    assert a == b


def test_different_seeds_change_the_stream():
    a = synth.generate(small_cfg(seed=1))
    b = synth.generate(small_cfg(seed=2))
    assert a != b


def test_malicious_event_count_matches_episode_spec():
    cfg = small_cfg(n_episodes=3, episode_chain_len=4)
    events = synth.generate(cfg)
    malicious = [e for e in events if e.is_malicious]
    assert len(malicious) == cfg.n_episodes * cfg.episode_chain_len


def test_malicious_events_have_clean_plain_str_fields():
    # Regression guard: numpy's Generator.choice returns numpy.str_
    # scalars, which must be cast back to plain str (see synth.py
    # comments) so downstream dict keys / CSV writers behave normally.
    events = synth.generate(small_cfg())
    malicious = [e for e in events if e.is_malicious]
    assert malicious, "expected at least one injected episode"
    for e in malicious:
        assert type(e.src_user) is str
        assert type(e.src_comp) is str
        assert type(e.dst_comp) is str


def test_episodes_each_have_one_actor_host_and_distinct_targets():
    cfg = small_cfg(n_episodes=2, episode_chain_len=5)
    events = synth.generate(cfg)
    malicious = [e for e in events if e.is_malicious]
    by_actor = {}
    for e in malicious:
        by_actor.setdefault(e.src_comp, []).append(e.dst_comp)
    assert len(by_actor) == cfg.n_episodes
    for targets in by_actor.values():
        assert len(targets) == cfg.episode_chain_len
        assert len(set(targets)) == cfg.episode_chain_len  # distinct hops


def test_default_episodes_stay_clear_of_training_cutoff():
    # Regression guard for the min_start_s fix: with a short benign
    # period and several episodes, naive evenly-spaced placement used
    # to put the first episode inside the 41-hour training window,
    # silently removing it from the test population.
    cfg = small_cfg(n_benign_days=15, n_episodes=4, episode_chain_len=4)
    events = synth.generate(cfg)
    malicious_times = [e.t for e in events if e.is_malicious]
    assert min(malicious_times) >= LANL_TRAIN_CUTOFF_S


def test_zero_episodes_gives_purely_benign_stream():
    cfg = small_cfg(n_episodes=0)
    events = synth.generate(cfg, episodes=[])
    assert all(not e.is_malicious for e in events)


def test_no_drift_keeps_host_population_fixed():
    cfg = small_cfg(n_episodes=0, drift_strength=0.0, drift_start_day=None)
    events = synth.generate(cfg, episodes=[])
    hosts = {e.src_comp for e in events} | {e.dst_comp for e in events}
    # population should not exceed the configured size (common servers
    # + desktops); allow equality only.
    assert len(hosts) <= cfg.n_hosts


def test_drift_grows_the_host_population():
    no_drift = small_cfg(n_benign_days=20, n_episodes=0, drift_strength=0.0, drift_start_day=None)
    with_drift = small_cfg(n_benign_days=20, n_episodes=0, drift_strength=1.0, drift_start_day=2)
    hosts_a = {e.src_comp for e in synth.generate(no_drift, episodes=[])} | {
        e.dst_comp for e in synth.generate(no_drift, episodes=[])
    }
    hosts_b = {e.src_comp for e in synth.generate(with_drift, episodes=[])} | {
        e.dst_comp for e in synth.generate(with_drift, episodes=[])
    }
    assert len(hosts_b) > len(hosts_a)


In [ ]:
%%writefile tests/test_detectors.py
import numpy as np
import pytest

from lmsafe.config import SynthConfig
from lmsafe import synth
from lmsafe.data.windows import iter_windows
from lmsafe.detectors import CountBaselineDetector, FeatureMLPDetector, GraphFoundationDetector

DETECTOR_FACTORIES = [
    lambda: CountBaselineDetector(alpha=0.5),
    lambda: FeatureMLPDetector(epochs_per_fit=2, hidden_dim=8, seed=0),
    lambda: GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=0),
]


def _windows():
    cfg = SynthConfig(
        n_users=15, n_hosts=12, n_benign_days=4, base_edges_per_window=10,
        n_episodes=0, seed=3,
    )
    events = synth.generate(cfg, episodes=[])
    return list(iter_windows(events, cfg.window_seconds))


@pytest.mark.parametrize("make_detector", DETECTOR_FACTORIES)
def test_score_is_finite_and_aligned_to_edges(make_detector):
    windows = _windows()
    det = make_detector()
    det.fit(windows[:20], warm_start=False)
    for w in windows[20:25]:
        scores = det.score(w)
        assert len(scores) == len(w.edges)
        assert np.all(np.isfinite(scores))


@pytest.mark.parametrize("make_detector", DETECTOR_FACTORIES)
def test_unseen_edge_scores_higher_than_a_frequently_seen_edge(make_detector):
    """Core sanity check of the score convention: a never-seen-before
    edge must score as MORE anomalous (higher) than an edge the
    detector was trained on heavily -- higher = more anomalous is the
    convention every policy/metric in this project relies on."""
    cfg = SynthConfig(
        n_users=6, n_hosts=5, n_benign_days=3, base_edges_per_window=30,
        n_episodes=0, seed=4,
    )
    events = synth.generate(cfg, episodes=[])
    windows = list(iter_windows(events, cfg.window_seconds))
    det = make_detector()
    det.fit(windows, warm_start=False)

    # Build one synthetic test window: one edge seen a lot during
    # training (if any), one edge guaranteed never seen (brand-new
    # host names).
    from lmsafe.data.windows import WindowGraph

    # pick the most frequent edge across training for the "familiar" case
    from collections import Counter

    counts = Counter()
    for w in windows:
        for (a, b), wt in zip(w.edges, w.edge_weight):
            counts[(w.nodes[a], w.nodes[b])] += wt
    familiar_src, familiar_dst = counts.most_common(1)[0][0]

    test_window = WindowGraph(
        window_start=10**9,
        window_seconds=cfg.window_seconds,
        nodes=[familiar_src, familiar_dst, "NEVER_SEEN_SRC", "NEVER_SEEN_DST"],
        node_index={familiar_src: 0, familiar_dst: 1, "NEVER_SEEN_SRC": 2, "NEVER_SEEN_DST": 3},
        edges=[(0, 1), (2, 3)],
        edge_weight=[1, 1],
        edge_is_malicious=[False, False],
        n_events=2,
    )
    scores = det.score(test_window)
    assert scores[1] > scores[0], (
        f"expected unseen edge to score higher (more anomalous) than a familiar edge, "
        f"got familiar={scores[0]}, unseen={scores[1]}"
    )


@pytest.mark.parametrize("make_detector", DETECTOR_FACTORIES)
def test_clone_fresh_is_independent_and_untrained(make_detector):
    windows = _windows()
    det = make_detector()
    det.fit(windows[:10], warm_start=False)
    fresh = det.clone_fresh()
    assert fresh.state_summary() != det.state_summary() or fresh.state_summary() == {}
    # Fitting the clone must not affect the original.
    before = det.state_summary()
    fresh.fit(windows[10:15], warm_start=False)
    assert det.state_summary() == before


@pytest.mark.parametrize("make_detector", DETECTOR_FACTORIES)
def test_warm_start_false_resets_prior_state(make_detector):
    windows = _windows()
    det = make_detector()
    det.fit(windows[:10], warm_start=False)
    summary_after_first_fit = dict(det.state_summary())
    det.fit(windows[10:12], warm_start=False)
    # A from-scratch refit on fewer windows should not retain the
    # larger edge/host vocabulary accumulated by the first fit.
    assert det.state_summary() != summary_after_first_fit


def test_detector_score_never_reads_ground_truth_label():
    """Static guard: no detector's score() method may reference
    edge_is_malicious (see lmsafe.data.schema's ground-truth boundary)."""
    import inspect

    from lmsafe.detectors import count_baseline, feature_mlp, rw_gnn

    for module in (count_baseline, feature_mlp, rw_gnn):
        src = inspect.getsource(module)
        assert "is_malicious" not in src, f"{module.__name__} must never read ground-truth labels"


In [ ]:
%%writefile tests/test_drift.py
import numpy as np

from lmsafe.drift import PageHinkley, SimplifiedADWIN


def test_page_hinkley_rarely_alarms_on_stationary_noise():
    # Note on parameters: Page-Hinkley's cumulative-sum test statistic
    # has a range that grows as O(sigma * sqrt(n)) under pure noise, so
    # `threshold` must be calibrated to the signal's variance, not a
    # fixed constant -- a threshold tuned for a low-variance per-window
    # *mean* anomaly score (this project's actual use in
    # lmsafe.policies.DriftTriggeredPolicy) would false-alarm
    # constantly on raw, unaveraged N(0, 1) noise as used here. 30 is
    # calibrated for *this test's* noise scale, empirically giving a
    # false-alarm rate well under 10% across 30 independent seeds.
    rng = np.random.default_rng(0)
    ph = PageHinkley(delta=0.01, threshold=30.0)
    alarms = sum(ph.update(x) for x in rng.normal(0, 1, 250))
    assert alarms == 0


def test_page_hinkley_detects_mean_shift():
    rng = np.random.default_rng(0)
    ph = PageHinkley(delta=0.01, threshold=30.0)
    stream = np.concatenate([rng.normal(0, 1, 300), rng.normal(3, 1, 300)])
    detected_at = next((i for i, x in enumerate(stream) if ph.update(x)), None)
    assert detected_at is not None
    assert 300 <= detected_at < 450


def test_adwin_rarely_alarms_on_stationary_noise():
    rng = np.random.default_rng(1)
    adwin = SimplifiedADWIN(delta=0.002, max_window=400, min_window=16)
    alarms = sum(adwin.update(x) for x in rng.normal(0, 1, 500))
    assert alarms == 0


def test_adwin_detects_mean_shift():
    rng = np.random.default_rng(1)
    adwin = SimplifiedADWIN(delta=0.002, max_window=400, min_window=16)
    stream = np.concatenate([rng.normal(0, 1, 250), rng.normal(3, 1, 250)])
    detected_at = next((i for i, x in enumerate(stream) if adwin.update(x)), None)
    assert detected_at is not None
    assert 250 <= detected_at < 400


def test_adwin_ignores_short_streams_below_min_window():
    adwin = SimplifiedADWIN(min_window=16)
    for x in [0.0] * 10:
        assert adwin.update(x) is False


In [ ]:
%%writefile tests/test_policies.py
import numpy as np

from lmsafe.config import SynthConfig
from lmsafe import synth
from lmsafe.data.windows import WindowGraph, iter_windows
from lmsafe.detectors import CountBaselineDetector
from lmsafe.drift import PageHinkley
from lmsafe.policies import (
    DriftTriggeredPolicy,
    FrozenPolicy,
    GatedPolicy,
    PeriodicPolicy,
    SlidingReplayPolicy,
)

from _doubles import RecordingDetector


def _synth_windows():
    cfg = SynthConfig(
        n_users=20, n_hosts=15, n_benign_days=10, base_edges_per_window=15,
        n_episodes=2, episode_chain_len=4, seed=5, drift_strength=0.2, drift_start_day=4,
    )
    events = synth.generate(cfg)
    windows = list(iter_windows(events, cfg.window_seconds))
    train_w = [w for w in windows if w.window_start < 41 * 3600]
    test_w = [w for w in windows if w.window_start >= 41 * 3600]
    return train_w, test_w


ALL_POLICIES = [
    lambda: FrozenPolicy(),
    lambda: PeriodicPolicy(period=12),
    lambda: SlidingReplayPolicy(window_size=48, retrain_period=12),
    lambda: DriftTriggeredPolicy(PageHinkley(delta=0.01, threshold=5.0), lookback=48),
    lambda: GatedPolicy(score_admission_quantile=0.9, quarantine_windows=12, min_corroborating_src_hosts=2, retrain_period=12),
]


def test_every_policy_runs_end_to_end_without_error():
    train_w, test_w = _synth_windows()
    for make_policy in ALL_POLICIES:
        det = CountBaselineDetector()
        pol = make_policy()
        pol.fit_initial(det, train_w)
        for w in test_w:
            s = det.score(w)
            assert np.all(np.isfinite(s))
            det = pol.observe(w, s, det)


def test_frozen_never_calls_fit_again_after_initial():
    train_w, _ = _synth_windows()
    det = RecordingDetector()
    pol = FrozenPolicy()
    pol.fit_initial(det, train_w)
    assert len(det.fit_calls) == 1
    for _ in range(50):
        w = WindowGraph(window_start=0, window_seconds=3600)
        det = pol.observe(w, np.zeros(0), det)
    assert len(det.fit_calls) == 1  # still just the initial fit


def test_periodic_retrains_every_period_windows_on_disjoint_batches():
    det = RecordingDetector()
    pol = PeriodicPolicy(period=3)  # lookback defaults to period -> non-overlapping
    pol.fit_initial(det, [])
    assert det.fit_calls == [(0, False)]

    empty_w = lambda i: WindowGraph(window_start=i, window_seconds=1)
    for i in range(9):
        det = pol.observe(empty_w(i), np.zeros(0), det)
    # One retrain every 3 windows -> 3 retrains, each on exactly 3 windows.
    assert det.fit_calls == [(0, False), (3, False), (3, False), (3, False)]


def test_sliding_replay_window_overlaps_across_retrains():
    det = RecordingDetector()
    pol = SlidingReplayPolicy(window_size=5, retrain_period=2)
    pol.fit_initial(det, [])
    empty_w = lambda i: WindowGraph(window_start=i, window_seconds=1)
    for i in range(6):
        det = pol.observe(empty_w(i), np.zeros(0), det)
    # buffer caps at 5; after 6 windows appended, retrains happen at
    # i=1 (buffer size 2), i=3 (buffer size 4), i=5 (buffer size 5, capped)
    assert det.fit_calls == [(0, False), (2, False), (4, False), (5, False)]


def test_periodic_and_sliding_are_not_equivalent_with_documented_defaults():
    """Regression test: these two policies must behave differently
    when PeriodicPolicy is used with its documented default (lookback
    == period, disjoint batches) even though both are internally
    implemented as a capped deque + periodic from-scratch refit."""
    train_w, test_w = _synth_windows()[0], _synth_windows()[1][:60]

    det_p = CountBaselineDetector()
    pol_p = PeriodicPolicy(period=12)
    pol_p.fit_initial(det_p, train_w)
    scores_p = []
    for w in test_w:
        s = det_p.score(w)
        scores_p.extend(s.tolist())
        det_p = pol_p.observe(w, s, det_p)

    det_s = CountBaselineDetector()
    pol_s = SlidingReplayPolicy(window_size=168, retrain_period=12)
    pol_s.fit_initial(det_s, train_w)
    scores_s = []
    for w in test_w:
        s = det_s.score(w)
        scores_s.extend(s.tolist())
        det_s = pol_s.observe(w, s, det_s)

    assert scores_p != scores_s


def test_drift_triggered_retrains_only_when_drift_detector_fires():
    det = RecordingDetector(score_fn=lambda w: np.zeros(len(w.edges)))
    # threshold absurdly high: PageHinkley should never fire on all-zero stats
    pol = DriftTriggeredPolicy(PageHinkley(delta=0.0, threshold=1e9), lookback=10)
    pol.fit_initial(det, [])
    empty_w = lambda i: WindowGraph(window_start=i, window_seconds=1)
    for i in range(20):
        s = det.score(empty_w(i))
        det = pol.observe(empty_w(i), s, det)
    assert len(det.fit_calls) == 1  # only the initial fit, never triggered


def test_gated_admission_excludes_high_scoring_edges():
    pol = GatedPolicy(score_admission_quantile=0.5)
    w = WindowGraph(
        window_start=0, window_seconds=3600,
        nodes=["A", "B", "C", "D"], node_index={"A": 0, "B": 1, "C": 2, "D": 3},
        edges=[(0, 1), (0, 2), (0, 3)], edge_weight=[1, 1, 1], edge_is_malicious=[False, False, False],
        n_events=3,
    )
    scores = np.array([1.0, 5.0, 9.0])  # median = 5.0
    admitted = pol._admit(w, scores)
    assert admitted == [0, 1]  # only scores <= median are admitted


def test_gated_promotes_only_corroborated_edges():
    """Hand-built scenario: a conditioning edge from a single source
    host is never corroborated by other hosts and must NOT be
    promoted; a destination independently reached by several other
    source hosts within the quarantine horizon must be promoted."""
    pol = GatedPolicy(
        score_admission_quantile=1.0,  # admit everything, isolate the corroboration step
        quarantine_windows=2,
        min_corroborating_src_hosts=2,
        retrain_period=1000,  # never auto-retrain; inspect the buffer directly
    )
    pol.fit_initial(CountBaselineDetector(), [])

    def w(i, edges):
        nodes = sorted({h for e in edges for h in e})
        idx = {h: j for j, h in enumerate(nodes)}
        return WindowGraph(
            window_start=i, window_seconds=1, nodes=nodes, node_index=idx,
            edges=[(idx[a], idx[b]) for a, b in edges],
            edge_weight=[1] * len(edges), edge_is_malicious=[False] * len(edges),
            n_events=len(edges),
        )

    det = CountBaselineDetector()
    scores_zero = lambda e: np.zeros(len(e.edges))

    # Window 0: attacker "ATTACKER" alone reaches "TARGET" (never corroborated).
    # Also "SRV" is reached by three different hosts across windows 0-1 (corroborated).
    det = pol.observe(w(0, [("ATTACKER", "TARGET"), ("H1", "SRV")]), scores_zero(w(0, [("ATTACKER", "TARGET"), ("H1", "SRV")])), det)
    det = pol.observe(w(1, [("H2", "SRV")]), scores_zero(w(1, [("H2", "SRV")])), det)
    det = pol.observe(w(2, [("H3", "SRV")]), scores_zero(w(2, [("H3", "SRV")])), det)
    # window_idx is now 3; the window-0 entry (admitted_at=1) matures once
    # 3 - 1 >= quarantine_windows(2), i.e. after this third observe() call.

    promoted_edges = set()
    for cand_window in pol._committed_buffer:
        for a, b in cand_window.edges:
            promoted_edges.add((cand_window.nodes[a], cand_window.nodes[b]))

    assert ("ATTACKER", "TARGET") not in promoted_edges
    assert ("H1", "SRV") in promoted_edges


In [ ]:
%%writefile tests/test_attack.py
from lmsafe import synth
from lmsafe.attack import ConditioningSpec, boiling_frog_scenario, generate_conditioning_events, inject_conditioning
from lmsafe.data.schema import Event


def make_episode(start_s=20 * 86400):
    return synth.LMEpisode(
        start_s=start_s,
        actor_user="U_attacker@DOM1",
        actor_host="C_attacker",
        target_hosts=["C_t1", "C_t2", "C_t3"],
        hop_seconds=30,
    )


def test_conditioning_events_stay_within_window_and_target_the_actor_pair():
    spec = ConditioningSpec(
        start_s=1000, end_s=1000 + 10 * 86400, actor_user="U@DOM1",
        actor_host="C_actor", target_host="C_target", events_per_day=5.0, seed=0,
    )
    events = generate_conditioning_events(spec)
    assert len(events) > 0
    for e in events:
        assert spec.start_s <= e.t < spec.end_s
        assert e.src_comp == "C_actor"
        assert e.dst_comp == "C_target"
        assert e.source == "synthetic_conditioning"


def test_conditioning_events_not_malicious_by_default():
    spec = ConditioningSpec(
        start_s=0, end_s=5 * 86400, actor_user="U@DOM1", actor_host="C_actor",
        target_host="C_target", events_per_day=5.0, seed=1,
    )
    events = generate_conditioning_events(spec)
    assert events, "expected some events to be generated"
    assert all(not e.is_malicious for e in events)


def test_conditioning_events_can_opt_into_malicious_label():
    spec = ConditioningSpec(
        start_s=0, end_s=5 * 86400, actor_user="U@DOM1", actor_host="C_actor",
        target_host="C_target", events_per_day=5.0, label_as_malicious=True, seed=1,
    )
    events = generate_conditioning_events(spec)
    assert events and all(e.is_malicious for e in events)


def test_empty_window_produces_no_events():
    spec = ConditioningSpec(start_s=100, end_s=100, actor_user="U", actor_host="A", target_host="B")
    assert generate_conditioning_events(spec) == []


def test_boiling_frog_scenario_targets_same_actor_and_first_hop():
    ep = make_episode()
    spec = boiling_frog_scenario(ep, conditioning_days=7, events_per_day=2.0, gap_seconds=3600, seed=0)
    assert spec.actor_host == ep.actor_host
    assert spec.actor_user == ep.actor_user
    assert spec.target_host == ep.target_hosts[0]
    assert spec.end_s == ep.start_s - 3600
    assert spec.start_s == spec.end_s - 7 * 86400


def test_boiling_frog_scenario_never_overlaps_the_real_attack():
    ep = make_episode()
    spec = boiling_frog_scenario(ep, conditioning_days=7, gap_seconds=3600, seed=0)
    assert spec.end_s < ep.start_s


def test_inject_conditioning_merges_and_keeps_stream_sorted():
    base = [
        Event(t=0, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2", source="synthetic"),
        Event(t=10 * 86400, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2", source="synthetic"),
    ]
    spec = ConditioningSpec(
        start_s=1000, end_s=5 * 86400, actor_user="U@DOM1", actor_host="C_a",
        target_host="C_b", events_per_day=10.0, seed=2,
    )
    merged = inject_conditioning(base, spec)
    assert [e.t for e in merged] == sorted(e.t for e in merged)
    assert len(merged) > len(base)
    # original list must not be mutated
    assert len(base) == 2


def test_inject_conditioning_with_no_events_returns_equivalent_list():
    base = [Event(t=0, src_user="a", dst_user="a", src_comp="C1", dst_comp="C2", source="synthetic")]
    spec = ConditioningSpec(start_s=5, end_s=5, actor_user="U", actor_host="A", target_host="B")
    merged = inject_conditioning(base, spec)
    assert merged == base
    assert merged is not base


In [ ]:
%%writefile tests/test_metrics.py
import math

import numpy as np
import pytest

from lmsafe.metrics import (
    WindowScores,
    block_bootstrap_ci,
    cluster_episodes,
    detection_delay_per_episode,
    episode_bootstrap_ci,
    episode_hit,
    recall_at_budget,
    weighted_ap,
    weighted_auc,
)


def ws(window_start, scores, is_malicious, weight=None):
    scores = np.asarray(scores, dtype=np.float64)
    is_malicious = np.asarray(is_malicious, dtype=bool)
    weight = np.ones(len(scores)) if weight is None else np.asarray(weight, dtype=np.float64)
    return WindowScores(window_start=window_start, scores=scores, is_malicious=is_malicious, weight=weight)


def test_weighted_auc_perfect_separation():
    records = [ws(0, [0.1, 0.2, 9.0, 9.5], [False, False, True, True])]
    assert weighted_auc(records) == pytest.approx(1.0)


def test_weighted_auc_nan_when_single_class():
    records = [ws(0, [0.1, 0.2], [False, False])]
    assert math.isnan(weighted_auc(records))


def test_weighted_ap_nan_when_no_positives():
    records = [ws(0, [0.1, 0.2], [False, False])]
    assert math.isnan(weighted_ap(records))


def test_weighted_auc_respects_event_weight():
    # A heavily-weighted misranked benign row should pull AUC down more
    # than a lightly-weighted one -- sanity check that sample_weight is
    # actually wired through, not silently ignored.
    light = [ws(0, [5.0, 1.0, 0.0], [True, False, False], weight=[1, 1, 1])]
    heavy = [ws(0, [5.0, 1.0, 0.0], [True, False, False], weight=[1, 100, 1])]
    assert weighted_auc(heavy) <= weighted_auc(light)


def test_recall_at_budget_basic_ranking():
    # window with 4 edges, budget=2: top-2 by score are indices 2,3 (scores 9,8)
    records = [ws(0, [1, 2, 9, 8], [False, True, True, False])]
    r = recall_at_budget(records, budget_per_window=2, weighted=False)
    # malicious rows: idx1 (score2, not in top2) and idx2 (score9, in top2)
    assert r == pytest.approx(0.5)


def test_recall_at_budget_ignores_windows_with_no_malicious_rows():
    records = [ws(0, [1, 2], [False, False]), ws(1, [1, 9], [False, True])]
    r = recall_at_budget(records, budget_per_window=1, weighted=False)
    assert r == pytest.approx(1.0)


def test_recall_at_budget_nan_when_no_malicious_anywhere():
    records = [ws(0, [1, 2], [False, False])]
    assert math.isnan(recall_at_budget(records, 1))


def test_recall_at_budget_weighted_uses_event_counts():
    # single malicious row with weight=5 caught; total malicious weight=5
    records = [ws(0, [9.0, 1.0], [True, False], weight=[5, 1])]
    assert recall_at_budget(records, budget_per_window=1, weighted=True) == pytest.approx(1.0)


def test_cluster_episodes_groups_by_gap():
    # malicious windows at positions 0,1 (episode A), then 5 (episode B)
    records = [
        ws(0, [9.0], [True]),
        ws(1, [9.0], [True]),
        ws(2, [1.0], [False]),
        ws(3, [1.0], [False]),
        ws(4, [1.0], [False]),
        ws(5, [9.0], [True]),
    ]
    episodes = cluster_episodes(records, gap_windows=1)
    assert len(episodes) == 2
    assert {i for i, _ in episodes[0]} == {0, 1}
    assert {i for i, _ in episodes[1]} == {5}


def test_cluster_episodes_merges_within_allowed_gap():
    records = [ws(0, [9.0], [True]), ws(1, [1.0], [False]), ws(2, [9.0], [True])]
    # one empty window between two malicious windows: gap_windows=1 should merge them
    episodes = cluster_episodes(records, gap_windows=1)
    assert len(episodes) == 1
    # gap_windows=0 should NOT merge them
    episodes0 = cluster_episodes(records, gap_windows=0)
    assert len(episodes0) == 2


def test_cluster_episodes_empty_when_no_malicious():
    records = [ws(0, [1.0], [False])]
    assert cluster_episodes(records) == []


def test_detection_delay_zero_when_caught_in_first_window():
    records = [ws(0, [9.0, 1.0], [True, False])]
    episodes = cluster_episodes(records, gap_windows=0)
    delays = detection_delay_per_episode(records, budget_per_window=1, episodes=episodes)
    assert delays == [0]


def test_detection_delay_counts_windows_until_caught():
    # episode spans windows 0 and 1; only caught (within budget=1) at window 1
    records = [
        ws(0, [1.0, 9.0], [True, False]),   # malicious row ranked 2nd -> missed at budget 1
        ws(1, [9.0, 1.0], [True, False]),   # malicious row ranked 1st -> caught
    ]
    episodes = cluster_episodes(records, gap_windows=0)
    delays = detection_delay_per_episode(records, budget_per_window=1, episodes=episodes)
    assert delays == [1]


def test_detection_delay_none_when_never_caught():
    records = [ws(0, [1.0, 9.0], [True, False])]
    episodes = cluster_episodes(records, gap_windows=0)
    delays = detection_delay_per_episode(records, budget_per_window=1, episodes=episodes)
    assert delays == [None]


def test_episode_hit_matches_delay_none():
    records = [ws(0, [1.0, 9.0], [True, False])]
    episodes = cluster_episodes(records, gap_windows=0)
    assert episode_hit(records, 1, episodes) == [False]


def test_episode_bootstrap_ci_on_constant_values():
    pt, lo, hi = episode_bootstrap_ci([2.0, 2.0, 2.0], n_boot=100, seed=0)
    assert pt == pytest.approx(2.0)
    assert lo == pytest.approx(2.0)
    assert hi == pytest.approx(2.0)


def test_episode_bootstrap_ci_empty_input():
    pt, lo, hi = episode_bootstrap_ci([], n_boot=10)
    assert math.isnan(pt) and math.isnan(lo) and math.isnan(hi)


def test_block_bootstrap_ci_point_estimate_matches_direct_call():
    records = [ws(i, [1.0, 9.0], [False, i % 5 == 0]) for i in range(40)]
    point, lo, hi = block_bootstrap_ci(weighted_ap, records, block_size=8, n_boot=50, seed=0)
    assert point == pytest.approx(weighted_ap(records))
    assert lo <= point + 1e-9  # point estimate should be a plausible value near the CI
    assert not math.isnan(lo) and not math.isnan(hi)


def test_block_bootstrap_ci_handles_empty_records():
    point, lo, hi = block_bootstrap_ci(weighted_ap, [], block_size=8, n_boot=10)
    assert math.isnan(point)


In [ ]:
%%writefile tests/test_runner.py
"""Tests for lmsafe.runner: the detector/policy/metrics glue.

Uses `RecordingDetector` with a caller-supplied `score_fn` so every
score the metrics see is exactly known -- this lets the expected
weighted_auc / rec_at_budget / episode_hit_rate / delay values be
worked out by hand instead of merely checking "it runs".
"""
import math

import numpy as np
import pytest

from lmsafe.data.windows import WindowGraph
from lmsafe.policies import FrozenPolicy, PeriodicPolicy
from lmsafe.runner import ExperimentResult, run_experiment_cell, run_grid

from _doubles import RecordingDetector


def _window(window_start, malicious, scores):
    """Two-edge window: edges[(0,1), (1,0)] over nodes h0,h1.

    `malicious` and `scores` are length-2 sequences aligned to the
    edges above.
    """
    return WindowGraph(
        window_start=window_start,
        window_seconds=3600,
        nodes=["h0", "h1"],
        node_index={"h0": 0, "h1": 1},
        edges=[(0, 1), (1, 0)],
        edge_weight=[1, 1],
        edge_is_malicious=list(malicious),
        n_events=2,
    )


def _score_fn_table(table):
    """table: dict[window_start] -> list[float] of per-edge scores."""

    def _fn(window):
        return np.asarray(table[window.window_start], dtype=np.float64)

    return _fn


def _fixed_scenario():
    """Three test windows, one malicious edge total, deterministically
    ranked so recall/delay/hit-rate are all hand-checkable:

    w0 (t=0):    both benign,       scores [1.0, 0.0]
    w1 (t=3600): edge0 malicious,   scores [9.0, 0.0]  -> malicious edge ranks #1
    w2 (t=7200): both benign,       scores [1.0, 0.0]
    """
    train_windows = [_window(-7200, [False, False], [0.0, 0.0])]
    test_windows = [
        _window(0, [False, False], [1.0, 0.0]),
        _window(3600, [True, False], [9.0, 0.0]),
        _window(7200, [False, False], [1.0, 0.0]),
    ]
    scores_by_start = {0: [1.0, 0.0], 3600: [9.0, 0.0], 7200: [1.0, 0.0]}
    return train_windows, test_windows, scores_by_start


def test_run_experiment_cell_basic_metrics():
    train_w, test_w, score_table = _fixed_scenario()
    det = RecordingDetector(score_fn=_score_fn_table(score_table))
    pol = FrozenPolicy()

    result = run_experiment_cell(
        train_w,
        test_w,
        det,
        pol,
        detector_name="recording",
        scenario="unit_test",
        budgets=(1, 2),
        gap_windows_for_episodes=0,
        delay_budget=1,
    )

    assert isinstance(result, ExperimentResult)
    assert result.detector_name == "recording"
    assert result.policy_name == pol.name
    assert result.scenario == "unit_test"
    assert result.n_test_windows == 3
    assert result.n_episodes == 1  # one malicious window, isolated

    # Rec@1: in w1 the malicious edge is top-ranked -> caught; no
    # malicious rows in w0/w2, so those windows don't count.
    assert result.rec_at_budget[1] == pytest.approx(1.0)
    assert result.rec_at_budget[2] == pytest.approx(1.0)

    # Only one episode, caught in its very first (only) window -> delay 0.
    assert result.episode_hit_rate == pytest.approx(1.0)
    assert result.mean_delay_windows == pytest.approx(0.0)

    # AUC: pooled across all 6 edge-rows, malicious row scores 9.0 which
    # is the global max -> perfect separation.
    assert result.weighted_auc == pytest.approx(1.0)
    assert not math.isnan(result.weighted_ap)


def test_run_experiment_cell_calls_fit_initial_once():
    train_w, test_w, score_table = _fixed_scenario()
    det = RecordingDetector(score_fn=_score_fn_table(score_table))
    pol = FrozenPolicy()
    run_experiment_cell(train_w, test_w, det, pol, detector_name="recording")
    # FrozenPolicy.fit_initial calls detector.fit exactly once, and
    # FrozenPolicy.observe never refits -- this is the same contract
    # test_policies.py checks directly, exercised here through the runner.
    assert len(det.fit_calls) == 1
    assert det.fit_calls[0] == (len(train_w), False)


def test_run_experiment_cell_missed_detection_increases_delay_and_drops_recall():
    # Same shape as the fixed scenario, but the malicious edge is now
    # ranked LAST in its window, and budget=1 can't catch it there,
    # only in a later (here: nonexistent) window -> never caught.
    train_windows = [_window(-7200, [False, False], [0.0, 0.0])]
    test_windows = [
        _window(0, [True, False], [0.0, 9.0]),  # malicious edge ranks #2 of 2
    ]
    score_table = {0: [0.0, 9.0]}
    det = RecordingDetector(score_fn=_score_fn_table(score_table))
    pol = FrozenPolicy()

    result = run_experiment_cell(
        train_windows,
        test_windows,
        det,
        pol,
        detector_name="recording",
        budgets=(1,),
        gap_windows_for_episodes=0,
        delay_budget=1,
    )
    assert result.rec_at_budget[1] == pytest.approx(0.0)
    assert result.episode_hit_rate == pytest.approx(0.0)
    # never caught -> no finite delay to average
    assert math.isnan(result.mean_delay_windows)


def test_to_row_flattens_rec_at_budget_and_diagnostics():
    train_w, test_w, score_table = _fixed_scenario()
    det = RecordingDetector(score_fn=_score_fn_table(score_table))
    # PeriodicPolicy (unlike FrozenPolicy) has scalar instance attrs
    # (period, lookback, _since_retrain), which is what exercises the
    # diag_* namespacing in ExperimentResult.to_row.
    pol = PeriodicPolicy(period=1)
    result = run_experiment_cell(
        train_w, test_w, det, pol, detector_name="recording", budgets=(1, 5)
    )
    row = result.to_row()
    assert row["detector"] == "recording"
    assert row["policy"] == pol.name
    assert "rec_at_1" in row and "rec_at_5" in row
    assert row["rec_at_1"] == pytest.approx(result.rec_at_budget[1])
    assert row["rec_at_5"] == pytest.approx(result.rec_at_budget[5])
    # diagnostics (policy's own scalar attrs) are namespaced diag_*
    assert any(k.startswith("diag_") for k in row)
    assert row.get("diag_policy_period") == 1


def test_run_grid_produces_one_row_per_cell():
    train_w, test_w, score_table = _fixed_scenario()
    score_fn = _score_fn_table(score_table)

    cells = [
        {
            "detector": RecordingDetector(score_fn=score_fn),
            "detector_name": "recording",
            "policy": FrozenPolicy(),
            "scenario_key": "scenario_a",
        },
        {
            "detector": RecordingDetector(score_fn=score_fn),
            "detector_name": "recording",
            "policy": FrozenPolicy(),
            "scenario_key": "scenario_b",
        },
    ]

    df = run_grid(
        cells,
        train_windows_fn=lambda cell: train_w,
        test_windows_fn=lambda cell: test_w,
        budgets=(1,),
    )
    assert len(df) == 2
    assert set(df["scenario"]) == {"scenario_a", "scenario_b"}
    assert "rec_at_1" in df.columns
    # both cells run the identical fixed scenario -> identical AUC
    assert df["weighted_auc"].nunique() == 1


def test_run_grid_respects_explicit_scenario_label_over_key():
    train_w, test_w, score_table = _fixed_scenario()
    score_fn = _score_fn_table(score_table)
    cells = [
        {
            "detector": RecordingDetector(score_fn=score_fn),
            "detector_name": "recording",
            "policy": FrozenPolicy(),
            "scenario_key": "ignored_key",
            "scenario": "explicit_label",
        }
    ]
    df = run_grid(
        cells,
        train_windows_fn=lambda cell: train_w,
        test_windows_fn=lambda cell: test_w,
        budgets=(1,),
    )
    assert df.iloc[0]["scenario"] == "explicit_label"


In [ ]:
%%writefile tests/test_no_label_leak.py
"""Static enforcement of this project's central fairness invariant:

    No `Detector.score()` / `Detector.fit()` and no `UpdatePolicy.observe()`
    / `UpdatePolicy.fit_initial()` implementation may read ground-truth
    labels (`Event.is_malicious` / `WindowGraph.edge_is_malicious`).

See `lmsafe/data/schema.py`'s module docstring and `lmsafe/policies.py`
line ~74 for the documented rationale: `is_malicious` exists purely for
`lmsafe.metrics` to grade detectors/policies after the fact, via the
single sanctioned read in `WindowScores.from_window`
(`lmsafe/metrics.py`). Anything scored or decided *during* the
experiment must be blind to it, or every AUC/AP/Rec@B number in the
paper would be measuring a detector that cheated.

This is checked with `ast`, not a plain substring search, because
`policies.py` and `detectors/base.py` legitimately *mention*
`edge_is_malicious` in their docstrings (to document the very
invariant this test enforces) -- a substring check over the whole
module source would false-positive on those comments. Parsing the AST
and looking only at `Attribute` nodes (`<expr>.is_malicious` /
`<expr>.edge_is_malicious`) ignores docstrings and comments entirely,
so it only fires on actual code that reads the field.
"""
from __future__ import annotations

import ast
import importlib
import inspect

import pytest

FORBIDDEN_ATTRS = {"is_malicious", "edge_is_malicious"}

# Every module that implements (or could implement) a Detector or an
# UpdatePolicy. Extend this list when a new detector/policy module is
# added -- `test_module_list_is_complete` below guards against
# forgetting to.
CHECKED_MODULES = [
    "lmsafe.policies",
    "lmsafe.detectors.base",
    "lmsafe.detectors.count_baseline",
    "lmsafe.detectors.feature_mlp",
    "lmsafe.detectors.rw_gnn",
]


def _attribute_accesses(tree: ast.AST) -> set[str]:
    """All `.attr` names referenced anywhere in actual code (assignments,
    conditionals, calls, ...) -- not inside string literals/docstrings,
    since those never become `ast.Attribute` nodes."""
    return {node.attr for node in ast.walk(tree) if isinstance(node, ast.Attribute)}


@pytest.mark.parametrize("module_name", CHECKED_MODULES)
def test_module_never_accesses_ground_truth_label(module_name):
    module = importlib.import_module(module_name)
    source = inspect.getsource(module)
    tree = ast.parse(source)
    accessed = _attribute_accesses(tree)
    leaked = accessed & FORBIDDEN_ATTRS
    assert not leaked, (
        f"{module_name} contains code that reads {leaked} -- "
        "detectors/policies must be blind to ground-truth labels "
        "(see lmsafe/data/schema.py and lmsafe/metrics.py:WindowScores.from_window)"
    )


def test_docstrings_may_still_document_the_invariant_by_name():
    """Sanity check on the test itself: policies.py and detectors/base.py
    are *expected* to mention `edge_is_malicious` in prose (that's how
    the invariant gets documented for future contributors) -- this
    confirms the AST check above tolerates that and isn't accidentally
    passing only because the docstrings were removed."""
    import lmsafe.detectors.base as base
    import lmsafe.policies as policies

    assert "edge_is_malicious" in inspect.getsource(policies)
    assert "edge_is_malicious" in inspect.getsource(base)


def test_module_list_is_complete():
    """Guard against silently forgetting to add a new detector/policy
    module to CHECKED_MODULES: every .py file under lmsafe/detectors/
    (except __init__) must be covered."""
    import pathlib

    import lmsafe.detectors as detectors_pkg

    pkg_dir = pathlib.Path(detectors_pkg.__file__).parent
    found = {
        f"lmsafe.detectors.{p.stem}"
        for p in pkg_dir.glob("*.py")
        if p.stem != "__init__"
    }
    assert found <= set(CHECKED_MODULES), (
        f"detector module(s) {found - set(CHECKED_MODULES)} are not covered by "
        "test_no_label_leak.py -- add them to CHECKED_MODULES"
    )


In [ ]:
%%writefile tests/test_build_cache_checkpoint.py
"""Regression tests for scripts/fetch_lanl_window.py's `build-cache`
checkpointing.

Context: a real 58-day LANL run was interrupted (Ctrl+C / a Colab
disconnect) at ~87% progress after several hours, and the user found
*nothing* written to disk -- the old `cmd_build_cache` only pickled
`(train_w, test_w)` once, at the very end, after the entire multi-hour
stream had been fully consumed. These tests exercise the fix: periodic
atomic checkpoints during the test-window loop, so an interruption
partway through still leaves a usable, loadable cache file.

`scripts/` has no `__init__.py` (it's a collection of CLI entry points,
not an importable package), so the module is loaded directly from its
file path rather than via a dotted import.
"""
import importlib.util
import pickle
import sys
from pathlib import Path

import pytest

from lmsafe.data.windows import WindowGraph

SCRIPT_PATH = Path(__file__).parent.parent / "scripts" / "fetch_lanl_window.py"


def _load_module():
    spec = importlib.util.spec_from_file_location("fetch_lanl_window", SCRIPT_PATH)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


fetch_lanl_window = _load_module()


def _window(i: int) -> WindowGraph:
    """A trivial, distinct, non-empty window so successive checkpoints
    are easy to tell apart by length alone."""
    w = WindowGraph(window_start=i * 3600, window_seconds=3600)
    w.nodes = [f"C{i}"]
    w.node_index = {f"C{i}": 0}
    w.n_events = 1
    return w


def test_atomic_pickle_dump_leaves_no_tmp_file(tmp_path):
    out = tmp_path / "cache.pkl"
    fetch_lanl_window._atomic_pickle_dump(([1, 2], [3, 4]), out)
    assert out.exists()
    assert not out.with_suffix(out.suffix + ".tmp").exists()
    with open(out, "rb") as fh:
        assert pickle.load(fh) == ([1, 2], [3, 4])


def test_atomic_pickle_dump_overwrite_keeps_latest(tmp_path):
    out = tmp_path / "cache.pkl"
    fetch_lanl_window._atomic_pickle_dump(([], ["a"]), out)
    fetch_lanl_window._atomic_pickle_dump(([], ["a", "b", "c"]), out)
    with open(out, "rb") as fh:
        train, test = pickle.load(fh)
    assert test == ["a", "b", "c"]
    assert not out.with_suffix(out.suffix + ".tmp").exists()


def test_build_cache_writes_checkpoint_before_completion(tmp_path, monkeypatch):
    """The core regression guard: simulate a `build-cache` run that gets
    interrupted partway through the test-window stream (the real-world
    failure mode -- a long-running Colab cell killed by Ctrl+C or a
    disconnect), and confirm a *usable* checkpoint was already on disk
    before the interruption, covering most but not all of the windows.
    """

    class _Interrupted(Exception):
        pass

    total_windows = 50
    checkpoint_every = 10
    kill_after = 37  # strictly between two checkpoint boundaries (30, 40)

    # `cmd_build_cache` calls `iter_windows` twice: once to build the
    # (small, eager) train-window list, once for the (long, streamed)
    # test-window list. Only the *second* call is the one a real
    # multi-hour run would be interrupted during -- the first call must
    # complete normally so the test exercises the intended code path.
    calls = {"n": 0}

    def fake_iter_windows(events_iter, window_seconds):
        calls["n"] += 1
        if calls["n"] == 1:
            yield _window(0)  # trivial, already-complete train pass
            return
        for i in range(total_windows):
            if i == kill_after:
                raise _Interrupted("simulated Ctrl+C / Colab disconnect")
            yield _window(i)

    monkeypatch.setattr(fetch_lanl_window, "iter_windows", fake_iter_windows)
    monkeypatch.setattr(
        fetch_lanl_window.lanl_stream, "resolve_base_url", lambda: "https://example.invalid/cyber1/"
    )
    monkeypatch.setattr(fetch_lanl_window.lanl, "load_redteam_records", lambda base_url: [])
    monkeypatch.setattr(
        fetch_lanl_window.lanl,
        "stream_auth_events",
        lambda cfg, base_url, redteam: iter(()),
    )
    monkeypatch.setattr(
        fetch_lanl_window.lanl, "split_train_test", lambda events_iter, cfg: ([], iter(()))
    )

    out = tmp_path / "lanl_dev_windows.pkl"
    args = type(
        "A",
        (),
        dict(
            end_s=100,
            start_s=0,
            train_cutoff_s=0,
            no_logon_filter=False,
            out=str(out),
            checkpoint_every=checkpoint_every,
        ),
    )()

    with pytest.raises(_Interrupted):
        fetch_lanl_window.cmd_build_cache(args)

    # The run never reached the end (window 50), so no *final* write ever
    # happened -- but it must still have left the last periodic
    # checkpoint (at window 30, the last multiple of 10 before the kill
    # at 37) on disk, loadable and usable right away.
    assert out.exists(), "interrupted run left nothing on disk -- checkpointing regressed"
    with open(out, "rb") as fh:
        train_w, test_w = pickle.load(fh)
    assert len(test_w) == 30, (
        f"expected the last checkpoint before the kill (30 windows), got {len(test_w)}"
    )
    assert not out.with_suffix(out.suffix + ".tmp").exists()


def test_build_cache_final_file_has_all_windows_on_clean_completion(tmp_path, monkeypatch):
    total_windows = 23  # deliberately not a multiple of checkpoint_every

    calls = {"n": 0}

    def fake_iter_windows(events_iter, window_seconds):
        calls["n"] += 1
        if calls["n"] == 1:
            yield _window(0)  # trivial train pass
            return
        for i in range(total_windows):
            yield _window(i)

    monkeypatch.setattr(fetch_lanl_window, "iter_windows", fake_iter_windows)
    monkeypatch.setattr(
        fetch_lanl_window.lanl_stream, "resolve_base_url", lambda: "https://example.invalid/cyber1/"
    )
    monkeypatch.setattr(fetch_lanl_window.lanl, "load_redteam_records", lambda base_url: [])
    monkeypatch.setattr(
        fetch_lanl_window.lanl,
        "stream_auth_events",
        lambda cfg, base_url, redteam: iter(()),
    )
    monkeypatch.setattr(
        fetch_lanl_window.lanl, "split_train_test", lambda events_iter, cfg: ([], iter(()))
    )

    out = tmp_path / "cache.pkl"
    args = type(
        "A",
        (),
        dict(
            end_s=100,
            start_s=0,
            train_cutoff_s=0,
            no_logon_filter=False,
            out=str(out),
            checkpoint_every=10,
        ),
    )()

    fetch_lanl_window.cmd_build_cache(args)

    with open(out, "rb") as fh:
        train_w, test_w = pickle.load(fh)
    # The *final* write must include every window, not just the last
    # checkpoint boundary (20) -- the trailing 3 windows (20, 21, 22)
    # must not be silently dropped.
    assert len(test_w) == total_windows
    assert not out.with_suffix(out.suffix + ".tmp").exists()


In [ ]:
%%writefile tests/test_run_experiment_lanl_checkpoint.py
"""Regression tests for scripts/run_experiment.py's `build_lanl_windows`
checkpointing against `lanl.cache_path`.

Context: this is the *actual* code path the user hit in the field -- a
58-day real LANL run invoked as `run_experiment.py configs/lanl_full.yaml`
printed `[run_experiment] streaming LANL auth events ...` (this
function's own message), reached ~87% progress over several hours, was
interrupted (Ctrl+C / a Colab disconnect), and left *nothing* at
`cache_path` -- the old code only ever pickled `(train_w, test_w)` once,
after the entire multi-hour stream had been consumed. `scripts/
fetch_lanl_window.py`'s separate `build-cache` CLI got the same class of
fix, but that is a different code path and does not, by itself, cover
`run_experiment.py`'s own independent streaming branch -- hence this
second, dedicated regression test for the path that actually failed.

`scripts/` has no `__init__.py`, so the module is loaded directly from
its file path rather than via a dotted import (matching
test_build_cache_checkpoint.py's approach for the sibling script).
"""
import importlib.util
import pickle
from pathlib import Path

from lmsafe.data.windows import WindowGraph

SCRIPT_PATH = Path(__file__).parent.parent / "scripts" / "run_experiment.py"


def _load_module():
    spec = importlib.util.spec_from_file_location("run_experiment", SCRIPT_PATH)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


run_experiment = _load_module()


def _window(i: int) -> WindowGraph:
    w = WindowGraph(window_start=i * 3600, window_seconds=3600)
    w.nodes = [f"C{i}"]
    w.node_index = {f"C{i}": 0}
    w.n_events = 1
    return w


def _patch_common(monkeypatch, iter_windows_fn):
    monkeypatch.setattr(run_experiment, "iter_windows", iter_windows_fn)
    monkeypatch.setattr(
        run_experiment.lanl.lanl_stream, "resolve_base_url", lambda: "https://example.invalid/cyber1/"
    )
    monkeypatch.setattr(run_experiment.lanl, "load_redteam_records", lambda base_url: [])
    monkeypatch.setattr(
        run_experiment.lanl, "stream_auth_events", lambda cfg, base_url, redteam: iter(())
    )
    monkeypatch.setattr(
        run_experiment.lanl, "split_train_test", lambda events_iter, cfg: ([], iter(()))
    )


def test_build_lanl_windows_checkpoints_before_interruption(tmp_path, monkeypatch):
    """The core regression guard, against the exact function and
    `cache_path` mechanism the field failure went through."""

    class _Interrupted(Exception):
        pass

    total_windows = 50
    kill_after = 37  # strictly between checkpoint boundaries 30 and 40
    calls = {"n": 0}

    def fake_iter_windows(events_iter, window_seconds):
        calls["n"] += 1
        if calls["n"] == 1:
            yield _window(0)  # the eager train-window pass
            return
        for i in range(total_windows):
            if i == kill_after:
                raise _Interrupted("simulated Ctrl+C / Colab disconnect")
            yield _window(i)

    _patch_common(monkeypatch, fake_iter_windows)

    cache_path = tmp_path / "lanl_full_windows.pkl"
    cfg_dict = {
        "end_s": 5_011_200,
        "train_cutoff_s": 147_600,
        "only_remote_logon": True,
        "cache_path": str(cache_path),
        "checkpoint_every": 10,
    }

    try:
        run_experiment.build_lanl_windows(cfg_dict)
        assert False, "expected the simulated interruption to propagate"
    except _Interrupted:
        pass

    assert cache_path.exists(), (
        "an interrupted real-LANL run left nothing at cache_path -- "
        "checkpointing regressed in run_experiment.py's own streaming path"
    )
    with open(cache_path, "rb") as fh:
        train_w, test_w = pickle.load(fh)
    assert len(test_w) == 30, f"expected the last checkpoint (30 windows), got {len(test_w)}"
    assert not cache_path.with_suffix(cache_path.suffix + ".tmp").exists()


def test_build_lanl_windows_final_file_complete_on_clean_run(tmp_path, monkeypatch):
    total_windows = 23
    calls = {"n": 0}

    def fake_iter_windows(events_iter, window_seconds):
        calls["n"] += 1
        if calls["n"] == 1:
            yield _window(0)
            return
        for i in range(total_windows):
            yield _window(i)

    _patch_common(monkeypatch, fake_iter_windows)

    cache_path = tmp_path / "cache.pkl"
    cfg_dict = {
        "end_s": 2_592_000,
        "train_cutoff_s": 147_600,
        "only_remote_logon": True,
        "cache_path": str(cache_path),
        "checkpoint_every": 10,
    }

    train_w, test_w = run_experiment.build_lanl_windows(cfg_dict)
    assert len(test_w) == total_windows
    with open(cache_path, "rb") as fh:
        cached_train, cached_test = pickle.load(fh)
    assert len(cached_test) == total_windows
    assert not cache_path.with_suffix(cache_path.suffix + ".tmp").exists()


def test_loading_a_partial_checkpoint_prints_a_short_coverage_warning(tmp_path, monkeypatch, capsys):
    """The exact scenario reported in the field: a real 58-day run was
    Ctrl+C'd partway through, leaving a valid, loadable checkpoint that
    covers far fewer days than the config's own `end_s` asks for.
    Simply reusing that cache_path on a later invocation must not look
    identical to a finished run in the logs -- it has to say, loudly,
    how much of the requested window it actually covers."""
    cache_path = tmp_path / "lanl_full_windows.pkl"
    # 30 hourly windows starting at t=0 -> covers up to hour 30 (~1.25
    # days), nowhere near the 58 days `end_s` below asks for.
    test_w = [_window(i) for i in range(30)]
    with open(cache_path, "wb") as fh:
        pickle.dump(([_window(0)], test_w), fh)

    cfg_dict = {
        "end_s": 5_011_200,  # 58 days
        "train_cutoff_s": 147_600,
        "only_remote_logon": True,
        "cache_path": str(cache_path),
    }
    train_w, loaded_test_w = run_experiment.build_lanl_windows(cfg_dict)
    assert len(loaded_test_w) == 30

    out = capsys.readouterr().out
    assert "WARNING" in out
    assert "partial checkpoint" in out
    assert "1.25" in out  # covered_days
    assert "58.0" in out  # requested_days


def test_loading_a_complete_cache_prints_no_warning(tmp_path, monkeypatch, capsys):
    cache_path = tmp_path / "cache.pkl"
    # end_s below is 30 hours; a cache whose last window reaches hour 30
    # exactly covers the full requested window, so no warning is due.
    test_w = [_window(i) for i in range(30)]
    with open(cache_path, "wb") as fh:
        pickle.dump(([_window(0)], test_w), fh)

    cfg_dict = {
        "end_s": 30 * 3600,
        "train_cutoff_s": 0,
        "only_remote_logon": True,
        "cache_path": str(cache_path),
    }
    run_experiment.build_lanl_windows(cfg_dict)

    out = capsys.readouterr().out
    assert "WARNING" not in out


def test_build_lanl_windows_without_cache_path_still_works(monkeypatch):
    """No `cache_path` means no checkpointing target -- the function
    must fall back to the plain in-memory list build (no pickling at
    all), not crash for lack of somewhere to checkpoint to."""
    total_windows = 5
    calls = {"n": 0}

    def fake_iter_windows(events_iter, window_seconds):
        calls["n"] += 1
        if calls["n"] == 1:
            yield _window(0)
            return
        for i in range(total_windows):
            yield _window(i)

    _patch_common(monkeypatch, fake_iter_windows)

    cfg_dict = {
        "end_s": 2_592_000,
        "train_cutoff_s": 147_600,
        "only_remote_logon": True,
    }
    train_w, test_w = run_experiment.build_lanl_windows(cfg_dict)
    assert len(test_w) == total_windows


In [ ]:
%%writefile tests/test_rw_gnn.py
"""Dedicated tests for lmsafe.detectors.rw_gnn.GraphFoundationDetector,
beyond the shared Detector-contract tests in test_detectors.py.

These check the two things that are specific to *this* detector and not
covered by the generic parametrized suite: (1) the neighborhood
aggregation actually changes scores relative to not having it (i.e. this
isn't secretly just FeatureMLPDetector under another name), and (2) a
pretrained checkpoint genuinely transfers to a network with a completely
disjoint host-naming scheme -- the whole point of building this detector
for the cross-network-transfer research question.
"""
import numpy as np
import torch

from lmsafe.config import SynthConfig
from lmsafe import synth
from lmsafe.data.windows import WindowGraph, iter_windows
from lmsafe.detectors.rw_gnn import GraphFoundationDetector, _GraphEncoder
from lmsafe.detectors.feature_mlp import HOST_FEATURE_DIM


def _windows(seed=3, n_hosts=12):
    cfg = SynthConfig(
        n_users=15, n_hosts=n_hosts, n_benign_days=4, base_edges_per_window=10,
        n_episodes=0, seed=seed,
    )
    events = synth.generate(cfg, episodes=[])
    return list(iter_windows(events, cfg.window_seconds))


def test_encoder_neighborhood_aggregation_is_not_a_no_op():
    """The whole reason this is a GNN and not just FeatureMLPDetector
    again: a node's encoded representation must depend on its
    neighbors' features, not only its own. Build two tiny windows with
    IDENTICAL node 0 but different neighbors, and confirm node 0's
    post-encoder representation differs between them."""
    torch.manual_seed(0)
    encoder = _GraphEncoder(HOST_FEATURE_DIM, hidden_dim=8)

    h0_a = torch.zeros(3, HOST_FEATURE_DIM)
    h0_a[1] = 1.0  # node 0's only neighbor has feature vector of all-ones
    edge_index_a = torch.tensor([[0], [1]], dtype=torch.long)

    h0_b = torch.zeros(3, HOST_FEATURE_DIM)
    h0_b[2] = 5.0  # same graph shape, but node 0's neighbor feature is different
    edge_index_b = torch.tensor([[0], [2]], dtype=torch.long)

    out_a = encoder(h0_a, edge_index_a)
    out_b = encoder(h0_b, edge_index_b)
    assert not torch.allclose(out_a[0], out_b[0]), (
        "node 0's representation did not change when its neighbor's "
        "feature changed -- the neighborhood aggregation is a no-op"
    )


def test_fit_and_score_runs_on_synthetic_windows():
    windows = _windows()
    det = GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=0)
    det.fit(windows[:15], warm_start=False)
    scores = det.score(windows[15])
    assert len(scores) == len(windows[15].edges)
    assert np.all(np.isfinite(scores))


def test_export_and_load_state_dict_round_trips_identical_scores():
    windows = _windows()
    det = GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=1)
    det.fit(windows[:15], warm_start=False)
    scores_before = det.score(windows[15])

    checkpoint = det.export_state_dict()
    fresh = GraphFoundationDetector(
        epochs_per_fit=2, hidden_dim=8, seed=1, pretrained_state_dict=checkpoint
    )
    # No fit() on `fresh` -- this is exactly the zero-shot transfer path:
    # load pretrained weights, score immediately.
    scores_after = fresh.score(windows[15])
    # `fresh` has not accumulated the running host-stats history that
    # `det` built up over windows[:15], so scores will legitimately
    # differ in general (different _HostStats) -- but feeding the same
    # fresh host-stats state should reproduce `det`'s own first-window
    # behavior exactly, which is what this test actually checks: replay
    # the SAME fit history on `fresh` and confirm bit-for-bit identical
    # scoring, proving the checkpoint carries all the model weights
    # (nothing silently left out of export_state_dict).
    fresh2 = GraphFoundationDetector(
        epochs_per_fit=2, hidden_dim=8, seed=1, pretrained_state_dict=checkpoint
    )
    fresh2._stats = det._stats
    fresh2._edge_counts = dict(det._edge_counts)
    fresh2._window_idx = det._window_idx
    scores_replayed = fresh2.score(windows[15])
    np.testing.assert_allclose(scores_before, scores_replayed, rtol=1e-6)
    assert np.all(np.isfinite(scores_after))


def test_freeze_encoder_prevents_encoder_weights_from_changing():
    windows = _windows()
    pretrainer = GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=2)
    pretrainer.fit(windows[:10], warm_start=False)
    checkpoint = pretrainer.export_state_dict()
    encoder_before = {k: v.clone() for k, v in checkpoint["encoder"].items()}

    frozen_det = GraphFoundationDetector(
        epochs_per_fit=3, hidden_dim=8, seed=2,
        pretrained_state_dict=checkpoint, freeze_encoder=True,
    )
    frozen_det.fit(windows[10:20], warm_start=True)
    encoder_after = frozen_det._encoder.state_dict()
    for k in encoder_before:
        assert torch.equal(encoder_before[k], encoder_after[k]), (
            f"encoder parameter {k} changed despite freeze_encoder=True"
        )


def test_observe_updates_stats_without_touching_weights():
    """`observe()` is the zero-shot-evaluation primitive: it must grow
    the running host-stats/edge-count state exactly like `fit()` does,
    but must never take a gradient step -- the whole point is "let the
    detector build local history on a new network without training on
    it"."""
    windows = _windows()
    det = GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=0)
    weights_before = {k: v.clone() for k, v in det._encoder.state_dict().items()}

    det.observe(windows[0])
    det.observe(windows[1])

    assert det.state_summary()["n_windows_fit"] == 2
    assert det.state_summary()["n_hosts_seen"] > 0
    weights_after = det._encoder.state_dict()
    for k in weights_before:
        assert torch.equal(weights_before[k], weights_after[k]), (
            f"observe() changed encoder parameter {k}, but it must only "
            "update running stats, never train"
        )


def test_pretrained_weights_transfer_to_a_disjoint_host_population():
    """The core cross-network-transfer claim, exercised in miniature
    without real OpTC data: pretrain on a "network A" whose hosts are
    named one way, then apply those exact weights (no further fit) to a
    "network B" whose hosts have a completely disjoint naming scheme --
    simulating LANL-style "C1234" names versus OpTC-style
    "SysClient0051" names -- and confirm scoring still produces finite,
    sensible output. This is the property that would be impossible for
    a per-host-ID-embedding model (the CyberGFM failure mode this
    project's literature review flags) but is exactly what the
    feature-based, ID-free design here is for.
    """
    windows_a = _windows(seed=10, n_hosts=10)  # "network A", hosts named by synth.generate
    pretrainer = GraphFoundationDetector(epochs_per_fit=2, hidden_dim=8, seed=5)
    pretrainer.fit(windows_a[:20], warm_start=False)
    checkpoint = pretrainer.export_state_dict()

    # Build "network B": same synthetic generator, different seed (so a
    # disjoint random host/user population), then relabel every host
    # with an OpTC-shaped name to make the disjointness explicit and
    # mechanical, not just "different random strings that happen to
    # also look like LANL hosts".
    windows_b_raw = _windows(seed=99, n_hosts=10)
    rename = {}

    def relabel(host: str) -> str:
        if host not in rename:
            rename[host] = f"SysClient{len(rename):04d}"
        return rename[host]

    windows_b = []
    for w in windows_b_raw:
        new_nodes = [relabel(h) for h in w.nodes]
        windows_b.append(
            WindowGraph(
                window_start=w.window_start,
                window_seconds=w.window_seconds,
                nodes=new_nodes,
                node_index={h: i for i, h in enumerate(new_nodes)},
                edges=list(w.edges),
                edge_weight=list(w.edge_weight),
                edge_is_malicious=list(w.edge_is_malicious),
                n_events=w.n_events,
            )
        )
    assert not (set(w for win in windows_a for w in win.nodes) & set(w for win in windows_b for w in win.nodes)), (
        "test setup bug: network A and B host names must be fully disjoint"
    )

    transferred = GraphFoundationDetector(
        epochs_per_fit=2, hidden_dim=8, seed=5, pretrained_state_dict=checkpoint
    )
    # Zero-shot: no fit() at all on network B, just score directly with
    # the pretrained-on-A weights and B's own (freshly initialized,
    # cold-start) running host stats.
    scores = transferred.score(windows_b[5])
    assert len(scores) == len(windows_b[5].edges)
    assert np.all(np.isfinite(scores)), "pretrained weights produced non-finite scores on a disjoint host population"


In [ ]:
%%writefile tests/test_optc.py
"""Tests for lmsafe.data.optc: the real OpTC FLOW-event reader.

The schema asserted here is not guessed -- it is confirmed against a real
241-line sample of ecar-bro/short (2019-09-25, one host-range partition)
obtained from the official OpTC Google Drive release and inspected
line-by-line. That real sample happens to contain only outbound
connections to a small set of *external* IPs (benign cert-check
beaconing, plus one Notepad++-update-hijack/C2 sequence -- no internal-
to-internal traffic at all), so it alone cannot exercise the
internal-to-internal host resolution path this reader's whole design
rests on. The tests below cover that path with small, realistically-
shaped synthetic FLOW lines (same field names/types as the real sample)
in addition to replaying real sample lines verbatim for the parts real
data *does* cover.
"""
import gzip
import json
from pathlib import Path

from lmsafe.data import optc
from lmsafe.data.optc import OptcConfig, _parse_optc_timestamp, build_ip_host_table, stream_optc_events

FIXTURES_DIR = Path(__file__).parent / "fixtures"
REAL_SAMPLE_FILE = FIXTURES_DIR / "optc_ecarbro_short_sample.json"


def _flow_line(
    hostname,
    direction,
    src_ip,
    dest_ip,
    dest_port="80",
    principal="SYSTEMIACOM\\Administrator",
    timestamp="2019-09-25T09:05:14.574-04:00",
):
    """Build one realistically-shaped FLOW JSON line, matching the real
    sample's confirmed field names exactly."""
    return json.dumps(
        {
            "timestamp": timestamp,
            "id": "98d29eef-5d3f-409c-b326-0cea2df109e7",
            "hostname": hostname,
            "objectID": "49e8ff2d-0c89-494e-92cc-b67981b7333e",
            "object": "FLOW",
            "action": "INFO",
            "actorID": "f247d0d1-6ce4-4f89-aab8-7ebd7fba2ce6",
            "pid": 2984,
            "ppid": 2820,
            "tid": -1,
            "principal": principal,
            "properties": {
                "acuity_level": "1",
                "bro_uid": "CxSHfc16kZy97tnun6",
                "dest_ip": dest_ip,
                "dest_port": dest_port,
                "direction": direction,
                "image_path": "powershell.exe",
                "l4protocol": "6",
                "src_ip": src_ip,
                "src_port": "55398",
            },
        }
    )


# ---------------------------------------------------------------------
# Timestamp parsing
# ---------------------------------------------------------------------


def test_parse_optc_timestamp_handles_variable_fraction_digits():
    """Real OpTC timestamps carry 0, 2, or 3+ fractional digits (all
    three shapes confirmed in the real sample) -- the hand-rolled parser
    exists specifically because `datetime.fromisoformat` is not reliably
    lenient about this across Python versions."""
    no_frac = _parse_optc_timestamp("2019-09-25T09:56:52-04:00")
    two_digit = _parse_optc_timestamp("2019-09-25T09:07:45.26-04:00")
    three_digit = _parse_optc_timestamp("2019-09-25T09:05:14.574-04:00")
    assert isinstance(no_frac, float) and isinstance(two_digit, float) and isinstance(three_digit, float)

    # ordering within one real, verbatim sequence from the sample must survive parsing
    t1 = _parse_optc_timestamp("2019-09-25T09:07:15.106-04:00")
    t2 = _parse_optc_timestamp("2019-09-25T09:07:45.26-04:00")
    t3 = _parse_optc_timestamp("2019-09-25T09:08:15.386-04:00")
    assert t1 < t2 < t3


def test_parse_optc_timestamp_rejects_garbage():
    import pytest

    with pytest.raises(ValueError):
        _parse_optc_timestamp("not a timestamp")


# ---------------------------------------------------------------------
# Real sample lines, replayed verbatim
# ---------------------------------------------------------------------

# Two consecutive lines taken verbatim from the real ecar-bro/short sample.
_REAL_LINE_1 = (
    '{"timestamp":"2019-09-25T09:05:14.574-04:00","id":"98d29eef-5d3f-409c-b326-0cea2df109e7",'
    '"hostname":"SysClient0358.systemia.com","objectID":"49e8ff2d-0c89-494e-92cc-b67981b7333e",'
    '"object":"FLOW","action":"INFO","actorID":"f247d0d1-6ce4-4f89-aab8-7ebd7fba2ce6","pid":2984,'
    '"ppid":2820,"tid":-1,"principal":"SYSTEMIACOM\\\\Administrator","properties":{"acuity_level":"1",'
    '"bro_uid":"CxSHfc16kZy97tnun6","dest_ip":"202.6.172.98","dest_port":"443","direction":"outbound",'
    '"image_path":"powershell.exe","l4protocol":"6","src_ip":"142.20.57.103","src_port":"55398"}}'
)
_REAL_LINE_NOT_FLOW = (
    '{"timestamp":"2019-09-25T09:05:14.574-04:00","id":"x","hostname":"SysClient0358.systemia.com",'
    '"objectID":"y","object":"PROCESS","action":"CREATE","actorID":"z","pid":1,"ppid":2,"tid":-1,'
    '"principal":"SYSTEM","properties":{}}'
)


def test_real_sample_line_parses_and_is_dropped_as_external_only(tmp_path):
    """The real sample's dominant pattern -- outbound to an external IP
    with no matching internal host -- must be dropped under the default
    `only_internal_internal=True`, the OpTC-equivalent of LANL's
    same-host/remote-logon fairness filter."""
    p = tmp_path / "sample.json"
    p.write_text(_REAL_LINE_1 + "\n" + _REAL_LINE_NOT_FLOW + "\n")

    records = list(optc._iter_flow_records([p]))
    assert len(records) == 1, "the PROCESS-object line must not be parsed as a FLOW record"
    assert records[0].hostname == "SysClient0358.systemia.com"
    assert records[0].src_ip == "142.20.57.103"
    assert records[0].dest_ip == "202.6.172.98"

    events = stream_optc_events([p], OptcConfig())
    assert events == [], "an outbound flow to an unresolvable (external) IP must be dropped by default"


def test_malformed_json_line_is_skipped_not_raised(tmp_path):
    p = tmp_path / "sample.json"
    p.write_text(_REAL_LINE_1 + "\n{not valid json\n")
    records = list(optc._iter_flow_records([p]))
    assert len(records) == 1


# ---------------------------------------------------------------------
# Internal-to-internal resolution (the path real data didn't exercise)
# ---------------------------------------------------------------------


def test_ip_host_table_built_purely_from_self_observation():
    lines = [
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "8.8.8.8"),
        _flow_line("SysClient0150.systemia.com", "inbound", "203.0.113.5", "142.20.57.50"),
    ]
    import tempfile
    from pathlib import Path as _P

    with tempfile.TemporaryDirectory() as d:
        p = _P(d) / "s.json"
        p.write_text("\n".join(lines) + "\n")
        records = list(optc._iter_flow_records([p]))
    table = build_ip_host_table(records)
    assert table["142.20.57.10"] == "SysClient0100.systemia.com"
    assert table["142.20.57.50"] == "SysClient0150.systemia.com"
    # the external IPs (8.8.8.8, 203.0.113.5) must never appear: there is
    # no self-observation that would attribute them to any host.
    assert "8.8.8.8" not in table
    assert "203.0.113.5" not in table


def test_outbound_internal_to_internal_flow_produces_correct_event(tmp_path):
    """SysClient0100 -> SysClient0150 on port 445 (SMB, a classic
    lateral-movement-relevant port) must become one Event with the
    source/destination hosts in the right order."""
    lines = [
        # self-observation: pins 142.20.57.50 to SysClient0150
        _flow_line("SysClient0150.systemia.com", "outbound", "142.20.57.50", "8.8.8.8", dest_port="80"),
        # the flow under test
        _flow_line(
            "SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.50",
            dest_port="445", principal="SYSTEMIACOM\\attacker",
        ),
    ]
    p = tmp_path / "sample.json"
    p.write_text("\n".join(lines) + "\n")

    events = stream_optc_events([p], OptcConfig())
    assert len(events) == 1
    e = events[0]
    assert e.src_comp == "SysClient0100.systemia.com"
    assert e.dst_comp == "SysClient0150.systemia.com"
    assert e.auth_type == "445"
    assert e.logon_type == "outbound"
    assert e.orientation == "LogOn"
    assert e.success is True
    assert e.is_malicious is False
    assert e.source == "optc"
    assert e.src_user == "SYSTEMIACOM\\attacker"
    assert e.dst_user == e.src_user
    assert e.is_remote_logon(), "an emitted OpTC event must satisfy the shared is_remote_logon() check unchanged"


def test_inbound_internal_to_internal_flow_swaps_src_dst_correctly(tmp_path):
    """When the *reporting* host's record has direction="inbound", that
    host is the destination, not the source -- getting this backwards
    would silently reverse every lateral-movement edge's direction."""
    lines = [
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "8.8.8.8"),
        # SysClient0150 reports *receiving* a connection from 142.20.57.10
        _flow_line("SysClient0150.systemia.com", "inbound", "142.20.57.10", "142.20.57.50", dest_port="3389"),
    ]
    p = tmp_path / "sample.json"
    p.write_text("\n".join(lines) + "\n")

    events = stream_optc_events([p], OptcConfig())
    assert len(events) == 1
    e = events[0]
    assert e.src_comp == "SysClient0100.systemia.com"
    assert e.dst_comp == "SysClient0150.systemia.com"
    assert e.auth_type == "3389"


def test_self_loop_is_dropped():
    """A host whose resolved source and destination are the same host
    (e.g. loopback-ish logging artifacts) is not a lateral-movement edge
    and must not be emitted."""
    import tempfile
    from pathlib import Path as _P

    lines = [
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.10"),
    ]
    with tempfile.TemporaryDirectory() as d:
        p = _P(d) / "s.json"
        p.write_text("\n".join(lines) + "\n")
        events = stream_optc_events([p], OptcConfig())
    assert events == []


def test_only_internal_internal_false_keeps_raw_ip_as_pseudo_host(tmp_path):
    lines = [
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "203.0.113.9", dest_port="443"),
    ]
    p = tmp_path / "sample.json"
    p.write_text("\n".join(lines) + "\n")

    events = stream_optc_events([p], OptcConfig(only_internal_internal=False))
    assert len(events) == 1
    assert events[0].src_comp == "SysClient0100.systemia.com"
    assert events[0].dst_comp == "203.0.113.9"


def test_t_is_relative_to_earliest_event_in_the_batch(tmp_path):
    lines = [
        _flow_line(
            "SysClient0150.systemia.com", "outbound", "142.20.57.50", "8.8.8.8",
            timestamp="2019-09-25T09:00:00-04:00",
        ),
        _flow_line(
            "SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.50", dest_port="445",
            timestamp="2019-09-25T09:00:30-04:00",
        ),
    ]
    p = tmp_path / "sample.json"
    p.write_text("\n".join(lines) + "\n")

    events = stream_optc_events([p], OptcConfig())
    assert len(events) == 1
    assert events[0].t == 30


# ---------------------------------------------------------------------
# gzip auto-detection
# ---------------------------------------------------------------------


def test_gzip_compressed_file_is_read_transparently(tmp_path):
    lines = [
        _flow_line("SysClient0150.systemia.com", "outbound", "142.20.57.50", "8.8.8.8"),
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.50", dest_port="445"),
    ]
    p = tmp_path / "sample.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")

    events = stream_optc_events([p], OptcConfig())
    assert len(events) == 1
    assert events[0].src_comp == "SysClient0100.systemia.com"


def test_empty_file_list_returns_empty():
    assert stream_optc_events([], OptcConfig()) == []


# ---------------------------------------------------------------------
# The actual real sample file bundled with this project (not a single
# line copy-pasted into this test file, but the full 241-line file the
# user obtained from their own ecar-bro/short Drive folder and that
# lmsafe_cache/README.md's "Real OpTC support" section and the Colab
# notebook both now run the real reader against end to end).
# ---------------------------------------------------------------------


def test_real_bundled_sample_file_is_parsed_and_yields_zero_internal_flows():
    """Pins down, against the actual bundled file (not a synthetic
    stand-in), the honest finding this reader was built against: all
    240 FLOW records in this specific ecar-bro/short partition are
    outbound to external IPs, so the default (fair) filter correctly
    yields zero events. If this ever starts returning non-zero events,
    either the file changed or the filter regressed -- both worth
    knowing immediately."""
    assert REAL_SAMPLE_FILE.exists(), "the real OpTC sample fixture is missing"

    records = list(optc._iter_flow_records([REAL_SAMPLE_FILE]))
    assert len(records) == 240

    events_default = stream_optc_events([REAL_SAMPLE_FILE], OptcConfig())
    assert events_default == [], "this real sample is known to contain zero internal-to-internal flows"

    events_keep_external = stream_optc_events(
        [REAL_SAMPLE_FILE], OptcConfig(only_internal_internal=False)
    )
    assert len(events_keep_external) == 240, "with the fairness filter off, every FLOW record becomes an event"
    assert all(e.source == "optc" and e.orientation == "LogOn" for e in events_keep_external)


# ---------------------------------------------------------------------
# Integration with the dataset-agnostic windows/split pipeline
# ---------------------------------------------------------------------


def test_events_feed_iter_windows_and_split_train_test_unchanged(tmp_path):
    """The whole point of the Event boundary: nothing downstream needs
    to know these events came from OpTC. Build a few internal-internal
    flows spread across more than one window and confirm `iter_windows`
    and `lanl.split_train_test` (duck-typed on `cfg.train_cutoff_s`)
    both work on OpTC events exactly as they do on LANL/synthetic ones.
    """
    from lmsafe.data import lanl
    from lmsafe.data.windows import iter_windows

    lines = [
        _flow_line("SysClient0150.systemia.com", "outbound", "142.20.57.50", "8.8.8.8",
                   timestamp="2019-09-25T09:00:00-04:00"),
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.50", dest_port="445",
                   timestamp="2019-09-25T09:00:30-04:00"),
        _flow_line("SysClient0100.systemia.com", "outbound", "142.20.57.10", "142.20.57.50", dest_port="445",
                   timestamp="2019-09-25T11:00:30-04:00"),
    ]
    p = tmp_path / "sample.json"
    p.write_text("\n".join(lines) + "\n")

    cfg = OptcConfig(window_seconds=3600, train_cutoff_s=1800)
    events = stream_optc_events([p], cfg)
    assert len(events) == 2

    train, test_iter = lanl.split_train_test(iter(events), cfg)
    assert len(train) == 1, "only the t=30 event is before the 1800s cutoff"
    test = list(test_iter)
    assert len(test) == 1

    windows = list(iter_windows(events, cfg.window_seconds))
    assert len(windows) == 2, "the two flows are two hours apart -- different 1-hour windows"
    assert windows[0].nodes == ["SysClient0100.systemia.com", "SysClient0150.systemia.com"]


In [ ]:
REDTEAM_FIXTURE_b64 = """
H4sIAOrXUFUAA82cSdIdtxGE9zwLF5gLtWOEvHV4xftfxZWJ7vdLjURLYVmWNyLF72Eu1ASgc09z9u8/R0k//vGvf+bvv2QbXuOPlOq33HOq4/tPa/O3tKYO
ONoU0MoE9Pj7z5Fzf1ScDRVX8xK4jt/S4i1gL9lFxT3+C1pd0dwLYBtJwFJbBu2e1XAqio7R1XBamelbifmYv52JmUqPP1r0KWh0/IXWXM50tqza7al9K8VS
L2qwbQK2pEpesBc1Exe0eYbRu3aGub+UHC16m63U5/RnUkzwiXpq40hrqt4VpZzG6o5hhzkMgciuoJUMaFmJxAVdysuCsXByafqwoLWanET0t8U0NbWvrrKt
dFn2plLEb9pbTJRVu2nsw/L9l5FLwJFdwdZR0ronOcU2UvrWrJSmly/2OvHUa79wbAGJcwjqtzZr63J9oRCC+sxn2r28lB2t1YOmaXOm2l565SO/YM/W7NRw
t9nqFJOd0+jf+ozxxpBmeajHMoaTjnZYxVF7ymoVe/FvFpIzsUqjP/oU+we0T1eUzQb2GXNV/Fk29HXQEIOoenp7TGUeORF3eymcK+YqheL/8UsNDf/1g/gf
/KCkt7Yrplq0nUsjnm9th3SqwjVmDtjtrWWHgVQtW7HglrAdZ2iZh5Lo6JjlGstccnkK0ABssHMnON6gJTki99aJXa7Ujed8wzNE9PvP5rbp+FDywPW19Ewx
H9VTlzj+PSp3f7oFKcd01dSqnMySZmiR4B2m6YVPyPfO8xyYt5qjnVfe4JbEGJ6TXkrQsF+kXztv4Ymmx4QLsMOwjBhZ/H3Iss1j81RLMEO1+UMGo6ek0I6i
bmPdIYL+iu29tMvSVNyB82HX5+wsXjGle3EqnGohxrJ0KAsDd9YuFmQapnxWTvlzb4amQulJ473TMNygnjHl/eHalpjyoJ5c1Vwbl9oLNP9BJ1Sv2F8bbSxp
2D05lOVD/hekvdlg/MuMvdVSTpKjcEsON3KTkTkHIdy9Awypl9sqljYbeIbpb/VpFsKnrmg599ZU5RDO0GEWixBmeFviHBIQGxpbOuKMbV9Am7SaMSM7rt1B
S6mKRoAwKnlYvnPplt5LN4jPC+9J1h46eC7sunEWthqNu42nCC3qoysa4QAUYWsZbpzq2sULfYBdjabUibssvhoPlVOgg9uz8uQD4tINbuLOJ0t3x5Y+0RGe
+hu1qagX0kyfaD5NXngNWM7wfKrCIw9SuADTzH780mf94Pg7aMeQjmVHld2KuAJ0wsKr6QqXJbglaMBYsOemLkvEjWv5Uj6P+lo+gqPzyKy0z7gjNku/2rzh
toN3f629zpexQxgUzc2cGOu5Gq+o6Cukpj6z0d+H5pCWCJWetQ9olpmantgMy9RmuOHQ7/bcBB6xEHhJv8PhForqZ86Y2Qg77KVzrczX0jMNOXM+0+JaqiKo
KNjCnpLmV/lwpNorn7B+Yb4fvowPGhOf0Kx7cTis36yH0m5q7sJelIUxNe2xnQBA6deeaAQC9Tfhy01h0sPnyFPQ6EnISw/LHoMa3R5Vw4fqiDFU0aUye5k0
+Ls4NHgivRYkNo4auddaadcfLYccxGr1sL+K1oLJRlIqq6avMYfTCecvPwStOBvuBYKwDSt7hRT3iNqbaDkigIARv7ooWxN6FXFrVxVnyF+PnT9U0TBsoAMx
93gucYRaaNasv9BJJ3+reXTsmwhQs+xVSAYaDoUBsQ5bsDmzKaYjPCc4u4f4N5YB/sJOGawFbTJ2tlVzHpIiQQWq212RNXJQutujDQOnFt62TEwYacOWOdLh
TVCqofAruVO3hiOMw6ByKqrd1ichtfNeNo+B+cr03QRPsxPLXtc+2TJDOTEj3hoqD6mXjTs81hHujxzWRXvVsx2OWyHHUr7weRqYZ2IYdFG8U1CqIYuiau8Z
tTfG13vUQjlqGX78BpcItlxVpmLOVW1D/i6s5pOueqekhYYaCUlXEQ1bLVNFWRPuB2BT9YaJZrM1y2b76CzdUhdVj0rYmTIUVa+W+5DB36T4hN8iG84RkRIz
n7zhRsGOPanmubfVaesqcmx5cMSe1AJSuQUsLntF1upLj3vKSmpyQZQVutPURK5Yd0QYl+XiJwpkL6bCWeRkQZlRFMPtrDp6J8xPBCLstK1s5NNcs9Up155B
ZcBsstrCos79qQ1m0KFW4AojxqAN2LHBsAXtsrAhxxl0qLTA2roj6+wpLU/IuzLTYU4XlfN0U87ieGYvLzhxlPCEFMUxDdIWfZZqbMUtW7UJ02BUFxsbSJgC
2jHlE9Rlumnlk4aV9JKNClzeMTXgIY0GPM9JuMBNJ+FWx+QsGzdBQDpS/pjKRlG2uly0J83mnMxaXnGDOdmzL2sn2IRX++xWcWo5m0MWvefakbcReZ0wUtie
MzFXlbazpBI9JseJ3Uv5kFjJZ+ZyzFJ0XunmDAVE98OxQP9ng2+7jT18KcCOxVSdi8iKvIuNUamX5qii4g+Lcr3052rdkzJO7V6TMsb7pI8p1yz0PBX9NJwr
PvuW14KH0S0ifV5xqg7aTNDYcbDxnmBP9yzg6panYUoWwoRc2NWs3IVdzGd0CrvDGYSca65djTccU8BGy7WXpcaNOUmKLm/J25BlbRl57/CMt3YnUlHDR9E7
IwLqxatQFoakyHDrSq/i0kT8ddKp2QeUlrV1r2K7h0qNmQyFrUR6uXA4QViB03P1qQsCVxVYrxDUQjSKin5zOPGsnNm1vfLc4H5Y6khNh+P+nK67OAPgvTil
J6iasDDkYFaGKmk2OSPMcjyL+kKW5HzEVmKrODWVQ6qkXRhAQ4rfMt1wUZKHSLbCIjEZHSsc8tNkooG52aBT0pxgpAK7zCrHFscyRgiQZWmmXgLDxVQ9u/CQ
uCGRaOHedkUv+coDO1XVDVUOsy7keoUBOCbtZziH0tXIb+A4EQb9meNaRjsoFmKjJRl75EPR7JTJcC4xF3k7f42wlrhLHC2jX7VA9sQRasmZtXM7qSPWyuJM
sesN0aRntvzTEM+s9DDT+nAzhKK9tEfjyefmgd60ZzmToDMU2yNvUkqL4J1Zlb+TYgefqb2W/evofO3VK122479OcxqjKgNecbMD1IrYn0gsECpfKjzusug8
maygkzfjtk0U4kroKgYJf5hUmu929XhK+wz9TOhiI9zQlKK64cxq8yHXQYpdvY+GQ/VUxRwyTgbleaAMPnIKu5/0PSSoC/Kir259cXm37ouPIXX3h9s7z4h/
hMv8xcVaQYUTNp2s+xSeh7z9xQsmR2jqm7dWRBqSZouYmdkTzrz1Jk0TaMf1hfxMjdS0IDezFIewh0iCP0vWMm8qJswLWYXLkyMg1T2OoidjCtqqnM1P6YaQ
8aX27soSfDo+kHsJ82c/WGQvP4bYe5/iltPLYhSemZ2xq3ARKTBCWD8cIxywNqvVLupCiyF9RorrILjqcho0f7AXN9LKGzYiACK0JFqO+OGisDWxP9uh5UqF
9dK1ylMx4dXhBjF/wNtxWhRbMulutEaaVWbqntIIHF7mpHHTC0dm1czbwIJmUgYjm2Uo9YIu5PcaTy9vEB7hEcJWbfDedc2acOjoapJ2HT2vieJtjYMFDVqU
ao0oYtGuzEro2kWVZ33X3Gk0YkFOWCZ+vzAPRHN76pmbZ3iUbT9MuzHviuvjCvDaxMBaLwuqMJn3kUhN7PRSslG2esewvewB2FV8JKVeQyQvKrbrh1oRYoDz
INKJo5TnS4zLXnRXnv0HKt8i5qgu+mKHRlH+2eW9BVUbbTl+EWRmMZpQimx1UNGfV3AwzXUUgKHPFT54Yo8fhdOSUqlfVM3lr+hr2fJalk9i+rNfbY6F6Uc9
38TclAd4e0BqdGkRYbsonLstXHCSq4Nd4q4i5U/Hiyvldmsow8Wil90c+1Xu9luHxabT5addfJ1UPHlNy0rZure+17/2JERcF1+1W9O66G6dGYYX7oyN9+r5
linZejCzrcwX5lFkH8/LvdfgJi/gngc/6R/sC+9LbczMpxN95lNxZHWGPU+BvrgpC3LTvp51PP3Bm7upBNxHm7oVtZkCs+8+qyjdl9H0qfIVIXFzUXvRPj6n
SqPiYgKpC3qpNZdeZFm61F0Nt7V00SKoMc6ZiRdht7HmnElzE6vQucS4BydmsRVfkG/PNp2TR0ur6i7ncUXKIXyqzznMN2julfbMnzeHl2KA7CXFL7UTXL/b
utxPHCC9cyumwpH4o7H7mdfs1A1D2rfgOD57q8DXRYFTBXhaoCpgpo/cD7c/r/I563uCH16ELH5qL00JY2HWZo5e5EUUdGpxfYHnwwfDhy1jy40QFD7tiRo9
tJqe7zwulR8cKlvwS3Jsmiy//PRpXiS+BWs2RMjn6p0vPV54hdwozuBlOq8QIdMvY77gbuoR1ly1D9w8PzqRwZmTEKO7anfE8MfynhLs5QvvaF8f9RFP+Qxm
PaPhD1y+o/n6AS+yvPyg0KrsI6yjLN44/+H6XPzKR+ItYuA6cENLZJziZ+RWqiw+VnG60i/FeW5w5q1BKZRSnza1MFsQHKu/8yvn5J1XKyUv5B02+zw7ETdK
vuxBYL13Vi4j8JSy7aMv7HJpSluVG+yf2Jh0hgIXua+voDJ4lY2PuYpPpFfFvqtLbjpNsxjaUlmx7ZLs/M1HSnpmK000znUlv9RWbExeEdnyY33kufjQk7f0
UnCkRPaV/yrPS0JHPl3PfhmJkunrfuuml+7p9/V2fvcGLs3gzCy+8EFPu7s62ALnk6ud58qIMTgN+sZLGkt+1r10UcHEsXzGCxYcJIoYyuBOg5vkawTBMyTo
hReGw/sI/cIofsQhp7J2xnC40zp4I+H5rrdCLwQezK6ecGUgseOcGmuvy5c4czrjZ772rmgelwByxvNwuflsLjql0ULSBdQlzbhsHrwkLdh46UfucGNPJg2n
y3yj88KHSuswQAW1ow8clPdIjiIzelIHRnfp3mYSrkB4tqSu/LvcCYe8c3Z3elR1QY9uP2BRp2PZ2KfBgFkcmizKKAY9jn94MMvYwGFXFeL7LI28nlBXkQ3d
vIBDJuOoTEF5LVI4YRc3nnud+eTLrI0vJzC4y1PPVCmREberC0I9se7JMy+VF88478rHw9Y/jem8/W24/uc4N+2XXCrASuO90rA0m/rCB2cy3p7rxwzFcGSc
cbNLWu9pPYP3xAty7bnRnaXHGFk9eZls3EZVz4tq87wwdGd5Jllu6lOHK0zu4fEKpuZZOmz1JOUNuidtXjmts+jXCsslAh9Kju+W+fLkWfeK8jKCNB3lzas0
Dxh2rTwLO+48+N1zU8zbAZcsMU/FgzOnuWfOQmNzn3nLb3rbonPQvX1LrHGDm1NJbvieGR/q1sNyhEBNXofyu+8+GCEpL2Kud94nOJEzqf353JqhESYe51A7
ZugZmNe0BKbZmJliqvBcGHbyjHkaueNl65DrOLTN0i0hmXMYdeNnndDC06JVYl6SPmPeBjlVzXvwEpZae9HfN/AEp+yP/yBMwo/1j3/xT/58V/+vf9Byhbel
P40TOLyjepSE0mrPDECSdKrAeRdm43RIgHlrbMPL2QPnpeKNw/AHHYXRwXbTf/TB6tu4Dkp07es7Vhu++94Gs6lPvBRWwXspnlNsQ+uwv8HrOibZpiZ75tRe
eYU9+E2VDYxC1VOeFcwOExm8Izjkc3DZwbGegx/63xFdKfWx/APw+jucsbnWjIG7yeLUq3gy6LL0hdtKDJRtaKSd3yc6FK7ht7j6FhSzVRWvDdNBPwVs6Q2u
RJ6C+D5fvjUK3hx8lPW88HjFcyncAw5VD/fXnkvhuHhQccaezh8GWD8Y7z9gCvTlB53fJnz7Qf5qYv8yAn4w4H7sLlvlCMd603Aa4aQawieFHooAAhGYT1eO
2JnfPKQRgOeLY4Nwnbd25EFyxaPvoTRkLtCgwfnevO8+HyAeIpzh4PeXYnBPpwhzEl4Z71jVp34wOMGB27nX4VWVMk57ZDYcRIpA0eB4gMo4+qZe1A2aRWOk
/FrBdj7qBovgNU/50Tx8i60UH+vhpfpMF2gt6lktPzIE3Jr+XBW/yxM/MH4y8fDxHHI/f6qrhNueO1/nDuwRnFh8eoCMT/wgPOl0TESDz5pPN+7wVcbUZDKY
5xjgpZxPacB5xVdla/HRzeCd2dy2GRyENuRdtv/h6/3p6atz+AVfiejvkQVufKR6+mATf+DqsU+5KLO5py8M8QddfkOIp4PBO08fD9/5AedX89TdHsDKdKK4
XQ3Y+/mVCrgN+VBpPYOLH4x1Y1J9iyOo5WEiWuZ3PEodrTB6OgnuiHEdv3kTGDcPD89TC76lpo60eRcPlF/X1J91CYznQfr+N2BXL5UuWEc9XzOB1+pTfdQR
gVnBt494oqW/1/IH+OQ3vOS3PeHb8MBQ5ihjTIkfZxSfqpyk9HxOdD05OtL+Tmc909gCdsot/G/wlhr4m6kxfSXn698sL6IWylkAAA==
"""
import base64, pathlib
pathlib.Path("tests/fixtures/redteam.txt.gz").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("tests/fixtures/redteam.txt.gz").write_bytes(base64.b64decode(REDTEAM_FIXTURE_b64))
print("wrote tests/fixtures/redteam.txt.gz", pathlib.Path("tests/fixtures/redteam.txt.gz").stat().st_size, "bytes")

In [ ]:
OPTC_SAMPLE_FIXTURE_b64 = """
...
"""
import base64, pathlib
pathlib.Path("tests/fixtures/optc_ecarbro_short_sample.json").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("tests/fixtures/optc_ecarbro_short_sample.json").write_bytes(base64.b64decode(OPTC_SAMPLE_FIXTURE_b64))
print("wrote tests/fixtures/optc_ecarbro_short_sample.json", pathlib.Path("tests/fixtures/optc_ecarbro_short_sample.json").stat().st_size, "bytes")

In [ ]:
!python -m pytest tests/ -q

## 7) Write the docs (`docs/`, `README.md`)

In [ ]:
%%writefile docs/make_architecture_diagram.py
#!/usr/bin/env python3
"""
Generates `docs/architecture.png` -- the pipeline figure referenced from
the top-level README.

Kept as a small, versioned script (rather than a hand-drawn image file
with no provenance) so the figure can be regenerated or tweaked the
same way the rest of this project is built: run it, read a diff, move
on. It has no runtime dependency on `lmsafe` itself -- it only draws
boxes and arrows describing the package's own data flow -- so it is
safe to run in any environment with matplotlib installed:

    python docs/make_architecture_diagram.py

Layout (top to bottom):
  1. Data sources (LANL / OpTC / synthetic)      -> a common `Event` stream
  2. Fair protocol + windowing                   -> train/test `WindowGraph`s
  3. The experiment loop: Detector <-> UpdatePolicy, one of five policies
  4. The boiling-frog conditioning attack feeding into that loop (RQ2/RQ3)
  5. Metrics -> the three research questions (RQ1/RQ2/RQ3)
"""
from __future__ import annotations

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.lines import Line2D

# -- palette: muted, print-safe, colorblind-conscious -----------------
INK = "#1f2430"
BG = "#ffffff"
BLUE = "#3B6EA5"      # data / sources
TEAL = "#2E8B8B"      # processing / windows
PURPLE = "#6B5B95"    # the core loop
AMBER = "#C97A2B"     # attack / adversarial
GREEN = "#4C8C4A"     # metrics / RQs
GREY = "#8A8F98"      # connective text / neutral boxes


def box(ax, xy, w, h, text, face, edge=INK, fontsize=10.5, fontweight="normal", text_color="white"):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.08",
        linewidth=1.3,
        facecolor=face,
        edgecolor=edge,
        mutation_aspect=1,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
             fontsize=fontsize, color=text_color, fontweight=fontweight, linespacing=1.35)
    return patch


def arrow(ax, p0, p1, color=INK, style="-|>", lw=1.6, connectionstyle="arc3,rad=0.0", ls="solid"):
    a = FancyArrowPatch(p0, p1, arrowstyle=style, mutation_scale=14,
                         linewidth=lw, color=color, connectionstyle=connectionstyle, linestyle=ls)
    ax.add_patch(a)


def main():
    fig, ax = plt.subplots(figsize=(13, 15.5), dpi=200)
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 15.5)
    ax.axis("off")
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    ax.text(6.5, 15.1, "lmsafe -- experimental pipeline", ha="center", va="center",
             fontsize=18, fontweight="bold", color=INK)
    ax.text(6.5, 14.65,
             "Reliable Graph Foundation Models for Lateral Movement Detection:\n"
             "Cross-Network Transfer under Drift and Contaminated Baselines",
             ha="center", va="center", fontsize=10.5, color=GREY, style="italic")

    # ---------------------------------------------------------------
    # Row 1: data sources -> lmsafe.data.schema.Event
    # ---------------------------------------------------------------
    y1 = 13.0
    box(ax, (0.4, y1), 3.1, 1.0, "LANL auth.txt.gz\n(streamed, never\ndownloaded whole)\nlmsafe.data.lanl", BLUE)
    box(ax, (4.2, y1), 3.1, 1.0, "OpTC\n(documented stub)\nlmsafe.data.optc", GREY, text_color="white")
    box(ax, (8.0, y1), 3.1, 1.0, "Synthetic generator\n(Zipf traffic, drift,\nLM episodes)\nlmsafe.synth", BLUE)

    box(ax, (3.5, y1 - 1.55), 5.6, 0.9, "Event  (src_user, dst_user, src_comp, dst_comp,\nt, logon_type, is_malicious*)\nlmsafe.data.schema", TEAL)
    arrow(ax, (1.95, y1), (5.3, y1 - 0.65))
    arrow(ax, (5.75, y1), (6.3, y1 - 0.65))
    arrow(ax, (9.55, y1), (7.5, y1 - 0.65))
    ax.text(12.6, y1 + 0.5, "*is_malicious:\nevaluation-only,\nnever read by a\nDetector/Policy", fontsize=8, color=GREY, ha="right", va="center", style="italic")

    legend_elems = [
        Line2D([0], [0], color=INK, lw=1.8, label="benign data flow"),
        Line2D([0], [0], color=AMBER, lw=1.8, linestyle="dashed", label="adversarial / gated-\ndefense path (RQ2, RQ3)"),
        Line2D([0], [0], color=GREY, lw=1.6, linestyle="dashed", label="drift-detector signal\n(DriftTriggered only)"),
    ]
    leg = ax.legend(handles=legend_elems, loc="center", bbox_to_anchor=(0.925, 0.665),
                     bbox_transform=fig.transFigure, ncol=1, frameon=True, framealpha=0.9,
                     fontsize=7.6, labelcolor=INK, title="Arrow key", title_fontsize=8.2,
                     edgecolor=GREY, handlelength=1.8, labelspacing=1.1)
    ax.add_artist(leg)

    # ---------------------------------------------------------------
    # Row 2: fair protocol + windowing -> train/test split
    # ---------------------------------------------------------------
    y2 = 10.1
    box(ax, (3.5, y2), 5.6, 0.9,
        "Fair evaluation protocol: keep all remote LogOn events,\n"
        "label-join on (t, dst_user, src_comp, dst_comp)\nlmsafe.data.lanl.apply_fair_protocol",
        TEAL)
    arrow(ax, (6.3, y1 - 0.65), (6.3, y2 + 0.9))

    y3 = 8.75
    box(ax, (3.5, y3), 5.6, 0.75, "iter_windows()  ->  WindowGraph per time bucket\nlmsafe.data.windows", TEAL)
    arrow(ax, (6.3, y2), (6.3, y3 + 0.75))

    y4 = 7.55
    box(ax, (1.2, y4), 3.6, 0.8, "Train windows\n( t < 41 h )", TEAL, edge=INK)
    box(ax, (6.6, y4), 2.9, 0.8, "Test windows\n( t >= 41 h )", TEAL, edge=INK)
    arrow(ax, (4.8, y3), (3.0, y4 + 0.8))
    arrow(ax, (7.5, y3), (8.0, y4 + 0.8))

    # ---------------------------------------------------------------
    # Row 3: the attack module feeding into the test stream
    # ---------------------------------------------------------------
    box(ax, (10.0, y4), 2.7, 0.8,
        '"Boiling-frog"\nconditioning attack\n(RQ2/RQ3 only)\nlmsafe.attack', AMBER, edge=INK, fontsize=9)
    arrow(ax, (10.0, y4 + 0.4), (9.5, y4 + 0.4), color=AMBER, ls="dashed")

    # ---------------------------------------------------------------
    # Row 4: the core experiment loop
    # ---------------------------------------------------------------
    y5 = 5.55
    box(ax, (0.4, y5), 2.6, 0.75, "fit_initial()\n(benign 41h fit)", GREY, text_color="white")
    arrow(ax, (3.0, y4), (1.7, y5 + 0.75))

    loop_x, loop_y, loop_w, loop_h = 0.4, 2.55, 12.2, 2.65
    ax.add_patch(FancyBboxPatch((loop_x, loop_y), loop_w, loop_h,
                                 boxstyle="round,pad=0.03,rounding_size=0.12",
                                 linewidth=1.6, facecolor="#F3F0FA", edgecolor=PURPLE))
    ax.text(loop_x + 0.15, loop_y + loop_h - 0.25,
            "per-window loop, one (detector, policy) cell  --  lmsafe.runner.run_experiment_cell",
            fontsize=9.5, color=PURPLE, fontweight="bold")

    box(ax, (0.9, loop_y + 1.25), 3.0, 0.8, "Detector.score(window)\n(CountBaseline / FeatureMLP)", PURPLE)
    box(ax, (5.1, loop_y + 1.25), 3.0, 0.8, "WindowScores\n(scores + labels + weight)", TEAL)
    box(ax, (9.3, loop_y + 1.25), 2.6, 0.8, "metrics\ncollector", GREEN)
    arrow(ax, (3.9, loop_y + 1.65), (5.1, loop_y + 1.65))
    arrow(ax, (8.1, loop_y + 1.65), (9.3, loop_y + 1.65))

    box(ax, (0.9, loop_y + 0.15), 3.0, 0.8,
        "UpdatePolicy.observe(window, scores, detector)\n"
        "Frozen | Periodic | SlidingReplay |\nDriftTriggered | Gated", PURPLE, fontsize=9)
    arrow(ax, (0.75, loop_y + 0.55), (0.75, loop_y + 1.65), color=PURPLE, lw=1.4,
          connectionstyle="arc3,rad=-0.9")
    ax.text(0.15, loop_y + 1.1, "updated\ndetector\nfor next\nwindow", fontsize=7.3, color=GREY, ha="right", va="center")

    box(ax, (5.1, loop_y + 0.15), 3.0, 0.8,
        "GatedPolicy only:\nscore-gated admission ->\nquarantine -> multi-host\ncorroboration -> commit", AMBER, fontsize=8.3)
    arrow(ax, (3.9, loop_y + 0.55), (5.1, loop_y + 0.55), color=AMBER, ls="dashed")

    box(ax, (9.3, loop_y + 0.15), 2.6, 0.8, "Drift detectors:\nPageHinkley /\nSimplifiedADWIN\n(DriftTriggered only)", GREY, fontsize=8.3)
    arrow(ax, (9.3, loop_y + 0.55), (8.1, loop_y + 0.55), color=GREY, lw=1.2, ls="dashed")

    # ---------------------------------------------------------------
    # Row 5: metrics -> research questions
    # ---------------------------------------------------------------
    y6 = 1.55
    box(ax, (0.6, y6), 4.4, 0.75,
        "weighted AUC/AP, Rec@B, episode\nclustering + delay, block/episode\nbootstrap CIs  --  lmsafe.metrics", GREEN, fontsize=9)
    arrow(ax, (10.6, loop_y), (2.8, y6 + 0.75))

    box(ax, (0.3, 0.1), 3.9, 1.0,
        "RQ1\ndrift cost per policy,\nno attack\n(run_rq1_drift_cost.py)", GREEN, edge=INK)
    box(ax, (4.55, 0.1), 3.9, 1.0,
        "RQ2\nupdate-poisoning via\nconditioning attack\n(run_rq2_poisoning.py)", AMBER, edge=INK)
    box(ax, (8.8, 0.1), 3.9, 1.0,
        "RQ3\ndoes Gated win on\nBOTH axes at once?\n(run_rq3_gated_defense.py)", PURPLE, edge=INK)

    arrow(ax, (1.8, y6), (2.2, 1.1))
    arrow(ax, (3.6, y6), (6.3, 1.1))
    arrow(ax, (4.6, y6), (10.5, 1.1))

    fig.tight_layout(pad=0.6)
    out_path = "docs/architecture.png"
    fig.savefig(out_path, facecolor=BG, bbox_inches="tight")
    print(f"wrote {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
ARCHITECTURE_PNG_b64 = """
...
SUVORK5CYII=
"""
import base64, pathlib
pathlib.Path("docs/architecture.png").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("docs/architecture.png").write_bytes(base64.b64decode(ARCHITECTURE_PNG_b64))
print("wrote docs/architecture.png", pathlib.Path("docs/architecture.png").stat().st_size, "bytes")

In [ ]:
%%writefile README.md
# lmsafe — Safe Continual Updating for Lateral-Movement Detection

Research code accompanying:

> **Reliable Graph Foundation Models for Lateral Movement Detection: Cross-Network Transfer under Drift and Contaminated Baselines**

This package treats the *updating mechanism* of a lateral-movement (LM) detector as part of its attack surface, not just its training algorithm. A detector trained once and frozen degrades as the network legitimately changes ("drift"); a detector that keeps retraining on recent traffic to track that drift can be slowly poisoned by an attacker who seeds low-volume "conditioning" traffic before the real attack step. This package builds the experimental harness that measures both failure modes side by side, on the same detectors, the same metrics, and the same fixed alert budget — and implements a proposed defense (**Gated**) designed to avoid both at once.

## Research questions

| | Question | Script |
|---|---|---|
| **RQ1** | How much detection quality does each update policy lose to *benign* drift alone (no attacker), and how fast does it recover? | `scripts/run_rq1_drift_cost.py` |
| **RQ2** | How much does an adaptive attacker degrade each policy by injecting low-volume "boiling-frog" conditioning traffic before the real lateral-movement step? | `scripts/run_rq2_poisoning.py` |
| **RQ3** | Does the proposed **Gated** policy recover RQ1's drift-tracking benefit *without* reopening RQ2's poisoning gap — at the same fixed alert budget as the naive policies? | `scripts/run_rq3_gated_defense.py` |
| **Cross-network transfer** | Does pretraining `GraphFoundationDetector` on one network's benign traffic improve lateral-movement detection on a *different* network with a disjoint host population, versus training on the target network alone? | `scripts/pretrain_foundation_model.py` + `scripts/run_cross_network_transfer.py` |

## Pipeline

![lmsafe architecture](docs/architecture.png)

*(regenerate with `python docs/make_architecture_diagram.py`; the figure is drawn in matplotlib so it stays in sync with the code it documents — see that script's own docstring)*

Every data source — real LANL, real OpTC, or the synthetic generator — is adapted into one shared `Event` representation (`lmsafe/data/schema.py`). From there, the pipeline is dataset-agnostic: a fair labeling protocol, a fixed 41-hour train/test split, fixed-length window graphs, then one shared experiment loop (`lmsafe/runner.py`) that scores each test window with a `Detector`, hands the result to an `UpdatePolicy`, and records every metric the paper reports.

## Package layout

```
lmsafe/
  __init__.py          research framing + module map (also reproduced above)
  config.py            central constants (time cutoffs) and LanlConfig/SynthConfig dataclasses
  data/
    schema.py           the Event dataclass shared by every data source; ground-truth boundary
    lanl_stream.py       dependency-free streaming reader for the real LANL .txt.gz files
                          (never downloads a file whole; the signed link is read at runtime only)
    lanl.py              fair evaluation protocol: label join, remote-LogOn filter, train/test split
    optc.py              real OpTC FLOW-event reader, schema-confirmed against an actual
                          ecar-bro sample; IP-to-host resolution, no red-team label join yet
    windows.py           groups events into fixed-length WindowGraphs
  synth.py              synthetic auth-log generator: Zipf traffic, population growth ("drift"),
                        injectable lateral-movement episodes with known ground truth
  detectors/
    base.py              the Detector interface (fit / score / clone_fresh / state_summary)
    count_baseline.py    Laplace-smoothed edge-frequency anomaly score (no torch)
    feature_mlp.py        small inductive torch MLP over per-host structural features
                          (never per-node identity, to stay inductive across new hosts)
    rw_gnn.py              GraphFoundationDetector: feature_mlp's inductive per-host features
                          plus one round of GraphSAGE-style neighborhood aggregation, trained
                          self-supervised, with weights that can be pretrained on one network
                          and transferred (zero-shot or fine-tuned) to a different one -- the
                          "graph foundation model" piece of the title; see its module docstring
                          for the precise, deliberately modest scope claimed by that name
  drift.py              PageHinkley and SimplifiedADWIN concept-drift detectors (trigger updates;
                        never used as the anomaly score itself)
  policies.py           the five update policies: Frozen, Periodic, SlidingReplay, DriftTriggered,
                        Gated (the proposed defense)
  attack.py             the "boiling-frog" conditioning-attack injector, offline/evaluation-only
  metrics.py            weighted AUC/AP, Rec@B, episode clustering + detection delay, two
                        time-aware bootstrap CIs (block bootstrap, episode bootstrap)
  factories.py          name -> class lookups used by YAML configs and the RQ scripts

scripts/
  run_rq1_drift_cost.py       RQ1: drift-strength sweep, 5 policies, fixed episode set
  run_rq2_poisoning.py        RQ2: one hand-placed target episode, clean vs. conditioned
  run_rq3_gated_defense.py    RQ3: 2x2 (drift x conditioning), the headline result
  run_experiment.py           generic YAML-driven runner -- the one script meant to point at
                               real LANL/OpTC data (every other script is a synthetic demo)
  fetch_lanl_window.py        small CLI for a Colab smoke-test of the LANL link + caching windows
  pretrain_foundation_model.py  pretrain GraphFoundationDetector on one network (synthetic or a
                               cached LANL window pickle), save the checkpoint for transfer
  run_cross_network_transfer.py  scratch vs. zero-shot vs. fine-tuned on a second, disjoint
                               network -- the cross-network-transfer experiment itself

configs/                 example YAML configs for run_experiment.py (synthetic + LANL dev/full)
docs/                    this figure + the script that generates it
tests/                   130 tests (pytest); see "Testing" below
results/                 output of the last run of each script (CSV + PNG)
```

## Installation

```bash
pip install -e .
```

Pulls in `numpy`, `pandas`, `scikit-learn`, `torch`, `pyyaml`, `matplotlib`. CPU-only `torch` is enough — `FeatureMLPDetector` is a small per-host MLP, not a GPU-scale model.

## Quick start (synthetic data, no network, no credentials)

```bash
python -m pytest tests/ -q                       # 92 tests, ~6s
python scripts/run_rq1_drift_cost.py              # -> results/rq1/
python scripts/run_rq2_poisoning.py               # -> results/rq2/
python scripts/run_rq3_gated_defense.py           # -> results/rq3/
python scripts/run_experiment.py configs/synthetic_quick.yaml
```

Each RQ script prints a per-policy table and writes a CSV + PNG figure to `results/<rq>/`. These are genuine, non-cherry-picked runs on synthetic data with a fixed random seed, used throughout development to validate the harness end-to-end before spending any LANL streaming budget. A representative pattern from the current code (your own run will match, since the seeds are fixed):

- **RQ1** — every policy's AP degrades monotonically as injected drift strength increases (e.g. `frozen`'s AP drops from ~0.0054 at `drift=0.0` to ~0.0023 at `drift=1.0`); `periodic`/`sliding_replay`/`gated` degrade more slowly than `frozen`, because they retrain.
- **RQ2** — under the conditioning attack, the naive retraining policies' target-hop rank fraction jumps from `0.00` (clean) to `0.24–0.37` (conditioned) — the attack measurably hides the first hop of the real attack from them. `frozen` and `gated` both stay at `0.00`.
- **RQ3** — the 2×2 grid shows `gated` matching or beating the best naive policy's AP under benign drift *and* staying at `hop_rank_frac = 0.000` under conditioning in every cell, including the hardest one (`drift=strong` **and** `conditioning=on`), where every naive policy's rank fraction jumps to `0.60`–`0.625`. This is the paper's central claim, reproduced on synthetic data.

## Running on real data (LANL / OpTC)

This sandbox cannot reach LANL's network, so real-data runs happen on your own machine or Colab notebook.

1. Get a signed download link from <https://csr.lanl.gov/data/cyber1/> (registration required). The link is a **temporary credential** — it is read once at runtime via the `LMSAFE_LANL_BASE_URL` environment variable (or an interactive, hidden `getpass` prompt if unset) and is never written to a config file, logged, or printed. See `lmsafe/data/lanl_stream.py` for the exact mechanism.
2. Smoke-test the link and cache a windowed slice:
   ```bash
   export LMSAFE_LANL_BASE_URL='<the signed link>'
   python scripts/fetch_lanl_window.py peek --file redteam --n 5
   python scripts/fetch_lanl_window.py redteam --out redteam.csv       # expect 749 rows
   python scripts/fetch_lanl_window.py build-cache --end-s 2592000 \
       --out lmsafe_cache/lanl_dev_windows.pkl
   ```
   Both `build-cache` above and `run_experiment.py`'s own LANL loader checkpoint the growing window cache atomically every `checkpoint_every` (default 24) test windows, not only once at the very end — a multi-hour real stream that gets interrupted (Colab disconnect, Ctrl+C, a dropped connection) still leaves a complete, loadable partial result on disk instead of nothing. **This does not auto-resume the network stream**: re-running with an existing cache file loads it as-is and treats it as final, so a true complete run requires deleting the cache file first. `stream_auth_events` also runs a raw-bytes prefilter before decoding/constructing an `Event` for each line, skipping that work entirely for the ~63% of real LANL rows the fair protocol would discard anyway — a provably-equivalent, test-covered optimization (see `tests/test_lanl_labels.py`) measured at ~2x CPU-only throughput on a synthetic benchmark; real wall-clock speedup is smaller since network/decompression time is untouched.
3. Run the full detector/policy sweep against the cached windows:
   ```bash
   python scripts/run_experiment.py configs/lanl_dev.yaml     # 30-day cut, fast dev iteration
   python scripts/run_experiment.py configs/lanl_full.yaml    # 58-day cut, final paper numbers
   ```
   `configs/lanl_dev.yaml`'s 30-day cutoff is not an arbitrary shortcut: inspecting the real `redteam.txt.gz` found all 749 labeled red-team events fall before day 29.6, so a 30-day cut already contains every known attack while reading roughly half the 7.2 GB auth stream. `configs/lanl_full.yaml` reads the complete 58-day release to match the LMDEval fair-evaluation protocol exactly, for the numbers that go in the paper.
4. Pretrain the graph foundation model and evaluate cross-network transfer (see "Cross-network transfer" below):
   ```bash
   python scripts/pretrain_foundation_model.py --source lanl-cache \
       --cache-path lmsafe_cache/lanl_full_windows.pkl --out checkpoints/foundation_lanl.pt
   python scripts/run_cross_network_transfer.py --network-a-cache-path lmsafe_cache/lanl_full_windows.pkl
   ```

A ready-to-run Colab notebook for this whole flow is at `lanl_colab.ipynb` (see below).

### Cross-network transfer

`lmsafe/detectors/rw_gnn.py`'s `GraphFoundationDetector` is this project's "graph foundation model": `feature_mlp`'s inductive, never-per-identity host features plus one round of pure-PyTorch GraphSAGE-style neighborhood aggregation (no `torch_geometric` dependency), trained self-supervised via link prediction. Because nothing in it is keyed to a specific host's identity, a checkpoint trained on one network's benign traffic (`scripts/pretrain_foundation_model.py`) can be applied to a second network with a completely disjoint host population — either zero-shot (no further training at all, just `GraphFoundationDetector.observe()` letting its running host-stats accumulate through the new stream) or fine-tuned on a small target-network training split. `scripts/run_cross_network_transfer.py` compares these two regimes against training from scratch on the target network alone, all three frozen after their one initial fit so the comparison isolates what each regime started from. This mirrors the zero-shot/fine-tune/from-scratch comparison Larroche's own prior GFM work on LANL/OpTC uses.

**Real OpTC support.** `lmsafe/data/optc.py` has a real FLOW-event reader, built and schema-confirmed against an actual `ecar-bro/short` sample from the official release (not guessed from the generic public eCAR docs, which don't document the FLOW-specific fields). A FLOW record carries no destination *hostname*, only a `dest_ip`/`src_ip` pair and a `direction`; the reader reconstructs host-to-host edges by building an `ip -> hostname` table purely from self-observation (every host's own outbound/inbound records reveal its own IP) and resolving each record's other endpoint through that table, dropping edges to unresolved (external) IPs by default — the OpTC-equivalent of LANL's "remote LogOn between two different hosts" fairness filter. Every emitted `Event` sets `orientation="LogOn"` so it satisfies the shared `is_remote_logon()` check unchanged, meaning every downstream piece (`iter_windows`, `lanl.split_train_test`, every detector/policy/metric) runs on OpTC with zero changes, exactly as the original stub's design promised. It does not yet join against a red-team label file (the likely source, `OpTCRedTeamGroundTruth.pdf`, ships alongside the data but isn't parsed yet), so a real-OpTC transfer run's AUC/AP/episode metrics currently read as NaN — the transfer *mechanism* is real, the grading isn't yet.

That real 241-line `ecar-bro/short` sample is bundled with this repo at `tests/fixtures/optc_ecarbro_short_sample.json` — not just referenced in prose. `lanl_colab.ipynb`'s cross-network-transfer section (step 3) runs `scripts/run_cross_network_transfer.py` against this exact file end to end, twice: once with the default fairness filter (reports, correctly, that this specific partition has zero internal-to-internal windows — it is entirely outbound beaconing to external IPs), and once with `--network-b-optc-keep-external` (runs the full read → resolve → window → score → observe path to completion and writes a results table, all malicious-label metrics NaN as expected, since there is no label join yet). Both are executed for real in every fresh-kernel verification of this notebook, not merely documented. Point `--network-b-optc-paths <file...>` at a different/larger real `ecar-bro` file (`.json` or `.json.gz`, auto-detected) — e.g. one fetched from `ecar-bro/evaluation`, the actual red-team engagement day — to use it as network B instead of the bundled sample; that engagement folder is the likelier source of real lateral-movement-relevant traffic, since the bundled `short`-partition sample has none.

## Methodology notes

- **Fairness protocol.** Adapted from Larroche's LMDEval (BSD-2-Clause): the train/test split is a fixed 41-hour cutoff; *all* remote `LogOn` events are kept (no NTLM-only filtering, no downsampling) unless a config explicitly opts into `filter_ntlm_only` for an ablation; the ground-truth label join key is `(t, dst_user, src_comp, dst_comp)` — **not** `src_user`, which is a documented bug in some prior LANL pipelines that silently drops most true positives.
- **Score convention.** Every detector and every metric in this package shares one convention: **higher score = more anomalous.** There is no sign flip anywhere.
- **Ground-truth boundary.** `is_malicious` / `edge_is_malicious` exists purely so `lmsafe.metrics` can grade a run after the fact. No `Detector.score()`/`fit()` or `UpdatePolicy.observe()`/`fit_initial()` implementation may read it — this is enforced both by a narrow check inside `tests/test_detectors.py` and by a project-wide static AST scan in `tests/test_no_label_leak.py` that would fail CI if a future detector or policy violated it.
- **Metrics.** Event-level weighted AUC/AP (via `sklearn`'s `sample_weight`, so edge-deduplicated rows still reflect true event counts); `Rec@B` = recall at a fixed **per-window** alert budget (not a global, hindsight-only budget); episode clustering by inter-window time gap; per-episode detection delay. Confidence intervals use two different bootstraps because the two statistics have different natural resampling units: `block_bootstrap_ci` resamples contiguous *blocks of windows* for AUC/AP/Rec@B (a standard moving-block bootstrap, since consecutive windows are correlated), and `episode_bootstrap_ci` resamples whole *episodes* for delay (since all of one episode's per-window correlation is already baked into a single scalar).
- **The Gated defense.** `GatedPolicy` (`lmsafe/policies.py`) combines three mechanisms: score-gated admission (don't even consider retraining on an edge the current model already finds too anomalous), a quarantine delay before any candidate edge can be committed, and a multi-host corroboration requirement before commit (an edge is only "confirmed benign drift" if independently corroborated by enough distinct source hosts — motivated by the real LANL red-team data, where 701 of 749 labeled events originate from a single host, so a single-host pattern is cheap for an attacker to fabricate but a multi-host one is not).

## Testing

```bash
python -m pytest tests/ -q
```

130 tests covering: the `Event`/`WindowGraph` schema and the LANL label-join protocol (checked against the real, included `redteam.txt.gz` fixture — 749 rows, first event at 150,885s, last at 2,557,047s — and against real/documented raw `auth.txt.gz` sample lines parsed end-to-end); the raw-bytes prefilter's exact agreement with the slow path, including on adversarial invalid-UTF-8 input; crash-safe checkpointing (a simulated mid-stream interruption leaves a complete partial cache, never a truncated/corrupt one); the OpTC FLOW reader (a real sample line replayed verbatim, variable-fraction-digit timestamp parsing, IP-to-host self-observation, both outbound/inbound direction cases, self-loop and external-IP dropping, that emitted events feed `iter_windows`/`lanl.split_train_test` completely unchanged, and an end-to-end read of the full real, bundled `tests/fixtures/optc_ecarbro_short_sample.json` fixture pinning its known zero-internal-flows finding); the synthetic generator; all three detectors (including `GraphFoundationDetector`'s neighborhood aggregation actually depending on neighbor features, checkpoint export/import round-tripping identical scores, `freeze_encoder` genuinely freezing weights, and a pretrained checkpoint producing finite scores on a fully disjoint, relabeled host population); the drift detectors; all five update policies (including a regression test that `Periodic` and `SlidingReplay` are *not* behaviorally identical under their documented defaults — an earlier, now-fixed bug); the attack/conditioning injector; every metric function; the end-to-end runner; and the ground-truth-isolation static check described above.

## Known limitations / next steps

- OpTC's red-team label file (likely `OpTCRedTeamGroundTruth.pdf`, shipped alongside the data) is not yet parsed/joined, so every OpTC `Event` currently has `is_malicious=False` — a real-OpTC cross-network-transfer run's AUC/AP/episode metrics read as NaN until that join exists. The transfer *mechanism* itself (reader → windows → pretrain/zero-shot/fine-tune) is real and tested against an actual sample.
- `ecar-bro` is partitioned by host range, not globally time-ordered, so `lmsafe.data.optc.stream_optc_events` loads every given file fully into memory and sorts in-memory rather than streaming like `lmsafe.data.lanl_stream` does — fine at the "a handful of real sample files" scale this project operates at, but it would need an external k-way merge before pointing at the full ~1 TB release.
- The paper's final numbers should come from `configs/lanl_full.yaml` (58-day cutoff) run end-to-end on real LANL data — everything in `results/` right now is the synthetic validation run, which exists to prove the harness itself is correct before spending real-data compute.
- `FeatureMLPDetector` is only spot-checked against `CountBaselineDetector` in the RQ1 sweep so far; a full RQ1–RQ3 sweep with `FeatureMLPDetector` as the primary detector is a natural next experiment once real-data numbers are in.
- `GraphFoundationDetector`'s cross-network-transfer numbers so far are on synthetic networks only; the real claim in the title needs a real second network (OpTC) once that reader exists.


In [ ]:
from IPython.display import Image, display

display(Image(filename="docs/architecture.png"))

## 8) Enter the LANL link (hidden)

Paste any one of the five links you received (e.g. the `auth.txt.gz`
link) -- the code below extracts the fixed part automatically. **Input is
hidden and is not saved in the notebook.**

In [ ]:
import os, getpass

os.environ["LMSAFE_LANL_BASE_URL"] = getpass.getpass("Signed LANL link: ")
print("Link saved to this session's environment variable only (never written to any file).")

## 9) Quick smoke test of the link (seconds)

You should see 3 lines from `redteam` then 3 lines from `auth`. If you get
an `HTTP 401/403/410` error, the link has expired -- get a fresh one from
the LANL page and re-run the previous cell.

In [ ]:
!python scripts/fetch_lanl_window.py peek --file redteam --n 3
!python scripts/fetch_lanl_window.py peek --file auth --n 3

## 10) The red-team file (tiny, 4.8K)

Expected: 749 rows (documented in LANL's public release notes, and
verified against the real file during this project's development).

In [ ]:
!python scripts/fetch_lanl_window.py redteam --out redteam.csv

## 11) Build a dev-cutoff window cache -- 30 days

`--end-s 2592000` = 30 days. This project inspected the real
`redteam.txt.gz` and found all 749 labeled events fall before day 29.6 --
so a 30-day cut already covers every known red-team attack while reading
roughly half of `auth.txt.gz` (7.2GB), and without ever downloading it
whole (it is only ever read while streaming). This is the slowest cell in
the notebook -- it prints progress periodically.

**Crash-safe**: `--checkpoint-every` (default 24 test windows, roughly
once per simulated day) writes an atomic partial `(train_w, test_w)`
checkpoint to `--out` as it goes, not just once at the very end. If this
cell is interrupted (Ctrl+C, a Colab disconnect, a lost connection) --
which is a real risk on a multi-hour run -- `--out` already holds the
most recent checkpoint and `run_experiment.py` can use it right away,
instead of the run leaving nothing on disk at all.

In [ ]:
!python scripts/fetch_lanl_window.py build-cache \
    --end-s 2592000 \
    --checkpoint-every 24 \
    --out lmsafe_cache/lanl_dev_windows.pkl

## 12) Run the full sweep: one detector x 5 update policies

`configs/lanl_dev.yaml` uses the cache we just built (`cache_path`), so
the data is never re-streamed from the network. It prints a table with
every metric this project reports (weighted AUC, weighted AP, Rec@B at
several alert budgets, and episode hit rate) for each of the five
policies (`frozen`, `periodic`, `sliding_replay`, `drift_triggered`,
`gated`).

In [ ]:
!python scripts/run_experiment.py configs/lanl_dev.yaml

## 13) Show the results as a table

In [ ]:
import pandas as pd

df = pd.read_csv("results/lanl_dev/results.csv")
df

## 14) (Optional) run the three research-question scripts on synthetic data

The three scripts below (`run_rq1_drift_cost.py`, `run_rq2_poisoning.py`,
`run_rq3_gated_defense.py`) are demonstrations built on **synthetic data**
(not real LANL) -- they validate the framework itself (drift cost,
update-poisoning attack, the Gated defense) in isolation and quickly.

In [ ]:
!python scripts/run_rq1_drift_cost.py
!python scripts/run_rq2_poisoning.py
!python scripts/run_rq3_gated_defense.py

## 15) (Optional, slow) the full 58-day cut -- final paper numbers

`configs/lanl_full.yaml` reads the complete 58-day public release (to
match the LMDEval protocol exactly), and is much slower than the dev
cut -- expect several hours. Run it only once you're ready to extract
the final numbers for the paper.

**If this cell gets interrupted** (Colab disconnect, Ctrl+C, closing the
tab, a network drop) -- a real risk over several hours -- it is now
crash-safe: `lanl.cache_path` with `checkpoint_every: 24` in the config
means `lmsafe_cache/lanl_full_windows.pkl` is updated with a complete,
loadable partial result every ~24 simulated test windows (about once per
simulated day), not only once at the very end. So an interruption near
the end (e.g. at day 50 of 58) still leaves a usable, near-complete
dataset on disk, instead of nothing.

**Important**: this does *not* auto-resume the network stream. If
`cache_path` already exists, the next run loads it as-is and treats it
as the final dataset -- it will *not* keep streaming the missing days.
If you want the true, complete 58 days (not a partial cut), you must
delete `lmsafe_cache/lanl_full_windows.pkl` and re-run this cell from
scratch (the stream itself restarts from the beginning, so this re-pays
the full time cost, not just the missing tail). If a partial cut is
good enough for a given purpose, just leave the checkpoint in place and
proceed -- `build_lanl_windows` now prints exactly how many days the
loaded cache covers against how many the config asked for, and prints a
loud `WARNING` if they don't match, so this is never silently ambiguous
on a later run.

**This happened in practice**, not just hypothetically: a real run of
this exact cell hit six separate network reconnects (the LANL signed
link is not perfectly stable over many hours) and was eventually
stopped by hand at day 40.64 of 58. That is a real, usable result, not
a failure to throw away -- a separate check of the real `redteam.txt.gz`
file (see step 6/7) already established that **all 749 labeled
red-team events occur before day 30**, so any run that reaches day 30+
already has 100% of the known ground truth. The only thing the
remaining days 41-58 add is exact comparability with LMDEval's published
58-day evaluation window -- they do not add any additional attacks to
detect. A partial run like this is honestly reported as "real LANL,
~40-day evaluation window (full red-team coverage)", not as "the 58-day
LMDEval-comparable number" -- those are two different, both legitimate,
claims.

In [ ]:
!python scripts/run_experiment.py configs/lanl_full.yaml

## 16) Graph foundation model: pretrain, then evaluate cross-network transfer

This is the "Graph Foundation Models" / "Cross-Network Transfer" piece
of the original proposal title, added in this revision of the project.

**What `GraphFoundationDetector` actually is** (see
`lmsafe/detectors/rw_gnn.py`'s module docstring for the full
rationale): the same inductive, ID-free per-host running features
`FeatureMLPDetector` uses, plus one round of GraphSAGE-style
neighborhood aggregation within each window's own graph. No per-host
embedding table anywhere, which is exactly what makes pretraining on
one network and applying the result to a *different* network's
completely disjoint host population a coherent thing to even attempt.
This is a deliberately small, CPU-trainable model -- minutes to
pretrain on a 2-CPU sandbox, not a claim to reproduce a large published
architecture at scale.

**Step 1 -- pretrain.** `scripts/pretrain_foundation_model.py` fits the
encoder + edge-scorer on one network's benign windows and saves a
checkpoint. The cell below uses `--source synthetic` so it runs fully
offline in seconds; point `--source lanl-cache --cache-path
lmsafe_cache/lanl_full_windows.pkl` at it instead to pretrain on real
LANL once that cache exists.

In [ ]:
!python scripts/pretrain_foundation_model.py --source synthetic \
    --n-synthetic-windows 300 --progress-every 100 \
    --out checkpoints/foundation_synthetic.pt

**Step 2 -- cross-network transfer.** `scripts/run_cross_network_transfer.py`
compares three regimes on a held-out "network B": training from
scratch on B alone, zero-shot transfer from a network-A pretrained
checkpoint with no exposure to B at all, and pretrain-then-fine-tune on
B's own small training split. By default both networks are synthetic,
with network B's hosts explicitly relabeled into an OpTC-style naming
scheme (`SysClient0001`, ...) fully disjoint from network A's, so the
transfer is real and not an artifact of two runs sharing a naming
convention -- see the script's own docstring for exactly how each
regime is evaluated.

In [ ]:
!python scripts/run_cross_network_transfer.py --out-dir results/cross_network_transfer

In [ ]:
import pandas as pd
pd.read_csv("results/cross_network_transfer/cross_network_transfer_results.csv")

**Step 3 -- the same script, run against a real OpTC file (not just
documented).** `lmsafe/data/optc.py` has a real FLOW-event reader,
built and schema-confirmed against an actual `ecar-bro/short` sample
obtained from the official OpTC Google Drive release (OpTC is ~1 TB
total, hosted on Google Drive rather than a streamable signed link like
LANL's, so a small representative sample is the realistic path here,
not the full release). It resolves host-to-host edges from raw
`(src_ip, dest_ip, direction)` triples by building an IP-to-hostname
table purely from self-observation (see the module docstring), and
reuses every downstream piece (`iter_windows`, `lanl.split_train_test`,
every detector/policy/metric) completely unchanged.

That exact 241-line real sample is bundled with this notebook at
`tests/fixtures/optc_ecarbro_short_sample.json` (written to disk back
in step 6, alongside the LANL `redteam.txt.gz` fixture) -- the cell
below runs `run_cross_network_transfer.py` against it for real, right
here, not as a documented-but-unexecuted example. Its own honest,
already-known outcome: this particular partition turned out to be all
outbound traffic to external IPs (benign cert-check beaconing plus one
Notepad++-update-hijack/C2 sequence), so the default fairness filter
(`only_internal_internal=True`, OpTC's equivalent of LANL's "remote
LogOn between two different hosts" rule) correctly finds **zero**
internal-to-internal windows -- the script detects this and reports it
plainly instead of faking a result.

In [ ]:
!python scripts/run_cross_network_transfer.py \
    --network-b-optc-paths tests/fixtures/optc_ecarbro_short_sample.json \
    --out-dir results/cross_network_transfer_optc_real

Passing `--network-b-optc-keep-external` relaxes that filter so the
real reader's full read -> resolve -> window -> score -> observe path
still runs end to end and writes a results table -- every malicious-label
metric reads as NaN/0 (there is no red-team label join yet, see "Known
limitations" in the README), but this proves the *mechanism* is real on
real data, not synthetic data wearing an OpTC-shaped name.

In [ ]:
!python scripts/run_cross_network_transfer.py \
    --network-b-optc-paths tests/fixtures/optc_ecarbro_short_sample.json \
    --network-b-optc-keep-external \
    --out-dir results/cross_network_transfer_optc_real_keep_external
import pandas as pd
pd.read_csv("results/cross_network_transfer_optc_real_keep_external/cross_network_transfer_results.csv")

**To use a different, larger real file** (e.g. from `ecar-bro/evaluation`,
the actual red-team engagement day, which is the likelier source of
real lateral-movement-relevant internal traffic -- the `short` sample
above had none), fetch it into this Colab session the same way you
fetched the sample above, then point `--network-b-optc-paths` at it
instead of the bundled fixture:

```bash
python scripts/run_cross_network_transfer.py \
    --network-b-optc-paths /content/your_evaluation_file.json.gz
```


## 17) Copy everything to Google Drive (so it survives the session ending)

`drive.mount()` uses Google's DriveFS, which can fail with `ValueError:
mount failed` -- most often a timeout, and more likely after a long,
heavy session (a multi-hour real-LANL run is exactly that). The cell
below tries the mount once with `force_remount=True` (clears a stale
previous mount attempt, which is a common cause on its own), and if
that still fails, falls back to zipping everything and downloading it
straight to your computer via `google.colab.files.download()` -- that
path never touches DriveFS at all, so it works even when Drive mounting
itself is broken for this session. If even the download fails (some
browsers block multi-file/large downloads), the zip still exists at
`/content/lmsafe_results.zip` in this session for you to grab by hand
from the Colab file browser on the left.

In [ ]:
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    get_ipython().system('mkdir -p /content/drive/MyDrive/lmsafe_results')
    get_ipython().system('cp -r results redteam.csv lmsafe_cache /content/drive/MyDrive/lmsafe_results/ 2>/dev/null')
    get_ipython().system('ls -lh /content/drive/MyDrive/lmsafe_results/')
    print("Copied to Google Drive: MyDrive/lmsafe_results/")
except Exception as e:
    print(f"Drive mount failed ({e!r}) -- falling back to a direct download instead.")
    shutil.make_archive("/content/lmsafe_results", "zip", ".", "results")
    # redteam.csv / lmsafe_cache may not exist depending on which cells ran -- best effort.
    import zipfile
    with zipfile.ZipFile("/content/lmsafe_results.zip", "a") as zf:
        for extra in ("redteam.csv",):
            if __import__("os").path.exists(extra):
                zf.write(extra)
        import os
        for root, _, files in os.walk("lmsafe_cache"):
            for f in files:
                zf.write(os.path.join(root, f))
    print("Wrote /content/lmsafe_results.zip -- downloading now.")
    from google.colab import files
    files.download("/content/lmsafe_results.zip")